# DeepCollector System Architecture and API Definition

This document defines the modular architecture of the DeepCollector system (V13+ Modular). It serves as the foundational context for iterative development using the "Freeze and Focus" strategy.

## 1. System Overview

DeepCollector is an agentic system designed to automatically discover, catalog, and verify time-series datasets from various repositories. It operates in a multi-phase workflow:

1.  **Phase 0: Bootstrapping:** Initialize environment, load Knowledge Base (KB), index initial sources.
2.  **Phase 1: Discovery:** Identify relevant dataset names and initial relevance.
3.  **Phase 2: Grounding:** Verify data sources, find URLs, and utilize Direct Data Inspection (DDI).
4.  **Phase 3: Extraction:** Fill detailed metadata using Cellular RAG.
5.  **Phase 4: Write-back:** Reconcile findings with the persistent KB (Google Sheets).

## 2. Modular Structure

The system is organized into the `deepcollector` Python package, created dynamically in Colab via `%%writefile`.

```plaintext
deepcollector/
├── config/
│   ├── settings.py         # AppConfig (Central configuration object)
│   └── schema.py           # KB Schema (V1.3) and Internal Agent Schema
├── core/
│   ├── agent.py            # (Orchestrator) CatalogAgent (Workflow Phases)
│   ├── state.py            # (State Management) CatalogState (In-memory state, LlamaIndex)
│   └── rag_engine.py       # (Frozen/Stable) RAGEngine (Cellular RAG, Planners, Auditors)
├── tools/
│   ├── ddi.py              # (Stable) Direct Data Inspection (File parsing)
│   ├── research.py         # (Stable) Web search, fetching, LLM wrappers
│   └── knowledge_processor.py # (Stable) Google Doc parsing
├── kb/
│   └── manager.py          # (Stable) KnowledgeBaseManager (Google Sheets I/O)
├── harvesting/             # (Active Development - New Capability)
│   ├── base_harvester.py   # (Stable) Abstract Base Class (The Harvester API) (Future)
│   ├── uci_harvester.py    # (Active) Specific implementation for UCI (Future)
│   └── (future_adapters.py)
└── utils/
    # (Stable) Profiling and Initialization utilities
3. Key APIs and Interfaces
These APIs define the contracts between modules, allowing components to be developed independently.

3.1. The Harvester API (Adapter Pattern)
Used during the Discovery Phase (Phase 1) when targeting structured repositories. (Future implementation)

Interface: deepcollector.harvesting.base_harvester.BaseHarvester
from abc import ABC, abstractmethod
from typing import List, Dict, Any

class BaseHarvester(ABC):
    def __init__(self, config: Any, tools: Any):
        # ... initialization ...
        pass

    @abstractmethod
    def discover_assets(self) -> List[Dict[str, Any]]:
        """
        Explores the repository and returns initial dataset candidates.

        Returns:
            List[Dict]: Format: [{'name': 'DatasetName', 'url': 'SourceURL', 'rationale': 'Description.'}]
        """
        pass

## **Old Description**

In [ ]:
# DeepCollector Dataset Collection Analyser
# This is trying to make a custom agentic AI that would implement the instructions below that we were built by hand to build a dataset Collection table
# This was implemented for Time Series but is of broader utility
# There are about 80 Time Series dataset Collections https://docs.google.com/document/d/1jNViEKDX_c3Em3MuqLNfdfuqI1_5h97sWuXRZCQ8uT4/edit?usp=sharing
#
# Now we describe the goals of processing a dataset ciollection
# Define Project Name: The XYZ Collection
# Give one to four links for web resources (papers, GitHubs) defining collection
#
# Please include all datasets and not just a representative set. Please include datasets used in both pretraining and evaluation
#
# Table Columns:
# Dataset Name,	Domain,  Number of Variables at each time point,	Number of Time Points,	Time interval between points,	Primary Source Repository,	Link to Data,	Detailed Description,	Comments,
# It is important that "When you provide a link, please format it as a Google Sheets formula, like this: =HYPERLINK("https://www.example.com", "Example Link Text")". You can only put one =HYPERLINK in a cell. If there is greater than one hyperlink in a single cell, please use just use pure text with each link defined by a non-clickable Display Text: URL syntax. Also only use a single line in cell
# If multiple URLs in a cell, make the most important as =HYPERLINK and just list the others as Google Spreadsheets olnly allow one =HYPERLINK
#
# Note Columns are defined but rows of table (psrticular datasets) are to be discovered
# We can also specify rules on cells of table such as
# a) No empty cells
# b) If variable numbber give a range of possible values
# c) We could establish or build ontologies for values of Domain column
#
# The DeepCollector is first run separately on each collection and then we integrate the ~80 different tables produced with further tasks implemented in Colab
#
# Merger: Merge all individual tables into a single Table with Pointers linking datasets to Collections. This gives around 750 datasets for ~80 Time Series Collections
# Reconciler: Compare different versions of a given dataset to determine if they are identical; if not, split them. This leads to around 950 datasets
# Deduplicator: Do the opposite. Merge different datasets. This gets you back to around 800 datasets
#
# The Reconciler and Deduplicator use simple LLMs (direct calls to Gemini Pro) to help distinguish datasets with greater precision than simple textual comparisons
#
# The first project is the Tempo times series foundation Model Ciollection
# -----------------------------------------------------------------------------------

# ------------------------------------------------------------------------------------------
# DeepCollector Dataset Collection Analyser
# Goal: Generalized and highly focused pipeline, using external KB strategically.
# -------------------------------------------------------------------------------------

#### **Cell 1: Environment Setup and Dependencies**

This cell installs dependencies, sets up the directory structure, and handles initial imports.

In [ ]:
# =============================================================================
# Cell 1: Environment Setup, Auth, and Global Definitions
# =============================================================================
import sys
import os
import time
import warnings
import subprocess
import signal

# 1. Check Python Environment
IN_COLAB = 'google.colab' in sys.modules

# =============================================================================
# --- MASTER TOGGLES ---
# =============================================================================
os.environ["DEEPCOLLECTOR_LLM_BACKEND"] = "GEMINI"
os.environ["DEEPCOLLECTOR_OUTPUT_FORMAT"] = "SHEET"
os.environ["DEEPCOLLECTOR_USE_VLLM"] = "False"  # 🚨 CRITICAL: vLLM DISABLED FOR COLAB

os.environ["DEEPCOLLECTOR_SEARCH_BACKEND"] = "GEMINI"
os.environ["DEEPCOLLECTOR_SEARXNG_URL"] = "http://localhost:8080"
os.environ["JAX_PLATFORMS"] = "cpu"

# =============================================================================
# 2. COMBINED AUTHENTICATION (Drive & Sheets)
# =============================================================================
gc = None
GSPREAD_AVAILABLE = False

if IN_COLAB:
    try:
        from google.colab import drive
        print("\n📁 Requesting Google Drive access (for local PDFs and CSV exports)...")
        drive.mount('/content/drive')
    except Exception as e:
        print(f"⚠️ Google Drive mount failed: {e}")

    try:
        from google.colab import auth
        import gspread
        from google.auth import default
        print("\n🔑 Requesting Google Sheets API access...")
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        GSPREAD_AVAILABLE = True
        print("✅ Google Sheets authentication successful.")
    except Exception as e:
        print(f"\n❌ Google Sheets authentication failed: {e}")
        print("\n" + "="*80)
        print("   👉 TROUBLESHOOTING 'credential propagation was unsuccessful':")
        print("      1. Your web browser is likely blocking Third-Party Cookies.")
        print("      2. Look at your URL bar and disable 'Brave Shields' or 'Tracking Protection' for Colab.")
        print("      3. Allow third-party cookies for 'colab.research.google.com'.")
        print("      4. Try opening this Colab notebook in an 'Incognito' or 'Private' window.")
        print("      5. Ensure you are only logged into ONE Google account in this browser profile.")
        print("="*80)
        sys.exit("🛑 ABORTING: Execution halted because Google Sheets access is required.")
else:
    print("\n🔑 Local DGX Environment Detected. Requesting Google Sheets API access...")
    try:
        import gspread
        from google.auth import default
        creds, _ = default()
        gc = gspread.authorize(creds)
        GSPREAD_AVAILABLE = True
        print("✅ Google Sheets authentication successful.")
    except Exception as e:
        print(f"\n⚠️ Google Sheets authentication failed locally (will run in offline mode): {e}")

# =============================================================================
# 3. Install Necessary Libraries
# =============================================================================
STABLE_PACKAGES = [
    "google-genai",
    "llama-index>=0.10.50",
    "llama-index-llms-gemini",
    "llama-index-llms-google-genai",
    "llama-index-embeddings-google-genai",
    "llama-index-llms-openai",
    "openai",
    "transformers",
    "huggingface-hub>=0.23.0",
    "accelerate>=0.27.0",
    "bitsandbytes>=0.41.0",
    "torch>=2.0.0",
    "torchvision",
    "torchaudio",
    "llama-index-llms-huggingface",
    "llama-index-embeddings-huggingface",
    "llama-index-retrievers-bm25",
    "gspread>=5.0.0",
    "requests",
    "beautifulsoup4",
    "lxml",
    "pypdf>=3.0.0",
    "nest_asyncio",
    "ucimlrepo",
    "tabulate",
    "tenacity",
]

GIT_PACKAGES = [
    "git+https://github.com/huggingface/transformers.git",
    "huggingface-hub"
]

print("\n🔧 Installing Step 1: Stable Packages...")
try:
    process1 = subprocess.Popen(
        [sys.executable, "-m", "pip", "install", "--upgrade"] + STABLE_PACKAGES,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in iter(process1.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
    process1.wait()

    print("\n🚀 Installing Step 2: Bleeding-Edge Transformers...")
    process2 = subprocess.Popen(
        [sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps"] + GIT_PACKAGES,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in iter(process2.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
    process2.wait()

    if process1.returncode == 0 and process2.returncode == 0:
        print("\n✅ Two-Step Pip Installation successful.")
    else:
        print("\n⚠️ Pip install finished with errors. Check logs above.")

except Exception as e:
    print(f"\n⚠️ Pip install execution failed: {e}")

try:
    import nest_asyncio
    nest_asyncio.apply()
    print("🔀 nest_asyncio applied.")
except ImportError:
    print("⚠️ nest_asyncio not found.")

# =============================================================================
# 4. CRITICAL PRE-FLIGHT CHECK & GPU WATCHDOG
# =============================================================================
print("\n🔍 Verifying Critical Dependencies...")

class DeadlockTimeoutException(Exception):
    pass

def deadlock_handler(signum, frame):
    raise DeadlockTimeoutException()

try:
    if hasattr(signal, 'SIGALRM'):
        signal.signal(signal.SIGALRM, deadlock_handler)
        signal.alarm(30)

    import llama_index.llms.gemini
    import torch
    import transformers

    if hasattr(signal, 'SIGALRM'):
        signal.alarm(0)
    print("    ✅ Core LLM dependencies (Gemini & HF) are present.")

except DeadlockTimeoutException:
    print("\n" + "="*80)
    print("🚨 CRITICAL HARDWARE DEADLOCK DETECTED 🚨")
    print("The Colab/DGX GPU is stuck in a corrupted state from a previous crash.")
    print("👉 YOU MUST RESTART THE RUNTIME: Click 'Runtime' in the top menu -> 'Restart session'")
    print("="*80)
    sys.exit("GPU Deadlock. Restart required.")
except ImportError as e:
    if hasattr(signal, 'SIGALRM'):
        signal.alarm(0)
    print("\n" + "="*60)
    print(f"🛑 CRITICAL FAILURE: Missing dependencies -> {e}")
    print("   Execution halted to prevent data loss/corruption.")
    print("="*60)
    sys.exit("MISSING_DEPENDENCY")

# =============================================================================
# 5. Global Imports and Variable Definitions
# =============================================================================
def check_import(module_name):
    try:
        __import__(module_name)
        return True
    except Exception as e:
        print(f"    ⚠️ Warning importing {module_name}: {e}")
        return False

if not GSPREAD_AVAILABLE:
    GSPREAD_AVAILABLE = check_import("gspread")

LLAMA_INDEX_AVAILABLE = check_import("llama_index.core")
UCIMLREPO_AVAILABLE = check_import("ucimlrepo")

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

try:
    from llama_index.retrievers.bm25 import BM25Retriever
except ImportError:
    BM25Retriever = None

# =============================================================================
# 6. Create Directory Structure
# =============================================================================
print("\n📁 Creating directory structure...")
os.makedirs("deepcollector/config", exist_ok=True)
os.makedirs("deepcollector/core", exist_ok=True)
os.makedirs("deepcollector/tools", exist_ok=True)
os.makedirs("deepcollector/kb", exist_ok=True)
os.makedirs("deepcollector/utils", exist_ok=True)
os.makedirs("deepcollector/harvesting", exist_ok=True)

for path in ["deepcollector", "deepcollector/config", "deepcollector/core", "deepcollector/tools", "deepcollector/kb", "deepcollector/utils", "deepcollector/harvesting"]:
    open(f"{path}/__init__.py", "a").close()

print("✅ Setup complete. Ready to run subsequent cells.")
time.sleep(1)


📁 Requesting Google Drive access (for local PDFs and CSV exports)...
Mounted at /content/drive

🔑 Requesting Google Sheets API access...
✅ Google Sheets authentication successful.

🔧 Installing Step 1: Stable Packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 1

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


    ✅ Core LLM dependencies (Gemini & HF) are present.

📁 Creating directory structure...
✅ Setup complete. Ready to run subsequent cells.


### Module Writing (The "Frozen" Components)

The following cells use `%%writefile` to create the Python package structure.

#### **Cell 2: deepcollector/config/schema.py**

Defines constants and schemas.

In [ ]:
%%writefile deepcollector/config/schema.py
# =============================================================================
# V83: Schema (URL Separation and Confidence Updates)
# Features:
# 1. Added 'Other URL' and 'Other URL (C)'.
# 2. Added 'Link to Data (Actual Source) (C)'.
# 3. Enforced strict rules against Google Scholar/Vertex AI in RAG prompts.
# =============================================================================

KB_SCHEMA_VERSION = "24.04"

# DEFINITION OF TABS AND COLUMNS
KB_SCHEMA = {
    "Jobs": [
        "JobID", "ProjectID", "Mode", "Start_Time", "End_Time", "Duration_Sec",
        "Status", "Items_Found", "Operational_Parameters", "JOB_COMMENT"
    ],
    "Datasets": [
        "DatasetID", "Canonical Name", "Variant Name", "Aliases", "Type",
        "Num Time Points", "Num Time Points (C)",
        "Num Locations/Series", "Num Locations/Series (C)",
        "Total Variables", "Total Variables (C)",
        "Variables per Location", "Variables per Location (C)",
        "Frequency", "Frequency (C)",
        "Domain", "Domain (C)",
        "Description",
        "Primary Creator",
        "Primary URL", "Primary URL (C)",
        "Link to Data (Actual Source)", "Link to Data (Actual Source) (C)",
        "Other URL", "Other URL (C)",
        "License", "Overall Confidence",
        "Job_Created", "Date_Created", "Project_Created",
        "Job_Updated", "Date_Updated", "Project_Updated"
    ],
    "Projects": [
        "ProjectID", "Name", "Description", "Type",
        "Last Analyzed", "Source URL"
    ],
    "Project_Dataset_Link": [
        "LinkID", "ProjectID", "DatasetID", "Actual Data URL Used",
        "Data Preparation Comments", "CitationID", "Assignment Confidence",
        "Link_Date", "Linked_By_Job"
    ],
    "Datasets_Quarantined": [
        "DatasetID", "Canonical Name", "Variant Name", "Aliases", "Type",
        "Num Time Points", "Num Time Points (C)",
        "Num Locations/Series", "Num Locations/Series (C)",
        "Total Variables", "Total Variables (C)",
        "Variables per Location", "Variables per Location (C)",
        "Frequency", "Frequency (C)",
        "Domain", "Domain (C)",
        "Description",
        "Primary Creator",
        "Primary URL", "Primary URL (C)",
        "Link to Data (Actual Source)", "Link to Data (Actual Source) (C)",
        "Other URL", "Other URL (C)",
        "License", "Overall Confidence",
        "Job_Created", "Date_Created", "Project_Created",
        "Job_Updated", "Date_Updated", "Project_Updated"
    ],
    "Links_Quarantined": [
        "LinkID", "ProjectID", "DatasetID", "Actual Data URL Used",
        "Data Preparation Comments", "CitationID", "Assignment Confidence",
        "Link_Date", "Linked_By_Job"
    ],
    "Datasets_Duplicates_Log": [
        "DatasetID (Dropped)", "DatasetID (Kept)", "Reason", "Merge Notes", "Dropped Name", "Kept Name"
    ],
    "Citations": [
        "CitationID", "Title", "Authors", "Venue", "Year", "DOI", "URL",
        "Full Citation Text"
    ],
    "WebLinks": ["WebLinkID", "URL", "Resource Type", "Description"]
}

# Internal Agent Schema (Mappings for RAG)
CATALOG_SCHEMA = {
    # Identifiers
    "Dataset Name": {"description": "The specific name of the dataset variant (e.g., ETTh1)."},
    "Canonical Name": {
        "description": "The standardized, primary group name (e.g., ETT, M4, PeMS).",
        "query": "What is the primary group, family, or canonical name for the {name} dataset? (e.g. if 'ETTh1', answer 'ETT'; if 'M4 Hourly', answer 'M4'). If it has no group, return the dataset name itself."
    },
    "Aliases": {"description": "Other names or variants related to the canonical name."},
    "Type": {
        "description": "The classification of the entity.",
        "query": "Classify the entity '{name}'. Is it a [Real-World Dataset | Synthetic Dataset | Synthetic Generator | Augmentation Tool | Evaluation Script | Collection | Provider]? Reply STRICTLY with exactly ONE of these options."
    },

    # Assignment Confidence
    "Assignment Confidence": {"description": "Confidence (C_relevance) that the dataset is relevant to the current project context."},
    "Assignment Rationale": {"description": "Justification for the relevance of the dataset to the project."},

    # Core Metadata
    "Domain": {"description": "The area of application (e.g., Finance, Energy).", "query": "What is the domain or area of application for the {name} dataset?"},
    "Detailed Description": {"description": "A single-line summary.", "query": "Provide a detailed, single-line description of the contents and context of the {name} dataset."},

    # Time Series Specifications
    "Time interval between points": {"description": "Frequency (e.g., 1 hour, 15 minutes).", "query": "What is the frequency or time interval between successive points specifically for the version of the {name} dataset used in this context?"},
    "Number of Time Points": {"description": "Length or total samples.", "query": "What is the total length (number of time points or samples at a given location) specifically for the version of the {name} dataset used in this context? Must be a number or range."},
    "Number of Locations/Series": {"description": "The number of distinct spatial units or clients.", "query": "How many distinct locations, clients, or independent time series are included in the specific version of the {name} dataset used here? Must be a number or range."},
    "Variables per Location": {
        "description": "The number of features measured at each location.",
        "query": "How many distinct variables/features/channels are measured at EACH single location/time-step for the {name} dataset? CRITICAL: Do NOT count the number of time points (length). If the shape is (1000, 6), the answer is 6."
    },
    "Total Variables": {
        "description": "The total dimensionality.",
        "query": "What is the total number of variables or features for the {name} dataset? CRITICAL: Do NOT count the number of time steps (length). If the dataset has 1000 time steps and 5 variables, the answer is 5. If strictly univariate, answer 1."
    },

    # Grounding & Provenance (URL UPDATES HERE)
    "Primary Source Repository": {
        "description": "The official originator or primary host.",
        "query": "CRITICAL: Identify the official institution or repository that *originated* the {name} dataset. Provide the Name."
    },
    "Primary URL": {
        "description": "Home Page for the dataset.",
        "query": "What is the official Home Page URL for the {name} dataset? MUST be a true URL. ABSOLUTELY NO links to Google Scholar, Vertex AI, or generic search results. If multiple, separate by commas."
    },
    "Link to Data (Actual Source)": {
        "description": "Link(s) directly to the DATASET LOCATION from which the data files can be downloaded.",
        "query": "Provide the direct download URL(s) or repository link(s) (e.g., GitHub, UCI, PhysioNet, Kaggle) for the actual data files of the {name} dataset. MUST be a true URL. ABSOLUTELY NO Google Scholar or Vertex AI links. If multiple, separate by commas."
    },
    "Other URL": {
        "description": "Academic papers, supplementary GitHub repositories, and any other useful links.",
        "query": "Provide academic paper URLs, documentation, or other supplementary links for the {name} dataset. MUST be true URLs. ABSOLUTELY NO Google Scholar or Vertex AI links. If multiple, separate by commas."
    },
    "Comments on Data Preparation": {"description": "Notes on modifications or preprocessing.", "query": "Describe any modifications, preprocessing, or version differences for the {name} dataset compared to its primary source. If standard, state 'Standard version'."},

    "Project Citations": {"description": "Academic papers associated with the project."},
    "Project WebLinks": {"description": "Web resources (GitHub, Homepages) associated with the project."},
}

GROUNDING_FIELDS = ["Primary Source Repository", "Primary URL", "Link to Data (Actual Source)", "Other URL", "Comments on Data Preparation"]
EXTRACTED_FIELDS = [field for field, definition in CATALOG_SCHEMA.items() if 'query' in definition]
ASSIGNMENT_FIELDS = ["Dataset Name", "Assignment Confidence", "Assignment Rationale"]

DDI_INSPECTABLE_FIELDS = [
    "Variables per Location", "Total Variables",
    "Number of Locations/Series", "Number of Time Points"
]

MISSING_DATA_PLACEHOLDERS = {
    "", "[missing]", "unknown", "n/a", "not found", "not specified",
    "not available", "not mentioned", "none", "missing", "nan", "null",
    "[implausible]", "[error]", "[stability_error]", "[timeout_error]", "[server_error]", "[retrieval_error]",
    "[extraction_error: dict_returned]", "[extraction_error: complex_string]"
}

PLAUSIBILITY_THRESHOLDS = {
    "Variables per Location": {"min": 1, "max": 50000},
    "Number of Locations/Series": {"min": 1, "max": 10000000},
    "Number of Time Points": {"min": 1, "max": 100000000},
    "Total Variables": {"min": 1, "max": 1000000000}
}
print("✅ deepcollector/config/schema.py written (V83: URL Separation).")

Writing deepcollector/config/schema.py


#### **Cell 3: `deepcollector/config/settings.py` (AppConfig)**

This defines the `AppConfig` dataclass, replacing global variables.

In [ ]:
%%writefile deepcollector/config/settings.py
# =============================================================================
# V92: Settings (Dual Folder ID Configuration - Fully Expanded & Uncompressed)
# =============================================================================
import os
import re
import importlib
import dataclasses
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any

# Force reload of schema to pick up changes immediately
import deepcollector.config.schema as schema_module
importlib.reload(schema_module)

from deepcollector.config.schema import (
    KB_SCHEMA,
    CATALOG_SCHEMA,
    GROUNDING_FIELDS,
    EXTRACTED_FIELDS,
    ASSIGNMENT_FIELDS,
    DDI_INSPECTABLE_FIELDS,
    MISSING_DATA_PLACEHOLDERS,
    PLAUSIBILITY_THRESHOLDS,
    KB_SCHEMA_VERSION
)

def check_import(module_name: str) -> bool:
    try:
        __import__(module_name)
        return True
    except ImportError:
        return False

PROFILES = {
    "BALANCED_STABILITY": {
        "ARCHITECTURE": "ROBUST",
        "PARALLEL_CONCURRENCY_LIMIT": 4,
        "CELLULAR_RAG_BATCH_SIZE": 50,
        "INDEXING_BATCH_SIZE": 8,
        "INDEXING_THROTTLE_DELAY": 1.5,
        "CELLULAR_RAG_THROTTLE_DELAY": 0.75,
    },
    "LOW_CONCURRENCY_ROBUST": {
        "ARCHITECTURE": "ROBUST",
        "PARALLEL_CONCURRENCY_LIMIT": 2,
        "CELLULAR_RAG_BATCH_SIZE": 30,
        "INDEXING_BATCH_SIZE": 5,
        "INDEXING_THROTTLE_DELAY": 3.0,
        "CELLULAR_RAG_THROTTLE_DELAY": 1.5,
    },
     "HIGH_THROUGHPUT_ROBUST": {
        "ARCHITECTURE": "ROBUST",
        "PARALLEL_CONCURRENCY_LIMIT": 10,
        "CELLULAR_RAG_BATCH_SIZE": 100,
        "INDEXING_BATCH_SIZE": 15,
        "INDEXING_THROTTLE_DELAY": 0.5,
        "CELLULAR_RAG_THROTTLE_DELAY": 0.25,
    },
    "STREAMLINED_BALANCED": {
        "ARCHITECTURE": "STREAMLINED",
        "PARALLEL_CONCURRENCY_LIMIT": 6,
        "CELLULAR_RAG_BATCH_SIZE": 60,
        "INDEXING_BATCH_SIZE": 10,
        "INDEXING_THROTTLE_DELAY": 1.0,
        "CELLULAR_RAG_THROTTLE_DELAY": 0.5,
    },
    "VLLM_HIGH_THROUGHPUT": {
        "ARCHITECTURE": "ROBUST",
        "PARALLEL_CONCURRENCY_LIMIT": 20,
        "CELLULAR_RAG_BATCH_SIZE": 100,
        "INDEXING_BATCH_SIZE": 15,
        "INDEXING_THROTTLE_DELAY": 0.1,
        "CELLULAR_RAG_THROTTLE_DELAY": 0.1,
    }
}

@dataclass
class AppConfig:
    CURRENT_PROJECT_ID: Optional[str] = None
    CURRENT_PROJECT_NAME: Optional[str] = None
    INITIAL_URLS: List[str] = field(default_factory=list)
    PROJECT_CONTEXT: Optional[str] = None
    PROJECT_METHOD: int = 1
    HARVESTER_OVERRIDE_URL: Optional[str] = None
    USER_CATALOG_URL: Optional[str] = None

    VERBOSITY_LEVEL: int = 1
    PROFILE_NAME: str = "BALANCED_STABILITY"
    DDI_STRESS_TEST_MODE: bool = False
    EXPORT_TO_NEW_SHEET: bool = True

    # --- HYBRID BACKEND & EXPORT TOGGLES ---
    LLM_BACKEND: str = os.environ.get("DEEPCOLLECTOR_LLM_BACKEND", "GEMINI")
    OUTPUT_FORMAT: str = os.environ.get("DEEPCOLLECTOR_OUTPUT_FORMAT", "SHEET")
    USE_vLLM: bool = os.environ.get("DEEPCOLLECTOR_USE_VLLM", "False") == "True"

    # --- SEARCH TOOLS ---
    SEARCH_BACKEND: str = os.environ.get("DEEPCOLLECTOR_SEARCH_BACKEND", "GEMINI")
    SEARXNG_URL: str = os.environ.get("DEEPCOLLECTOR_SEARXNG_URL", "http://localhost:8080")
    SEARCH_NUM_RESULTS: int = 7

    # --- CONFIGURABLE DRIVE LOCATIONS ---
    # 1. Folder for Spreadsheet/Catalog CSV Exports
    GOOGLE_DRIVE_SHEET_FOLDER_ID: str = "15nvgAVcvoigcQUyla9T9A66KDdj7ZPoK"

    # 2. Folder for System/Console Logs
    GOOGLE_DRIVE_LOG_FOLDER_ID: str = "1mw42keL9BNmaNgrW_ssDFGxe8_lcXBQ2"

    # 3. Knowledge Injection Registry Doc
    KNOWLEDGE_MASTER_DOC_URL: str = "https://docs.google.com/document/d/16oN5NyOC2lFBQvLgept4y9yiTUbgKqiz-mLRXnokefw/export?format=html"

    # 4. Deep Research Model Target
    DEEP_RESEARCH_AGENT_MODEL: str = "deep-research-pro-preview-12-2025"

    # --- RAG TUNING PARAMETERS (Context Window Management) ---
    RAG_DISCOVERY_TOP_K: int = 15
    RAG_DISCOVERY_MAX_CHARS: int = 45000
    RAG_CELLULAR_TOP_K: int = 10
    RAG_CELLULAR_MAX_CHARS: int = 35000
    RAG_CELLULAR_FALLBACK_CHARS: int = 15000

    # --- Deep Research Settings ---
    ENABLE_DEEP_RESEARCH: bool = True
    DEEP_RESEARCH_TIMEOUT_MINUTES: int = 45
    DEEP_RESEARCH_MAX_RETRIES: int = 6
    ABORT_ON_DEEP_RESEARCH_FAILURE: bool = True
    FORCE_ASYNC_POLLING: bool = False
    DEEP_RESEARCH_POLLING_INTERVAL_SECONDS: int = 15
    ENABLE_LOCAL_DEEP_RESEARCH: bool = True

    # --- Advanced RAG Engine Features ---
    ENABLE_PREFLIGHT_CRAWLER: bool = True
    ENABLE_ARBITRATION_PROMPT: bool = True
    ENABLE_STRICT_TAXONOMY: bool = True
    ENABLE_VARIANT_MAPPING: bool = True
    ENABLE_MULTI_QUERY_RAG: bool = True
    ENABLE_GOLDEN_FASTPATH: bool = True

    # --- MISSING KWARGS RESTORED HERE ---
    ENABLE_SINGLETON_VERIFICATION: bool = True
    ENABLE_ORACLE_SEARCH: bool = True
    SCRAPING_METHOD: int = 1
    WIPE_CURRENT_PROJECT_ONLY: bool = False
    _CUDA_OOM_ABORT: bool = False
    LLM_ARBITRATION_LIMIT: int = 2500

    MIN_ASSIGNMENT_CONFIDENCE_GATE: float = 0.40
    CONFIDENCE_THRESHOLD: float = 0.80
    CONFIDENCE_LOCK_THRESHOLD: float = 0.95
    GROUNDING_CONFIDENCE_THRESHOLD: float = 0.90

    # --- Iteration limits (Now configurable via __init__) ---
    MAX_DISCOVERY_ITERATIONS: int = 2
    MAX_GROUNDING_ITERATIONS: int = 3
    MAX_EXTRACTION_ITERATIONS: int = 8

    GOOGLE_SHEET_KB_INPUT: Optional[str] = None
    GOOGLE_SHEET_HINTS_INPUT: Optional[str] = None
    GOOGLE_SHEET_PROJECT_LIST_INPUT: Optional[str] = None
    KNOWLEDGE_INJECTION_DATA: List[Dict[str, str]] = field(default_factory=list)
    SECRETS: Dict[str, Optional[str]] = field(default_factory=dict)

    DATA_INSPECTION_ENABLED: bool = True
    MAX_DOWNLOAD_PREVIEW_BYTES: int = 1024 * 1024 * 1
    MAX_DOWNLOAD_ARCHIVE_BYTES: int = 1024 * 1024 * 50
    INCLUDE_RAG_TELEMETRY_IN_REPORT: bool = False

    # --- Runtime Flags ---
    IN_COLAB: bool = False
    GSPREAD_AVAILABLE: bool = False
    LLAMA_INDEX_AVAILABLE: bool = False
    PdfReader: Optional[Any] = None
    BM25Retriever: Optional[Any] = None
    UCIMLREPO_AVAILABLE: bool = False

    EXECUTION_ARCHITECTURE: str = field(init=False, default="UNKNOWN")
    GOOGLE_SHEET_KB_ID: Optional[str] = field(init=False, default=None)
    GOOGLE_SHEET_HINTS_ID: Optional[str] = field(init=False, default=None)
    GOOGLE_SHEET_PROJECT_LIST_ID: Optional[str] = field(init=False, default=None)

    PARALLEL_CONCURRENCY_LIMIT: int = field(init=False, default=4)
    CELLULAR_RAG_BATCH_SIZE: int = field(init=False, default=50)
    INDEXING_BATCH_SIZE: int = field(init=False, default=8)
    INDEXING_THROTTLE_DELAY: float = field(init=False, default=1.5)
    CELLULAR_RAG_THROTTLE_DELAY: float = field(init=False, default=0.75)

    @property
    def KB_SCHEMA(self):
        return KB_SCHEMA

    @property
    def KB_SCHEMA_VERSION(self):
        return KB_SCHEMA_VERSION

    @property
    def CATALOG_SCHEMA(self):
        return CATALOG_SCHEMA

    @property
    def GROUNDING_FIELDS(self):
        return GROUNDING_FIELDS

    @property
    def EXTRACTED_FIELDS(self):
        return EXTRACTED_FIELDS

    @property
    def ASSIGNMENT_FIELDS(self):
        return ASSIGNMENT_FIELDS

    @property
    def DDI_INSPECTABLE_FIELDS(self):
        return DDI_INSPECTABLE_FIELDS

    @property
    def MISSING_DATA_PLACEHOLDERS(self):
        return MISSING_DATA_PLACEHOLDERS

    @property
    def PLAUSIBILITY_THRESHOLDS(self):
        return PLAUSIBILITY_THRESHOLDS

    def __post_init__(self):
        if self.CURRENT_PROJECT_ID:
            self.CURRENT_PROJECT_ID = self.CURRENT_PROJECT_ID.upper()

        self._apply_profile()
        self._calculate_iterations()
        self._process_sheet_ids()

        self.IN_COLAB = check_import("google.colab")
        self.GSPREAD_AVAILABLE = check_import("gspread")
        self.LLAMA_INDEX_AVAILABLE = check_import("llama_index.core")
        self.UCIMLREPO_AVAILABLE = check_import("ucimlrepo")

        try:
            from pypdf import PdfReader
            self.PdfReader = PdfReader
        except ImportError:
            self.PdfReader = None

        try:
            from llama_index.retrievers.bm25 import BM25Retriever
            self.BM25Retriever = BM25Retriever
        except ImportError:
            self.BM25Retriever = None

    def get_operational_report(self) -> Dict[str, Any]:
        report = {}
        for f in dataclasses.fields(self):
            k = f.name
            v = getattr(self, k)
            if k == 'SECRETS':
                report[k] = "[REDACTED FOR SECURITY]"
            elif isinstance(v, type):
                # Type Serialization fix to prevent JSON crash when writing log
                report[k] = str(v)
            elif isinstance(v, list) and k in ['KNOWLEDGE_INJECTION_DATA', 'INITIAL_URLS']:
                report[k] = f"[{len(v)} items]"
            else:
                report[k] = v
        return report

    def is_complete(self) -> bool:
        if not self.CURRENT_PROJECT_ID or not self.PROJECT_CONTEXT or not self.CURRENT_PROJECT_NAME:
            return False
        return True

    def validate_configuration(self) -> bool:
        if self.is_complete():
            if self.INITIAL_URLS is None:
                self.INITIAL_URLS = []
            return True

        if self.VERBOSITY_LEVEL >= 0:
             print(f"❌ CRITICAL: Missing essential project configuration.")
        return False

    def recalculate_runtime_parameters(self):
        self._calculate_iterations()
        self._process_harvester_override()

    def _process_harvester_override(self):
        self.HARVESTER_OVERRIDE_URL = None
        initial_urls = self.INITIAL_URLS or []

        if self.PROJECT_METHOD > 1 and initial_urls and len(initial_urls) == 1:
            self.HARVESTER_OVERRIDE_URL = initial_urls[0]

    def _apply_profile(self):
        if self.LLM_BACKEND in ["LOCAL_PRO", "LOCAL_CLASSROOM"]:
            self.PROFILE_NAME = "VLLM_HIGH_THROUGHPUT" if getattr(self, 'USE_vLLM', False) else "LOCAL_GPU_SAFE"
        elif self.PROFILE_NAME not in PROFILES:
             self.PROFILE_NAME = "BALANCED_STABILITY"

        profile = PROFILES.get(self.PROFILE_NAME)
        self.EXECUTION_ARCHITECTURE = profile["ARCHITECTURE"]
        self.PARALLEL_CONCURRENCY_LIMIT = profile["PARALLEL_CONCURRENCY_LIMIT"]
        self.CELLULAR_RAG_BATCH_SIZE = profile["CELLULAR_RAG_BATCH_SIZE"]
        self.INDEXING_BATCH_SIZE = profile["INDEXING_BATCH_SIZE"]
        self.INDEXING_THROTTLE_DELAY = profile["INDEXING_THROTTLE_DELAY"]
        self.CELLULAR_RAG_THROTTLE_DELAY = profile.get("CELLULAR_RAG_THROTTLE_DELAY", 0.5)

    def _calculate_iterations(self):
        if self.EXECUTION_ARCHITECTURE == "STREAMLINED":
            self.MAX_DISCOVERY_ITERATIONS = 1
            self.MAX_GROUNDING_ITERATIONS = 0
            self.MAX_EXTRACTION_ITERATIONS = 6
        else:
            self.MAX_DISCOVERY_ITERATIONS = 2
            self.MAX_GROUNDING_ITERATIONS = 3
            self.MAX_EXTRACTION_ITERATIONS = 8

        if self.PROJECT_METHOD > 1:
             self.MAX_DISCOVERY_ITERATIONS = 0
             if self.EXECUTION_ARCHITECTURE == "STREAMLINED":
                  self.MAX_EXTRACTION_ITERATIONS = 5

    def _process_sheet_ids(self):
        self.GOOGLE_SHEET_KB_ID = self._extract_sheet_id(self.GOOGLE_SHEET_KB_INPUT)
        self.GOOGLE_SHEET_HINTS_ID = self._extract_sheet_id(self.GOOGLE_SHEET_HINTS_INPUT)
        self.GOOGLE_SHEET_PROJECT_LIST_ID = self._extract_sheet_id(self.GOOGLE_SHEET_PROJECT_LIST_INPUT)

    def _extract_sheet_id(self, input_str):
        if not input_str or "YOUR_GOOGLE_SHEET_ID_HERE" in input_str:
            return None

        input_str = str(input_str).strip()
        match = re.search(r'/spreadsheets/d/([a-zA-Z0-9-_]+)', input_str)

        if match:
            return match.group(1)

        if re.match(r'^[a-zA-Z0-9-_]{30,}$', input_str):
            return input_str

        return None

print("✅ deepcollector/config/settings.py written (100% Fully Expanded & Uncompressed).")

Writing deepcollector/config/settings.py


#### **Cell 4: `deepcollector/utils/profiler.py`**

Profiling and Logs

In [ ]:
%%writefile deepcollector/utils/profiler.py
# =============================================================================
# Profiler Utility
# =============================================================================
import time
from collections import defaultdict
from functools import wraps
import pandas as pd

class Profiler:
    """A simple profiling utility to track execution time and counts."""
    def __init__(self):
        self.stats = defaultdict(lambda: {'count': 0, 'total_time': 0.0})

    def track(self, category):
        """Decorator to track the execution of a function."""
        def decorator(func):
            @wraps(func)
            def wrapper(*args, **kwargs):
                start_time = time.time()
                try:
                    result = func(*args, **kwargs)
                    return result
                finally:
                    end_time = time.time()
                    duration = end_time - start_time
                    self.update_stats(category, duration)
            return wrapper
        return decorator

    def update_stats(self, category, duration, count=1):
        """Updates the statistics for a given category."""
        self.stats[category]['count'] += count
        self.stats[category]['total_time'] += duration

    def get_report(self) -> pd.DataFrame:
        """Generates a pandas DataFrame report of the profiling statistics."""
        if not self.stats:
            return pd.DataFrame(columns=['Category', 'Count', 'Total Time (s)', 'Avg Time (s)'])

        report_data = []
        for category, data in self.stats.items():
            avg_time = data['total_time'] / data['count'] if data['count'] > 0 else 0
            report_data.append({
                'Category': category,
                'Count': data['count'],
                'Total Time (s)': data['total_time'],
                'Avg Time (s)': avg_time
            })

        df = pd.DataFrame(report_data)
        # Sort by Total Time descending
        df = df.sort_values(by='Total Time (s)', ascending=False).reset_index(drop=True)
        # Format floating point numbers
        df['Total Time (s)'] = df['Total Time (s)'].map('{:.3f}'.format)
        df['Avg Time (s)'] = df['Avg Time (s)'].map('{:.4f}'.format)
        return df

    def reset(self):
        """Resets the profiler statistics."""
        self.stats = defaultdict(lambda: {'count': 0, 'total_time': 0.0})

# Create a global instance for easy access
profiler = Profiler()
print("✅ deepcollector/utils/profiler.py written.")

Writing deepcollector/utils/profiler.py


#### Cell 5: `deepcollector/utils/initialization.py`

Handles API setup, retries, and Gemini, Search LlamaIndex configuration.

In [ ]:
%%writefile deepcollector/utils/initialization.py
# =============================================================================
# V185: Initialization (DGX / Gemma-Compatible & Warnings Suppressed)
# =============================================================================
import os
import sys
import time
import warnings
import logging
import requests
import gc
from typing import Tuple, Dict, Any, List, Optional

try:
    import torch
except ImportError:
    torch = None

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8,max_split_size_mb:32"

# --- SILENCE NOISY THIRD-PARTY WARNINGS ---
warnings.filterwarnings("ignore", category=UserWarning, module="google.generativeai")
warnings.filterwarnings("ignore", category=FutureWarning, module="google.generativeai")
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")
warnings.filterwarnings("ignore", module="llama_index.retrievers.bm25.base")
warnings.filterwarnings("ignore", module="pypdf._reader")
warnings.filterwarnings("ignore", module="google_auth_httplib2")
warnings.filterwarnings("ignore", message=".*Interactions usage is experimental.*")

logging.getLogger('bm25s').setLevel(logging.ERROR)
logging.getLogger('pypdf').setLevel(logging.ERROR)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# =============================================================================
# --- SEARXNG CLIENT ---
# =============================================================================
class SearXNGClient:
    """Robust client for interacting with SearXNG proxy arrays."""
    def __init__(self, base_url="http://localhost:8080"):
        self.base_url = base_url.rstrip("/")
        self.fallback_urls = [
            "https://searx.be",
            "https://searx.tiekoetter.com",
            "https://search.ononoki.org",
            "https://searx.work"
        ]

    def search(self, query: str, max_results: int = 7) -> List[Dict[str, str]]:
        print(f"    🌐 [Tool: SearXNG] Executing Query: '{query}'")

        urls_to_try = [self.base_url] + self.fallback_urls

        for url in urls_to_try:
            try:
                params = {
                    "q": query,
                    "format": "json",
                    "engines": "google,bing,duckduckgo,wikipedia,github",
                    "language": "en"
                }

                response = requests.get(f"{url}/", params=params, headers=HEADERS, timeout=15)

                if response.status_code == 404:
                    response = requests.get(f"{url}/search", params=params, headers=HEADERS, timeout=15)

                response.raise_for_status()
                data = response.json()

                results = []
                for item in data.get("results", [])[:max_results]:
                    results.append({
                        "title": item.get("title", ""),
                        "url": item.get("url", ""),
                        "content": item.get("content", "")
                    })

                if results:
                    if url != self.base_url:
                        print(f"       ⚠️ Localhost 404/Refused. Automatically failed over to public instance: {url}")
                    print(f"       ✅ SearXNG returned {len(results)} usable results.")
                    return results

            except Exception as e:
                continue

        print(f"       ❌ SearXNG Search failed across all attempted instances (Local & Public).")
        return []

# =============================================================================
# --- RETRY STRATEGIES ---
# =============================================================================
def get_network_retry_strategy(verbosity=1):
    from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
    from requests.exceptions import RequestException, ConnectionError, Timeout, HTTPError

    def log_retry(retry_state):
        if verbosity >= 2:
            print(f"    ⚠️ [Network] Retry {retry_state.attempt_number}...")

    return retry(
        retry=retry_if_exception_type((ConnectionError, Timeout, HTTPError, RequestException)),
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        before_sleep=log_retry,
        reraise=True
    )

def get_gemini_retry_strategy(verbosity=1):
    from tenacity import retry, stop_after_attempt, wait_exponential

    def log_retry(retry_state):
        if verbosity >= 1:
            ex = retry_state.outcome.exception()
            if ex:
                msg = str(ex)[:100]
            else:
                msg = "Unknown"
            print(f"    ⚠️ [LLM API] Retry {retry_state.attempt_number}: {msg}...")

    return retry(
        stop=stop_after_attempt(5),
        wait=wait_exponential(multiplier=2, min=2, max=60),
        before_sleep=log_retry,
        reraise=True
    )

class SystemMonitor:
    @staticmethod
    def print_vram_status(label=""):
        try:
            if torch is not None and torch.cuda.is_available():
                reserved = torch.cuda.memory_reserved(0) / (1024**3)
                allocated = torch.cuda.memory_allocated(0) / (1024**3)
                total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
                free = total - reserved
                print(f"       📊 VRAM Status [{label}]: {allocated:.1f}GB Allocated | {reserved:.1f}GB Reserved | {free:.1f}GB Free (out of {total:.1f}GB)")
        except Exception:
            pass

# =============================================================================
# --- CORE INITIALIZATION ---
# =============================================================================
def initialize_apis(config: Any) -> Tuple[Dict[str, str], Any]:
    keys = {}
    backend = getattr(config, 'LLM_BACKEND', 'GEMINI')
    use_vllm = getattr(config, 'USE_vLLM', False)

    # 1. ALWAYS Initialize Gemini (Required for Agentic Planning/Routing)
    api_key = config.SECRETS.get("GEMINI_API_KEY")
    if not api_key:
        try:
            from google.colab import userdata
            api_key = userdata.get("GEMINI_API_KEY")
        except ImportError:
            pass
        except Exception:
            pass

    if not api_key:
        print("❌ GEMINI_API_KEY not found in Secrets or Config.")
        sys.exit("🛑 ABORTING: Missing Gemini API Key.")

    keys["GEMINI"] = api_key
    os.environ["GOOGLE_API_KEY"] = api_key

    try:
        from google import genai
        client = genai.Client(api_key=api_key)

        candidates_pro = [
            "gemini-3.1-pro-preview",
            "gemini-3-pro-preview",
            "gemini-2.5-pro",
            "gemini-pro-latest"
        ]

        candidates_flash = [
            "gemini-3.5-flash",
            "gemini-3-flash-preview",
            "gemini-2.5-flash",
            "gemini-2.0-flash",
            "gemini-2.0-flash-lite"
        ]

        def verify_pool(model_list, pool_name):
            verified = []
            print(f"    🔍 Verifying {pool_name} Pool...")

            for model in model_list:
                try:
                    # Quick ping to check access permissions
                    client.models.count_tokens(model=model, contents="Test")
                    verified.append(model)
                except Exception:
                    pass

            if verified:
                print(f"       ✅ Verified {len(verified)} models: {verified}")
            else:
                print(f"       ❌ No models verified in {pool_name} pool.")

            return verified

        verified_pro = verify_pool(candidates_pro, "Pro (Reasoning)")
        verified_flash = verify_pool(candidates_flash, "Flash (Extraction)")

        if not verified_pro and not verified_flash:
            raise RuntimeError("CRITICAL: ALL models failed verification. Check API Key validity.")

        if not verified_pro:
            print("    ⚠️ WARNING: Pro pool empty. Promoting Flash models to Pro role.")
            verified_pro = verified_flash

        if not verified_flash:
            print("    ⚠️ WARNING: Flash pool empty. Using Pro models for extraction.")
            verified_flash = verified_pro

        class Models:
            CLIENT = client
            # Use only the verified lists
            PRO_POOL = verified_pro
            FLASH_POOL = verified_flash

            CURRENT_PRO_INDEX = 0
            CURRENT_FLASH_INDEX = 0

            # Roles - Defaults to the leader of the verified pool
            MODEL_PLANNER = verified_pro[0]
            MODEL_SYNTHESIZER = verified_pro[0]
            MODEL_STANDARD = verified_flash[0]
            MODEL_RAG_PRIMARY = verified_flash[0]

        print(f"    ✅ Gemini Client initialized.")
        print(f"    🎭 Active Roles:")
        print(f"       - Planner/Search: {Models.MODEL_PLANNER}")
        print(f"       - RAG Extraction: {Models.MODEL_RAG_PRIMARY}")

        return keys, Models

    except ImportError:
        sys.exit("🛑 ABORTING: google-genai SDK not installed.")
    except Exception as e:
        sys.exit(f"🛑 ABORTING: Failed to initialize Gemini Client: {e}")

# -----------------------------------------------------------------------------
# 5. Native Embedding Class
# -----------------------------------------------------------------------------
try:
    from llama_index.core.base.embeddings.base import BaseEmbedding
    from llama_index.core.bridge.pydantic import PrivateAttr
    from google import genai

    class NativeGeminiEmbedding(BaseEmbedding):
        """
        Direct wrapper for google-genai SDK embeddings to bypass
        dependency issues with official llama-index-embeddings-gemini.
        """
        _client: Any = PrivateAttr()
        _model_name: str = PrivateAttr()

        def __init__(self, api_key: str, model_name: str = "models/text-embedding-004", **kwargs: Any) -> None:
            super().__init__(**kwargs)
            self._model_name = model_name
            self._client = genai.Client(api_key=api_key)

        @classmethod
        def class_name(cls) -> str:
            return "NativeGeminiEmbedding"

        def _get_query_embedding(self, query: str) -> List[float]:
            return self._embed(query)

        async def _aget_query_embedding(self, query: str) -> List[float]:
            return self._embed(query)

        def _get_text_embedding(self, text: str) -> List[float]:
            return self._embed(text)

        async def _aget_text_embedding(self, text: str) -> List[float]:
            return self._embed(text)

        def _embed(self, text: str) -> List[float]:
            try:
                response = self._client.models.embed_content(model=self._model_name, contents=text)
                return response.embeddings[0].values
            except Exception as e:
                raise ValueError(f"Embedding failed: {e}")

except ImportError:
    NativeGeminiEmbedding = None

# -----------------------------------------------------------------------------
# 6. LlamaIndex Configuration
# -----------------------------------------------------------------------------
def configure_llama_index(config, models, keys) -> bool:
    if not getattr(config, 'LLAMA_INDEX_AVAILABLE', False):
        print("    ℹ️  LlamaIndex not available in environment.")
        return False

    print("🔧 Configuring LlamaIndex (Native Interface)...")

    try:
        from llama_index.core import Settings
        from llama_index.llms.gemini import Gemini

        if models and models.FLASH_POOL:
            default_model = models.FLASH_POOL[0]
            # Safety wrapper for LlamaIndex which strictly expects the "models/" prefix
            if not default_model.startswith("models/"):
                default_model = f"models/{default_model}"
        else:
            default_model = "models/gemini-2.5-flash"

        llm = Gemini(model_name=default_model, api_key=keys.get("GEMINI", ""), temperature=0.1)
        Settings.llm = llm

        active_embed = None

        if NativeGeminiEmbedding:
            embedding_candidates = [
                "models/text-embedding-004",
                "models/gemini-embedding-001",
                "models/embedding-001"
            ]

            print("    🔍 Initializing Native Embedding Model (Cascade)...")

            for model_name in embedding_candidates:
                try:
                    candidate = NativeGeminiEmbedding(api_key=keys.get("GEMINI", ""), model_name=model_name)

                    # Verify it actually works by attempting to embed a short string
                    vec = candidate.get_text_embedding("Startup Verification")

                    if vec and len(vec) > 0:
                        active_embed = candidate
                        print(f"      ✅ Selected: {model_name} (via google.genai)")
                        break
                except Exception:
                    pass

        if active_embed:
            Settings.embed_model = active_embed
            print("    ✅ LlamaIndex Configured Successfully.")
            return True
        else:
            print("    ❌ CRITICAL: All Gemini embedding models failed to connect.")
            return False

    except ImportError as e:
        print(f"\n🛑 CRITICAL ERROR: LlamaIndex Dependency Missing: {e}")
        return False
    except Exception as e:
        print(f"    ❌ LlamaIndex Global Config Error: {e}")
        return False

print("✅ deepcollector/utils/initialization.py written (Warnings Suppressed).")

Writing deepcollector/utils/initialization.py


#### **Cell 6: DDI `deepcollector/tools/ddi.py`**

In [ ]:
%%writefile deepcollector/tools/ddi.py
import pandas as pd
import io
import zipfile
import requests
# Import necessary typing hints
from typing import Dict, Any, Optional, Tuple, List
from urllib.parse import urljoin
import re
import textwrap
import sys
import time

# Attempt to import BeautifulSoup
try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None

# Import utilities
# Robust imports to handle potential failures if previous cells didn't run
try:
    from deepcollector.utils.profiler import profiler
    from deepcollector.utils.initialization import get_network_retry_strategy, HEADERS
except ImportError as e:
    print(f"❌ CRITICAL: Failed to import utilities in ddi.py: {e}.")
    # Define placeholders
    class DummyProfiler:
        def track(self, category):
            def decorator(func): return func
            return decorator
    profiler = DummyProfiler()
    get_network_retry_strategy = lambda x: lambda f: f
    HEADERS = {}


class DataInspectionTools:
    """Handles direct inspection of data files and navigation of directory listings."""

    # (Class attributes identical to previous versions)
    PARSABLE_EXTENSIONS = ['.csv', '.tsv', '.txt', '.data']
    ARCHIVE_EXTENSIONS = ['.zip']
    COMPRESSION_EXTENSIONS = ['.gz', '.bz2']

    DIRECTORY_LISTING_PRIORITY = {
        '.zip': 100, '.tar.gz': 95, '.tgz': 95, '.rar': 90,
        '.data': 80, '.csv': 75, '.txt': 70, '.arff': 60,
        '.names': 10, '.info': 10,
    }

    IGNORE_PATTERNS = [
        r'^/$', r'^\\?C=', r'parent directory', r'name', r'last modified', r'size', r'description',
        r'\\.pdf$', r'\\.md$', r'readme', r'index\\.html', r'license',
        r'\\.jpg$', r'\\.png$', r'\\.gif$'
    ]

    class SizeLimitExceeded(Exception):
        pass

    def __init__(self, config: Any):
        self.config = config
        self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)
        # Initialize the retry strategy dynamically based on config
        self.NETWORK_RETRY_STRATEGY = get_network_retry_strategy(self.verbosity)

        # Apply the retry decorator dynamically in __init__.
        # We wrap the methods that perform network operations.
        self._fetch_resource_head = self.NETWORK_RETRY_STRATEGY(self._fetch_resource_head)
        self._fetch_file_preview = self.NETWORK_RETRY_STRATEGY(self._fetch_file_preview)
        self._fetch_full_file = self.NETWORK_RETRY_STRATEGY(self._fetch_full_file)

    # (The implementation of the methods below remains the same as the previous stable version)

    # --- Main Entry Point and Dispatcher ---

    @profiler.track("Tool: Data File Inspection")
    def inspect_file(self, url: str, timeout=45, recursion_depth=0) -> Dict[str, Any]:
        """Main entry point: Resolves the URL (handling directories) and executes inspection strategy."""

        MAX_RECURSION = 3
        start_time = time.time()

        # Use injected configuration instead of globals
        if not getattr(self.config, 'DATA_INSPECTION_ENABLED', True):
            return {"status": "disabled", "error": "Data inspection is disabled in configuration."}

        if recursion_depth > MAX_RECURSION:
            if self.verbosity >= 1:
                print(f"    ⚠️ [Data Inspector] Maximum recursion depth exceeded for URL: {url}")
            return {"status": "error", "error": "Maximum redirection/recursion depth exceeded."}

        if self.verbosity >= 1:
            prefix = "    " * recursion_depth + ("" if recursion_depth == 0 else "↪️ ")
            print(f"{prefix}🔬 [Data Inspector] Analyzing resource: {textwrap.shorten(url, width=80)}")

        # Basic URL validation (Access MISSING_DATA_PLACEHOLDERS via config)
        MISSING_DATA = getattr(self.config, 'MISSING_DATA_PLACEHOLDERS', set())
        if not url or url.lower() in MISSING_DATA:
             return {"status": "error", "error": "Invalid or missing URL provided."}

        # Heuristic: Handle GitHub URLs
        original_url = url
        if "github.com" in url and "/blob/" in url:
            url = url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
            if self.verbosity >= 2:
                 print(f"    🔧 [Data Inspector] Converted GitHub URL to raw: {url}")

        # --- Step 1: Determine Resource Type ---
        try:
            # Call the wrapped method
            headers, initial_content = self._fetch_resource_head(url, timeout)
            content_type = headers.get('Content-Type', '').lower()

        except Exception as e:
            if self.verbosity >= 1:
                print(f"    ⚠️ [Data Inspector] Failed to fetch resource head: {type(e).__name__} - {e}")

            # Handle HTTP errors specifically for feedback mechanism
            is_http_error = False
            try:
                if isinstance(e, requests.exceptions.HTTPError):
                    is_http_error = True
            except (AttributeError, TypeError):
                 pass

            if is_http_error:
                 return {"status": "error", "error": f"HTTP Error during fetch: {e}"}

            return {"status": "error", "error": f"Failed to analyze resource: {e}"}


        # --- Step 2: Handle Directory Listings ---
        is_html = 'text/html' in content_type

        if is_html and BeautifulSoup and urljoin:
            content_str = initial_content.decode(errors='ignore')
            # Basic heuristic to detect directory listings
            is_directory_listing = ("Index of" in content_str[:500] or "Directory Listing" in content_str[:500])

            if is_directory_listing:
                if self.verbosity >= 1:
                    print(f"    📂 [Data Inspector] Directory listing detected. Scraping for data files...")

                data_url = self._scrape_directory_listing(content_str, url)

                if data_url:
                    if self.verbosity >= 1:
                        print(f"    💡 [Data Inspector] Found candidate data file: {textwrap.shorten(data_url, width=80)}. Redirecting inspection.")
                    # Recursively call inspect_file
                    return self.inspect_file(data_url, timeout, recursion_depth + 1)
                else:
                    if self.verbosity >= 1:
                        print(f"    ⚠️ [Data Inspector] Could not find suitable data files in the directory listing.")
                    return {"status": "unsupported", "error": "Directory listing found, but no suitable data files detected."}

            else:
                # It's an HTML page but not a directory listing
                if self.verbosity >= 1:
                    print(f"    ℹ️ [Data Inspector] HTML page detected (not a directory listing). Skipping inspection.")
                return {"status": "unsupported", "error": "URL points to an HTML page (e.g., homepage), not a direct data file or directory listing."}


        # --- Step 3: Handle Direct Files (ZIP or Delimited) ---
        is_zip = 'application/zip' in content_type or url.lower().endswith('.zip')

        if is_zip:
            result = self._inspect_zip_archive(url, timeout)
        else:
            result = self._inspect_direct_file(url, timeout)

        # Add timing information
        if result:
            result["duration_s"] = round(time.time() - start_time, 3)

        return result


    # --- Directory Listing Navigation ---

    @staticmethod
    def _scrape_directory_listing(html_content: str, base_url: str) -> Optional[str]:
        """Parses HTML directory listing and uses heuristics to find the best data file URL."""
        if not BeautifulSoup or not urljoin:
            return None

        # Determine parser
        try:
            import lxml
            parser = 'lxml'
        except ImportError:
            parser = 'html.parser'

        soup = BeautifulSoup(html_content, parser)
        candidates = []

        # Compile the ignore patterns regex
        ignore_regex = re.compile("|".join(DataInspectionTools.IGNORE_PATTERNS), re.IGNORECASE)

        for link in soup.find_all('a'):
            href = link.get('href')
            text = link.get_text(strip=True)

            if not href or not text:
                continue

            # Filter out irrelevant links
            if ignore_regex and (ignore_regex.search(href) or ignore_regex.search(text)):
                continue

            # Determine the priority based on extension
            priority = 0
            file_extension = None
            for ext, prio in DataInspectionTools.DIRECTORY_LISTING_PRIORITY.items():
                if href.lower().endswith(ext):
                    if prio > priority:
                        priority = prio
                        file_extension = ext

            if priority == 0:
                # Ignore subdirectories
                if href.endswith('/'):
                    continue
                # Default priority for unrecognized extensions
                priority = 5

            candidates.append({
                'url': urljoin(base_url, href),
                'priority': priority,
                'extension': file_extension
            })

        if not candidates:
            return None

        # Sort by priority and return the top candidate
        candidates.sort(key=lambda x: x['priority'], reverse=True)
        return candidates[0]['url']


    # --- Strategy Implementations (ZIP and Direct) ---

    def _inspect_direct_file(self, url: str, timeout: int) -> Dict[str, Any]:
        """Strategy: Inspects direct (non-ZIP) files (CSV, GZ, TXT) using efficient preview."""
        try:
            # Access configuration via self.config
            MAX_DOWNLOAD_PREVIEW_BYTES = getattr(self.config, 'MAX_DOWNLOAD_PREVIEW_BYTES', 1024*1024)
            # Call the wrapped method
            file_content = self._fetch_file_preview(url, MAX_DOWNLOAD_PREVIEW_BYTES, timeout)

        except DataInspectionTools.SizeLimitExceeded as e:
            return {"status": "error", "error": str(e)}
        except Exception as e:
            if self.verbosity >= 1:
                print(f"    ⚠️ [Data Inspector] Failed to fetch file preview: {type(e).__name__}")
            return {"status": "error", "error": f"Failed to download preview: {e}"}

        if not file_content:
            return {"status": "error", "error": "Downloaded content is empty."}

        return self._process_delimited_content(file_content, url)


    def _inspect_zip_archive(self, url: str, timeout: int) -> Dict[str, Any]:
        """Strategy: Inspects ZIP archives."""
        MAX_DOWNLOAD_ARCHIVE_BYTES = getattr(self.config, 'MAX_DOWNLOAD_ARCHIVE_BYTES', 50*1024*1024)

        if self.verbosity >= 1:
            print(f"    📦 [Data Inspector ZIP] Starting download (Limit: {MAX_DOWNLOAD_ARCHIVE_BYTES/(1024*1024):.1f} MB)...")

        try:
            # Call the wrapped method
            archive_content = self._fetch_full_file(url, MAX_DOWNLOAD_ARCHIVE_BYTES, timeout)
        except DataInspectionTools.SizeLimitExceeded as e:
             return {"status": "error", "error": str(e)}
        except Exception as e:
            if self.verbosity >= 1:
                print(f"    ⚠️ [Data Inspector ZIP] Failed to download archive: {e}")
            return {"status": "error", "error": f"Failed to download archive: {e}"}

        if not archive_content:
             return {"status": "error", "error": "Downloaded archive is empty."}

        # Analyze the archive contents
        try:
            with zipfile.ZipFile(io.BytesIO(archive_content)) as zf:
                file_list = zf.infolist()

                if not file_list:
                    return {"status": "error", "error": "ZIP archive is empty."}

                candidates = []
                for f_info in file_list:
                    filename = f_info.filename
                    # Skip directories and hidden/macOS artifacts
                    if f_info.is_dir() or filename.startswith('__MACOSX/') or filename.split('/')[-1].startswith('.'):
                        continue

                    # Check if the file extension is parsable
                    if any(filename.lower().endswith(ext) for ext in DataInspectionTools.PARSABLE_EXTENSIONS):
                        candidates.append(f_info)

                if not candidates:
                    # Check for compressed files within the ZIP
                    if any(f.filename.lower().endswith(tuple(DataInspectionTools.COMPRESSION_EXTENSIONS)) for f in file_list):
                         return {"status": "unsupported", "error": f"ZIP contains compressed files (e.g., .gz) which require specialized handling."}
                    return {"status": "unsupported", "error": f"ZIP archive does not contain parsable files ({', '.join(DataInspectionTools.PARSABLE_EXTENSIONS)})."}

                # Heuristic: Select the largest candidate file
                candidates.sort(key=lambda x: x.file_size, reverse=True)
                target_file_info = candidates[0]

                if self.verbosity >= 1:
                    print(f"    📄 [Data Inspector ZIP] Inspecting largest file: '{target_file_info.filename}'...")

                # Extract the target file content
                with zf.open(target_file_info.filename) as f:
                    extracted_content = f.read()

                # Process the extracted content
                results = self._process_delimited_content(extracted_content, target_file_info.filename)

                # Add context to the results
                if results and results.get("status") == "success":
                    results["file_type"] += " (via ZIP)"
                    results["extracted_file_name"] = target_file_info.filename

                return results

        except zipfile.BadZipFile:
            return {"status": "error", "error": "Invalid or corrupted ZIP file."}
        except Exception as e:
            return {"status": "error", "error": f"Error during ZIP processing: {e}"}

    # --- Fetch Helpers ---
    # Note: These are the methods that the retry strategy wraps (applied in __init__)

    def _fetch_resource_head(self, url: str, timeout: int, preview_bytes=1024*5) -> Tuple[Dict[str, str], bytes]:
        """Fetches headers and a small preview of the content."""
        response = requests.get(url, headers=HEADERS, timeout=timeout, stream=True, allow_redirects=True)
        response.raise_for_status()

        response_headers = response.headers
        content_preview = b""
        try:
            # Iterate over content to get a preview
            for chunk in response.iter_content(chunk_size=1024):
                content_preview += chunk
                if len(content_preview) >= preview_bytes:
                    break
        finally:
            # Ensure the connection is closed
            response.close()

        return response_headers, content_preview[:preview_bytes]

    def _fetch_file_preview(self, url: str, max_bytes: int, timeout: int) -> bytes:
        """Downloads only the beginning segment of a file using HTTP Range headers."""
        headers = HEADERS.copy()
        # Request a specific range of bytes
        headers['Range'] = f'bytes=0-{max_bytes-1}'
        response = requests.get(url, headers=headers, timeout=timeout, stream=True, allow_redirects=True)

        # 200 OK (if server ignores Range) or 206 Partial Content (if server respects Range)
        if response.status_code not in [200, 206]:
            response.raise_for_status()

        content = b""
        try:
            for chunk in response.iter_content(chunk_size=8192):
                content += chunk
                if len(content) >= max_bytes:
                    break
        finally:
            response.close()

        return content[:max_bytes]

    def _fetch_full_file(self, url: str, max_bytes: int, timeout: int) -> bytes:
        """Downloads the full file content, enforcing a maximum size limit."""

        # Step 1: Check Content-Length using a HEAD request (Optimization)
        try:
            # Short timeout for HEAD request
            head_response = requests.head(url, headers=HEADERS, timeout=5, allow_redirects=True)
            if head_response.status_code == 200 and 'Content-Length' in head_response.headers:
                try:
                    file_size = int(head_response.headers['Content-Length'])
                    if file_size > max_bytes:
                        raise DataInspectionTools.SizeLimitExceeded(f"File size ({file_size / (1024*1024):.2f} MB) exceeds limit based on HEAD request.")
                except ValueError:
                     pass # Ignore if Content-Length is not a valid integer
        except Exception as e:
             if isinstance(e, DataInspectionTools.SizeLimitExceeded):
                  raise e
             # If HEAD fails (e.g., times out or server doesn't support it), proceed to streaming download

        # Step 2: Stream the download
        response = requests.get(url, headers=HEADERS, timeout=timeout, stream=True, allow_redirects=True)
        response.raise_for_status()

        content = b""
        try:
            for chunk in response.iter_content(chunk_size=8192):
                content += chunk
                if len(content) > max_bytes:
                    # Abort the download if the limit is exceeded
                    raise DataInspectionTools.SizeLimitExceeded(f"Download exceeded the size limit ({max_bytes/(1024*1024):.1f} MB) during transfer. Aborted.")
        finally:
            response.close()

        return content

    # --- Parsing Helpers ---

    def _process_delimited_content(self, file_content: bytes, filename_for_context: str) -> Dict[str, Any]:
        """Inspects CSV/TSV like content using Pandas and formats the results."""
        try:
            # Pass verbosity to the helper
            inspection_results = self._inspect_delimited_with_pandas(file_content, filename_for_context, self.verbosity)

            if inspection_results:
                inspection_results["status"] = "success"
                return inspection_results
            else:
                return {"status": "error", "error": "Failed to parse delimited file structure."}

        except Exception as e:
            if self.verbosity >= 1:
                print(f"    ⚠️ [Data Inspector] Error during file parsing: {type(e).__name__}")
            return {"status": "error", "error": f"Error during parsing: {e}"}

    # Static method is appropriate here as it's a pure utility function.
    @staticmethod
    def _inspect_delimited_with_pandas(file_content: bytes, filename_for_context: str, verbosity_level: int) -> Optional[Dict[str, Any]]:
        """The core logic for inspecting delimited content using Pandas."""

        # Handle potential compression based on filename context
        compression = 'gzip' if filename_for_context.lower().endswith('.gz') else 'infer'

        # Define robust parsing parameters
        parsing_params = {
            'sep': None,          # Auto-detect separator
            'engine': 'python',   # Use Python engine for better separator detection
            'nrows': 1000,        # Limit inspection to the first 1000 rows
            'compression': compression,
        }
        # Handle 'on_bad_lines' parameter compatibility across Pandas versions
        try:
            if pd.__version__ >= '1.3.0':
                parsing_params['on_bad_lines'] = 'skip'
            else:
                # Older versions use error_bad_lines/warn_bad_lines
                parsing_params['error_bad_lines'] = False
                parsing_params['warn_bad_lines'] = True
        except Exception:
             # Handle potential import issues with pd version check
             pass

        try:
            file_like_object = io.BytesIO(file_content)

            # Attempt 1: Standard read (assuming headers)
            try:
                df_preview = pd.read_csv(file_like_object, **parsing_params)

            except Exception as e:
                # Check if the exception is a ParserError
                is_parser_error = False
                # Check exception type by name (robust across environments)
                if type(e).__name__ == 'ParserError': is_parser_error = True
                # Check exception type by instance (if pd.errors is available)
                elif hasattr(pd.errors, 'ParserError'):
                     try:
                         if isinstance(e, pd.errors.ParserError): is_parser_error = True
                     except TypeError: pass # Handle potential type errors during isinstance check

                if is_parser_error:
                    # Attempt 2: Handle files without headers if parsing failed
                    file_like_object.seek(0) # Reset buffer position
                    parsing_params['header'] = None
                    df_preview = pd.read_csv(file_like_object, **parsing_params)
                    # Assign generic column names
                    df_preview.columns = [f"Column_{i+1}" for i in range(len(df_preview.columns))]
                else:
                    # If it's another error, re-raise it
                    raise e

        except Exception as e:
            if verbosity_level >= 1:
                print(f"    ⚠️ [Data Inspector] Pandas failed to parse the file preview. Error: {e}")
            return None

        if df_preview.empty and len(df_preview.columns) == 0:
            return None

        # (Result generation logic)
        column_count = len(df_preview.columns)
        headers_sample = df_preview.columns.tolist()[:10] # Limit sample size

        # Heuristic for data type estimation
        if column_count > 2: data_type = "Multivariate"
        elif column_count == 1: data_type = "Univariate"
        else: data_type = "Univariate (Potentially)" # Often time index + value

        results = {
            "file_type": "CSV/Delimited",
            "column_count": column_count,
            "headers_sample": headers_sample,
            "data_type_estimate": data_type,
            "preview_rows_parsed": len(df_preview)
        }
        return results

print("✅ deepcollector/tools/ddi.py written.")

Writing deepcollector/tools/ddi.py


#### **Cell 7: Agent Tools `deepcollector/tools/research.py`**

In [ ]:
%%writefile deepcollector/tools/research.py
# =============================================================================
# V260: Research Tools (Full Untruncated Native PyTorch 4-Bit Waterfall)
# =============================================================================
import requests
import time
import re
import io
import sys
import gc
import traceback
import asyncio
import json
import functools
import concurrent.futures
import threading

try:
    import torch
except ImportError:
    torch = None

from collections import defaultdict
from typing import List, Dict, Any, Optional
from tenacity import RetryError
from requests.exceptions import RequestException, HTTPError, ConnectionError, Timeout, SSLError

try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None

try:
    from deepcollector.utils.profiler import profiler
    from deepcollector.utils.initialization import get_network_retry_strategy, get_gemini_retry_strategy, HEADERS
except ImportError:
    class DummyProfiler:
        def track(self, category): return lambda f: f
    profiler = DummyProfiler()
    def get_network_retry_strategy(verbosity): return lambda f: f
    def get_gemini_retry_strategy(verbosity): return lambda f: f
    HEADERS = {}

try:
    from google.genai import types
except ImportError:
    types = None

class ResearchTools:
    MAX_FETCH_LENGTH = 1000000
    MAX_PDF_PAGES = 50

    def __init__(self, config: Any, keys: Any, models: Any):
        self.config = config
        self.keys = keys
        self.models = models
        self.PdfReader = getattr(config, 'PdfReader', None)
        self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)

        self.thread_pool = concurrent.futures.ThreadPoolExecutor(max_workers=30)
        self.search_failure_count = 0
        self.model_usage_stats = defaultdict(lambda: {"count": 0, "time": 0.0, "time_sq": 0.0})

        self.pool_lock = threading.Lock()
        self.local_llm_lock = threading.Lock()
        self.slow_strikes = defaultdict(int)
        self.SLOW_THRESHOLD_SEC = 50.0
        self.MAX_SLOW_STRIKES = 2

        self.SEARCH_ENABLED = bool(models and getattr(models, 'CLIENT', None)) and bool(types)

        self.NETWORK_RETRY_STRATEGY = get_network_retry_strategy(self.verbosity)
        self.GEMINI_API_RETRY_STRATEGY = get_gemini_retry_strategy(self.verbosity)

        self._fetch_page_content_impl = self.NETWORK_RETRY_STRATEGY(self._fetch_page_content_impl)
        self._generate_content_cascade = self.GEMINI_API_RETRY_STRATEGY(self._generate_content_cascade)

    def _get_active_pool(self, pool_name: str) -> list:
        with self.pool_lock:
            pool = self.models.PRO_POOL if pool_name == "PRO" else self.models.FLASH_POOL
            if len(pool) > 1:
                leader = str(pool[0])
                if self.slow_strikes[leader] >= self.MAX_SLOW_STRIKES:
                    demoted = pool.pop(0)
                    pool.append(demoted)
                    self.slow_strikes[str(demoted)] = 0
                    if self.verbosity >= 1:
                        print(f"\n    ⚠️ [Health Monitor] '{demoted}' hung >{self.SLOW_THRESHOLD_SEC}s {self.MAX_SLOW_STRIKES} times. Demoted.")
            return list(pool)

    def _record_timing(self, target_model: str, duration: float, tracker_key: str):
        t_model_str = str(target_model)
        t_key_str = str(tracker_key)
        with self.pool_lock:
            self.model_usage_stats[t_key_str]["time"] += duration
            self.model_usage_stats[t_key_str]["time_sq"] += duration ** 2
            self.model_usage_stats[t_key_str]["count"] += 1
            if duration > self.SLOW_THRESHOLD_SEC:
                self.slow_strikes[t_model_str] += 1
            else:
                self.slow_strikes[t_model_str] = 0

    @profiler.track("Tool: Web Fetching")
    def _fetch_page_content(self, url: str, timeout=15, minimal_cleaning=False) -> str:
        return self._fetch_page_content_impl(url, timeout, minimal_cleaning)

    def _fetch_page_content_impl(self, url: str, timeout=15, minimal_cleaning=False) -> str:
        def truncate(text):
             if len(text) > self.MAX_FETCH_LENGTH:
                 return text[:self.MAX_FETCH_LENGTH] + "... [TRUNCATED]"
             return text

        if url.startswith('/content/drive/'):
            try:
                if self.verbosity >= 2: print(f"    📁 [Local Fetch] Reading {url} directly from Drive...")
                with open(url, 'rb') as f:
                    content = f.read()
                content_type = 'application/pdf' if url.lower().endswith('.pdf') else 'text/plain'
            except Exception:
                return ""
        else:
            if "github.com" in url and "/blob/" not in url and "/tree/" not in url:
                 try:
                    parts = url.rstrip('/').split('/')
                    if len(parts) >= 5:
                        user, repo = parts[-2], parts[-1]
                        for branch in ['main', 'master']:
                            raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/README.md"
                            try:
                                response = requests.get(raw_url, headers=HEADERS, timeout=5)
                                if response.status_code == 200:
                                    return truncate(response.text)
                            except Exception:
                                pass
                 except Exception:
                     pass

            if "arxiv.org" in url:
                url = url.replace("/abs/", "/pdf/").replace("http://", "https://")
                if not url.lower().endswith(".pdf") and '/pdf/' in url:
                    url += ".pdf"

            try:
                with requests.get(url, headers=HEADERS, timeout=timeout, stream=True) as response:
                    response.raise_for_status()
                    content = b""
                    start_time = time.time()
                    for chunk in response.iter_content(chunk_size=8192):
                        if time.time() - start_time > timeout:
                            break
                        content += chunk
                        if len(content) > self.MAX_FETCH_LENGTH:
                            break
                    content_type = response.headers.get('Content-Type', '').lower()
            except Exception:
                return ""

        text = ""
        if 'application/pdf' in content_type or (url.lower().endswith('.pdf') and 'text/html' not in content_type):
            if self.PdfReader:
                try:
                    pdf_reader = self.PdfReader(io.BytesIO(content))
                    for page in pdf_reader.pages[:self.MAX_PDF_PAGES]:
                        text += (page.extract_text() or "") + "\n"
                except Exception:
                    pass
        else:
            if not BeautifulSoup:
                text = content.decode('utf-8', errors='ignore')
            else:
                try:
                    parser = 'lxml'
                except ImportError:
                    parser = 'html.parser'
                try:
                    soup = BeautifulSoup(content, parser)
                    if minimal_cleaning:
                        text = soup.get_text(separator=' ')
                    else:
                        elements = soup.find_all(['p', 'table', 'li', 'h1', 'h2', 'h3', 'h4', 'div', 'span', 'pre', 'code'])
                        text = ' '.join([elem.get_text(strip=True, separator=' ') for elem in elements])
                        if len(text.split()) < 50:
                            text = soup.get_text(strip=True, separator=' ')
                except Exception:
                    text = content.decode('utf-8', errors='ignore')

        if text is None: text = ""
        if not minimal_cleaning: text = re.sub(r'\s+', ' ', text).strip()
        return truncate(text)

    def tool_pre_flight_crawl(self, text: str, max_links: int = 5) -> List[str]:
        if not text: return []
        prompt = (
            "Extract the most important data-related outbound URLs from the following text. "
            "Focus specifically on links to GitHub repositories, HuggingFace data cards, Zenodo, "
            "Kaggle, or supplementary PDFs that might contain detailed dataset configurations or technical appendices.\n\n"
            f"TEXT:\n{text[:25000]}\n\n"
            "Return ONLY a JSON list of URL strings. E.g., [\"https://github.com/...\", \"https://huggingface.co/...\"] Return maximum 5 URLs."
        )
        try:
            model = self.models.MODEL_PLANNER
            response = self.generate_content_planner(model, prompt)
            urls = self._extract_json_robustly(response.text)
            if isinstance(urls, list):
                urls = [u for u in urls if isinstance(u, str) and u.startswith('http')]
                return urls[:max_links]
        except Exception as e:
            if self.verbosity >= 2:
                print(f"    ⚠️ [Pre-Flight Crawler Error] {e}")
        return []

    @profiler.track("Tool: Gemini Search")
    def tool_search_and_fetch(self, query: str, num_results=None) -> List[Dict[str, str]]:
        query = self._clean_query_string(query)
        if self.verbosity >= 1: print(f"🌐 [Tool: Search/Fetch] Query: '{query}'")
        if num_results is None: num_results = getattr(self.config, 'SEARCH_NUM_RESULTS', 10)

        if getattr(self.config, 'SEARCH_BACKEND', 'GEMINI') == "SEARXNG":
            try:
                from deepcollector.utils.initialization import SearXNGClient
                client = SearXNGClient(base_url=getattr(self.config, 'SEARXNG_URL', "http://localhost:8080"))
                results = client.search(query, max_results=num_results)
                if results: return results
                else:
                    simple_query = self._simplify_query(query)
                    if simple_query != query and len(simple_query) > 3:
                        results = client.search(simple_query, max_results=num_results)
                        if results: return results
            except Exception as e:
                if self.verbosity >= 1: print(f"    ⚠️ SearXNG Failed ({e}). Falling back to Gemini Search...")

        if not self.SEARCH_ENABLED:
            return []

        try:
            results = self._perform_gemini_search(query, num_results)
            if not results:
                simple_query = self._simplify_query(query)
                if simple_query != query and len(simple_query) > 3:
                    if self.verbosity >= 1: print(f"    ⚠️ 0 Results. Retrying with simplified query: '{simple_query}'")
                    results = self._perform_gemini_search(simple_query, num_results)
            if results:
                if self.verbosity >= 1: print(f"    ✅ Gemini Search returned {len(results)} usable results.")
                return results[:num_results]
            else:
                self.search_failure_count += 1
                if self.verbosity >= 1: print("    ❌ Gemini Search returned 0 usable results.")
        except Exception as e:
            self.search_failure_count += 1

        return []

    def _clean_query_string(self, query: str) -> str:
        return str(query).replace("**", "").replace("__", "").strip()

    def _perform_gemini_search(self, query, num_results):
        prompt = (
            f"Perform a Google Search for: '{query}'. "
            f"Return the top {num_results} most relevant results. "
            "You MUST output raw HTTP links to datasets, Githubs, or Archives. "
            "Provide the Title, URL, and Summary."
        )

        pool = self._get_active_pool("PRO")
        for current_idx, target_model in enumerate(pool):
            target_model_str = str(target_model)
            api_start = time.time()
            tracker_key = f"{target_model_str} (Search)"

            try:
                response = self.models.CLIENT.models.generate_content(
                    model=target_model_str,
                    contents=str(prompt),
                    config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
                )

                duration = time.time() - api_start
                self._record_timing(target_model_str, duration, tracker_key)

                results = []
                text_content = response.text if response.text else ""

                md_links = re.findall(r'\[(.*?)\]\((https?://[^\)]+)\)', text_content)
                for title, url in md_links:
                    results.append({"url": url.strip(), "title": title.strip(), "content": "Extracted via Markdown", "type": "Gemini Grounding"})

                raw_urls = re.findall(r'(https?://[^\s>\"\'\)]+)', text_content)
                for url in raw_urls:
                        clean_url = url.rstrip('.,;:')
                        if not any(r['url'] == clean_url for r in results):
                            results.append({"url": clean_url, "title": "Raw Extracted Link", "content": "Extracted via Omni-Regex", "type": "Gemini Grounding"})

                if not results and response.candidates:
                    cand = response.candidates
                    if cand[0].grounding_metadata and cand[0].grounding_metadata.grounding_chunks:
                        for chunk in (cand[0].grounding_metadata.grounding_chunks or []):
                            if getattr(chunk, 'web', None):
                                uri = getattr(chunk.web, 'uri', '')
                                if "vertexaisearch" in uri or "scholar.google" in uri: continue
                                results.append({"url": uri, "title": getattr(chunk.web, 'title', 'Untitled'), "content": "Grounding Source Metadata", "type": "Gemini Metadata"})

                return results

            except Exception as e:
                duration = time.time() - api_start
                self._record_timing(target_model_str, duration, tracker_key)
                error_str = str(e).lower()
                if "429" in error_str or "quota" in error_str or "503" in error_str or "timeout" in error_str or duration > self.SLOW_THRESHOLD_SEC:
                    time.sleep(1.0)
                    if current_idx < len(pool) - 1: continue
                    else: return []
                else: return []
        return []

    def _simplify_query(self, query):
        query = str(query).replace('"', '').replace(" OR ", " ").replace(" AND ", " ")
        query = re.sub(r'site:\S+', '', query)
        query = re.sub(r'^(Look for|Search for|Find|Identify|Locate)\s+', '', query, flags=re.IGNORECASE)
        query = re.sub(r'\s+(with its attributes|with attributes|and provide|details about).*$', '', query, flags=re.IGNORECASE)
        return query.strip()

    def _extract_json_robustly(self, text: str) -> Any:
        if not text or text == "[missing]": return []
        if not isinstance(text, str): text = str(text)

        text = re.sub(r',\s*([\]}])', r'\1', text)
        text = re.split(r'\n\s*thought\b', text, maxsplit=1, flags=re.IGNORECASE)[0]
        text = re.split(r'\n\s*Wait, I noticed', text, maxsplit=1, flags=re.IGNORECASE)[0]
        text = re.split(r'======', text, maxsplit=1)[0]

        match = re.search(r"`{3}(?:json)?\n(.*?)\n`{3}", text, re.DOTALL | re.IGNORECASE)
        if match:
            try: return json.loads(match.group(1))
            except Exception: pass

        def extract_balanced(s, open_char, close_char):
            start = s.find(open_char)
            if start == -1: return None
            count = 0
            for i in range(start, len(s)):
                if s[i] == open_char: count += 1
                elif s[i] == close_char: count -= 1
                if count == 0:
                    try: return json.loads(s[start:i+1])
                    except: return None
            return None

        res = extract_balanced(text, '[', ']')
        if res is not None: return res

        res = extract_balanced(text, '{', '}')
        if res is not None: return res

        return []

    def tool_load_url(self, url: str) -> List[Dict[str, str]]:
        try:
            content = self._fetch_page_content(url)
            if content and len(content.split()) >= 15:
                return [{"url": url, "content": content, "title": f"Direct Load: {url[:50]}", "type": "Direct Load"}]
        except Exception: pass
        return []

    def tool_inspect_data_file(self, url: str, ddi_tool: Any = None) -> Dict:
        if ddi_tool:
            try: return ddi_tool.inspect_remote_file(url)
            except Exception as e: return {"status": "error", "error": str(e)}
        try:
            from deepcollector.tools.ddi import DDITools
            temp_tool = DDITools(self.config)
            return temp_tool.inspect_remote_file(url)
        except Exception as e:
            return {"status": "skipped", "error": f"DDI Tool missing: {e}"}

    def _generate_content_local(self, prompt: str, **kwargs):
        """COLAB ARCHIVAL: Pure Native PyTorch 4-Bit Waterfall."""
        api_start = time.time()
        model_name_label = f"Gemma ({getattr(self.config, 'LLM_BACKEND', 'LOCAL')})"

        class MockResponseWrapper:
            def __init__(self, text): self.text = text

        with self.local_llm_lock:
            inputs = None
            outputs = None
            out_tokens_list = []

            model = getattr(self.models, 'LOCAL_MODEL', None)
            tokenizer = getattr(self.models, 'LOCAL_TOKENIZER', None)

            if not model or not tokenizer or isinstance(model, str):
                if getattr(self.config, 'VERBOSITY_LEVEL', 1) >= 2:
                    print(f"    ⚠️ Local PyTorch model not loaded (Found {type(model)}). Falling back to Cloud Gemini...")
                return self._generate_content_cascade("PRO" if "strategic planner" in prompt else "FLASH", prompt, **kwargs)

            sys_prefix = "You are a strict data extraction AI. You MUST output ONLY valid JSON format.\n\n"
            chat = [{"role": "user", "content": sys_prefix + prompt}]

            try:
                formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            except Exception:
                formatted_prompt = f"<bos><start_of_turn>user\n{sys_prefix}{prompt}<end_of_turn>\n<start_of_turn>model\n"

            # 4-bit gives us tons of room. Start with massive limit and back off dynamically.
            current_max_len = 32000
            req_max_new = min(kwargs.get("max_new_tokens", 1024), 2048)

            while current_max_len >= 2000:
                try:
                    if torch is not None and torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        torch.cuda.ipc_collect()
                    gc.collect()

                    inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=current_max_len).to(model.device)

                    terminators = [tokenizer.eos_token_id]
                    if hasattr(tokenizer, "get_vocab"):
                        vocab = tokenizer.get_vocab()
                        for t in ["<end_of_turn>", "<|eot_id|>", "<|im_end|>"]:
                            if t in vocab:
                                terminators.append(vocab[t])

                    with torch.inference_mode():
                        with torch.autocast("cuda", dtype=torch.bfloat16):
                            outputs = model.generate(
                                **inputs,
                                max_new_tokens=req_max_new,
                                do_sample=False, # Greedy decoding (NaN immune)
                                use_cache=True,
                                pad_token_id=tokenizer.eos_token_id,
                                eos_token_id=terminators,
                                output_attentions=False,
                                output_hidden_states=False,
                                return_dict_in_generate=False
                            )

                    # SAFE PYTORCH SLICING
                    prompt_len = inputs["input_ids"].shape[1]
                    out_tensor = outputs[0][prompt_len:]
                    out_tokens_list = out_tensor.cpu().tolist()
                    break

                except Exception as e:
                    err_str = str(e).lower()
                    if "cuda out of memory" in err_str or "outofmemoryerror" in err_str or "alloc" in err_str:
                        if getattr(self.config, 'VERBOSITY_LEVEL', 1) >= 1:
                            print(f"    ⚠️ [VRAM Ceiling Hit] OOM at {current_max_len} tokens. Truncating by 25% and retrying...")

                        current_max_len = int(current_max_len * 0.75)

                        if 'inputs' in locals() and inputs is not None: del inputs
                        if 'outputs' in locals() and outputs is not None: del outputs

                        if hasattr(sys, 'last_traceback'): sys.last_traceback = None
                        if hasattr(sys, 'last_type'): sys.last_type = None
                        if hasattr(sys, 'last_value'): sys.last_value = None
                        if hasattr(e, "__traceback__") and e.__traceback__: traceback.clear_frames(e.__traceback__)
                        del e
                        continue
                    else:
                        if getattr(self.config, 'VERBOSITY_LEVEL', 1) >= 1:
                            print(f"    ⚠️ [Local PyTorch Error] {e}. Falling back to Cloud...")
                        return self._generate_content_cascade("PRO" if "strategic planner" in prompt else "FLASH", prompt, **kwargs)

            if not out_tokens_list:
                return self._generate_content_cascade("PRO" if "strategic planner" in prompt else "FLASH", prompt, **kwargs)

            response_text = tokenizer.decode(out_tokens_list, skip_special_tokens=True)
            del prompt
            duration = time.time() - api_start
            self._record_timing(model_name_label, duration, model_name_label)

            clean_res = response_text.replace("`" * 3 + "json", "").replace("`" * 3, "").strip()
            return MockResponseWrapper(clean_res)

    def _generate_content_cascade(self, pool_name: str, prompt: str, **kwargs):
        if not getattr(self.models, 'CLIENT', None): raise ValueError("Gemini Client not initialized.")
        pool = self._get_active_pool(str(pool_name))

        max_tokens = kwargs.pop("max_new_tokens", None)
        for k in ["do_sample", "temperature", "top_p", "top_k", "repetition_penalty", "return_dict_in_generate", "output_scores", "stop"]:
            kwargs.pop(k, None)

        if max_tokens and types:
            config_obj = kwargs.get("config")
            if config_obj is None:
                config_obj = types.GenerateContentConfig()
            config_obj.max_output_tokens = int(max_tokens)
            kwargs["config"] = config_obj

        for current_idx, target_model in enumerate(list(pool)):
            api_start = time.time()
            try:
                response = self.models.CLIENT.models.generate_content(model=str(target_model), contents=str(prompt), **kwargs)
                duration = time.time() - api_start
                self._record_timing(str(target_model), duration, str(target_model))
                return response
            except Exception as e:
                duration = time.time() - api_start
                self._record_timing(str(target_model), duration, str(target_model))

                error_str = str(e).lower()
                if ("429" in error_str or "quota" in error_str or "503" in error_str or "timeout" in error_str or duration > self.SLOW_THRESHOLD_SEC):
                    time.sleep(1.0)
                    if current_idx < len(pool) - 1: continue
                    else: raise ResourceWarning(f"All models in {pool_name} pool exhausted. Last Error: {e}")
                else: raise e
        raise ResourceWarning(f"All models in {pool_name} pool exhausted.")

    @profiler.track("LLM: Planner")
    def generate_content_planner(self, model_name, prompt, **kwargs):
        if getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"] and hasattr(self.models, 'LOCAL_MODEL'):
            return self._generate_content_local(prompt, **kwargs)
        return self._generate_content_cascade("PRO", prompt, **kwargs)

    def generate_content_synthesizer(self, model_name, prompt, **kwargs):
        if getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"] and hasattr(self.models, 'LOCAL_MODEL'):
            return self._generate_content_local(prompt, **kwargs)
        return self._generate_content_cascade("PRO", prompt, **kwargs)

    @profiler.track("LLM: Standard")
    def generate_content_standard(self, model_name, prompt, **kwargs):
        if getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"] and hasattr(self.models, 'LOCAL_MODEL'):
            return self._generate_content_local(prompt, **kwargs)
        return self._generate_content_cascade("PRO", prompt, **kwargs)

    @profiler.track("LLM: RAG")
    def generate_content_rag(self, prompt, **kwargs):
        if getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"] and hasattr(self.models, 'LOCAL_MODEL'):
            return self._generate_content_local(prompt, **kwargs)
        return self._generate_content_cascade("FLASH", prompt, **kwargs)

    async def generate_content_synthesizer_async(self, model_name, prompt, **kwargs):
        if getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]:
            return self.generate_content_synthesizer(model_name, prompt, **kwargs)
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(self.thread_pool, functools.partial(self.generate_content_synthesizer, model_name, prompt, **kwargs))

    async def generate_content_rag_async(self, prompt, **kwargs):
        if getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]:
            return self.generate_content_rag(prompt, **kwargs)
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(self.thread_pool, functools.partial(self.generate_content_rag, prompt, **kwargs))

print("✅ deepcollector/tools/research.py written (Native PyTorch 4-Bit Waterfall Restored).")

Writing deepcollector/tools/research.py


#### **Cell 8: AdvancedKnowledgeProcessor `deepcollector/tools/knowledge_loader.py`**

In [ ]:
%%writefile deepcollector/tools/knowledge_loader.py
# V18.5 (Based on V18.2): Improved Google Sheets Error Clarity
import pandas as pd
import re
from typing import Any, Dict, List, Optional
try:
    import gspread
    # V18.5: Import specific exceptions for better error handling
    from gspread.exceptions import SpreadsheetNotFound, APIError
except ImportError:
    gspread = None
    SpreadsheetNotFound = Exception
    APIError = Exception

# Robust profiler import
try:
    from deepcollector.utils.profiler import profiler
except ImportError:
    class DummyProfiler:
        def track(self, category):
            def decorator(func): return func
            return decorator
    profiler = DummyProfiler()


class ExternalKnowledgeManager:
    """
    Manages the loading and parsing of external knowledge sources
    (Expert Hints and Canonical Projects) from Google Sheets.
    """
    def __init__(self, config: Any):
        self.config = config
        self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)
        # Robust access to GSPREAD_AVAILABLE
        self.GSPREAD_AVAILABLE = getattr(config, 'GSPREAD_AVAILABLE', False)
        self.gc = None
        self.is_initialized = False

        # Configuration IDs
        self.hints_sheet_id = getattr(config, 'GOOGLE_SHEET_HINTS_ID', None)
        self.projects_sheet_id = getattr(config, 'GOOGLE_SHEET_PROJECT_LIST_ID', None)

        # Data storage
        self.expert_hints_data: List[Dict[str, str]] = []
        self.canonical_projects_data: List[Dict[str, Any]] = []

    # (initialize, load_all remain the same)
    def initialize(self, authenticated_gc: Any):
        """Initializes the connection using a pre-authenticated gspread client."""
        if not self.GSPREAD_AVAILABLE or gspread is None:
            # Suppress warning if verbosity is low, as this might be expected
            if self.verbosity >= 1:
                print("⚠️ [ExternalKnowledge] Google Sheets libraries not available. Cannot load external knowledge.")
            return False

        # Check if any sheets are configured
        if not self.hints_sheet_id and not self.projects_sheet_id:
            # This is common if only KB is used, so suppress the log message.
            return True # Not a failure, just not configured

        self.gc = authenticated_gc
        self.is_initialized = True
        if self.verbosity >= 1:
            print(f"🌐 [ExternalKnowledge] Initialization complete. Ready to load sources.")
        return True

    @profiler.track("ExternalKnowledge: Load All")
    def load_all(self):
        """Loads data from all configured external sources."""
        if not self.is_initialized or not self.gc:
            return

        if self.hints_sheet_id:
            self._load_expert_hints()

        if self.projects_sheet_id:
            self._load_canonical_projects()

    # V18.5: Updated error handling for clarity
    def _load_spreadsheet_values(self, sheet_id: str, source_name: str) -> Optional[List[List[str]]]:
        """Helper to safely load spreadsheet values using gspread."""
        # V18.5: Check explicitly for placeholder text before attempting API call.
        if not sheet_id or "YOUR_GOOGLE_SHEET_ID_HERE" in sheet_id:
             # Increased verbosity level (>=0) for this critical configuration check
             if self.verbosity >= 0:
                print(f"⚠️ [ExternalKnowledge] Skipping {source_name}. Sheet ID is missing or placeholder text is present.")
             return None

        if self.verbosity >= 1:
            # Display only the first few characters of the ID for security/brevity
            display_id = f"{sheet_id[:8]}...{sheet_id[-4:]}" if len(sheet_id) > 12 else sheet_id
            print(f"💡 [ExternalKnowledge] Loading {source_name} from Sheet ID: {display_id}...")

        try:
            spreadsheet = self.gc.open_by_key(sheet_id)
            # Assumption: Data is in the first worksheet (index 0)
            worksheet = spreadsheet.get_worksheet(0)
            data = worksheet.get_all_values() # Get as list of lists (strings)

            if not data:
                 if self.verbosity >= 1:
                    print(f"    ⚠️ [ExternalKnowledge] {source_name} sheet is empty.")
                 return None

            if self.verbosity >= 1:
                # Handle case where data might only contain a header
                record_count = max(0, len(data)-1)
                print(f"    ✅ [ExternalKnowledge] Loaded {record_count} records from {source_name} (excluding header).")
            return data

        # V18.5: Specific exception handling
        except SpreadsheetNotFound:
            if self.verbosity >= 0:
                # This error specifically means the ID is invalid OR the user lacks permission.
                display_id = f"{sheet_id[:8]}...{sheet_id[-4:]}" if len(sheet_id) > 12 else sheet_id
                print(f"❌ [ExternalKnowledge] CRITICAL: {source_name} spreadsheet not found (ID: {display_id}).")
                print(f"   -> Check if the ID is correct AND that the authenticated Google account has access rights.")
            return None
        except APIError as e:
             if self.verbosity >= 0:
                # Handle other potential API errors (e.g., quota exceeded)
                print(f"❌ [ExternalKnowledge] Google Sheets API Error while loading {source_name}: {e}")
             return None
        except Exception as e:
            if self.verbosity >= 0:
                print(f"❌ [ExternalKnowledge] Unexpected error loading {source_name}: {e}")
            return None

    # (_load_expert_hints remains the same)
    def _load_expert_hints(self):
        """Loads and parses the Expert Hints spreadsheet."""
        data = self._load_spreadsheet_values(self.hints_sheet_id, "Expert Hints")
        if data is None: return

        # Expected format: Hint Type | Keyword | Hint 1 | Hint 2 | ...

        for index, row in enumerate(data[1:]): # Skip header
            if len(row) < 3:
                continue # Need at least Type, Keyword, and one Hint

            hint_type = row[0].strip()
            keyword = row[1].strip()

            if not hint_type or not keyword:
                continue

            # Remaining columns are hints
            hints = [h.strip() for h in row[2:] if h.strip()]

            # We store each hint individually for better RAG granularity
            for i, hint_text in enumerate(hints):
                # Store data in a format suitable for RAG indexing
                self.expert_hints_data.append({
                    'type': hint_type,
                    'keyword': keyword,
                    'hint_text': hint_text,
                    # Create a unique ref_id for indexing (using row index+1 because we skipped header)
                    'ref_id': f"HINT:{keyword[:30]}_{index+1}_{i}"
                })

    # (_load_canonical_projects remains the same as V18.2)
    def _load_canonical_projects(self):
        """Loads and parses the Canonical Projects spreadsheet."""
        data = self._load_spreadsheet_values(self.projects_sheet_id, "Canonical Projects")
        if data is None: return

        # V18.2 Expected format:
        # A: Canonical Project ID
        # B: Project Name (Informal Names)
        # C: Links (comma-separated URLs)
        # D: Project Context Description (Optional)
        # E: Project Method (Optional, defaults to 1)

        EXPECTED_COLS = 5

        for index, row in enumerate(data[1:]): # Skip header
            # Ensure we have 5 columns, padding if necessary
            if len(row) < EXPECTED_COLS:
                 # Pad the row if columns are missing entirely
                 row += [''] * (EXPECTED_COLS - len(row))

            # 1. Parse Columns
            canonical_id = row[0].strip().upper()
            project_name = row[1].strip()
            links_str = row[2].strip()
            context_desc = row[3].strip()
            method_str = row[4].strip()

            if not canonical_id:
                continue

            # 2. Process Data
            # Parse Method
            try:
                method = int(method_str) if method_str else 1
            except ValueError:
                method = 1

            # Parse Comma-Separated URLs
            initial_urls_list = [url.strip() for url in links_str.split(',') if url.strip() and re.match(r'^https?://', url.strip())]

            # Construct Full Context (Name: Description)
            if project_name and context_desc:
                full_project_context = f"{project_name}: {context_desc}"
            else:
                # Fallback if name or description is missing
                full_project_context = project_name or context_desc or canonical_id

            # 3. Store Data (used by ProjectLoader and RAG indexing)
            self.canonical_projects_data.append({
                'project_id': canonical_id,
                'project_name': project_name,
                'initial_urls': initial_urls_list, # Stored as list for ProjectLoader
                'project_context': full_project_context,
                'method': method,
                'ref_id': f"PROJ_DEF:{canonical_id}",
                # Fields specifically for RAG indexing format (backward compatibility)
                'informal_names': project_name,
                'links': links_str, # Kept as string for RAG indexing display
            })

print("✅ deepcollector/tools/knowledge_loader.py written.")

Writing deepcollector/tools/knowledge_loader.py


#### **Cell 9 deepcollector/utils/project_loader.py (V18.2)**

In [ ]:
%%writefile deepcollector/utils/project_loader.py
# =============================================================================
# V32: Project Loader (Full Restoration + Gspread V5/V6 API Safe Fetch)
# =============================================================================
import sys
import re
import pandas as pd
import traceback
from tenacity import retry, stop_after_attempt, wait_exponential

class ExternalKnowledge:
    def __init__(self, config, gc_client, verbosity=1):
        self.config = config
        self.gc = gc_client
        self.verbosity = verbosity
        self.sheet_id = getattr(config, 'GOOGLE_SHEET_PROJECT_LIST_ID', None)
        self.projects = []
        self.headers = []
        self.worksheet = None

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=10))
    def _fetch_sheet_data(self):
        try:
            sh = self.gc.open_by_key(self.sheet_id)

            # Safely grab the first worksheet regardless of gspread version
            try:
                self.worksheet = sh.get_worksheet(0)
                if self.worksheet is None:
                    self.worksheet = sh.worksheets()
            except Exception:
                self.worksheet = sh.worksheets()

            # CRITICAL FIX: Bulletproof cascade to support both Gspread v5 and v6+
            try:
                if hasattr(self.worksheet, 'get_values'):
                    return self.worksheet.get_values(value_render_option='FORMULA')
                else:
                    try:
                        return self.worksheet.get_all_values(value_render_option='FORMULA')
                    except TypeError:
                        return self.worksheet.get_all_values()
            except Exception:
                # Ultimate fallback
                return self.worksheet.get_all_values()

        except Exception as e:
            if self.verbosity >= 1:
                print(f"      ⚠️ [Google Sheets API] Fetch failed: {type(e).__name__} - {e}")
            raise e

    def load(self):
        if self.verbosity >= 1:
            print("🌐 [ExternalKnowledge] Initialization complete. Ready to load sources.")
        if not self.gc or not self.sheet_id:
            if self.verbosity >= 1:
                print("    ⚠️ [ExternalKnowledge] Google credentials or Sheet ID missing.")
            return

        if self.verbosity >= 1:
            print(f"💡 [ExternalKnowledge] Loading Canonical Projects from Sheet ID: {self.sheet_id[:8]}...{self.sheet_id[-4:]}")

        try:
            data = self._fetch_sheet_data()

            if data and len(data) > 1:
                self.headers = [str(h).strip() for h in data]
                self.projects = [dict(zip(self.headers, row)) for row in data[1:]]
                if self.verbosity >= 1:
                    print(f"    ✅ [ExternalKnowledge] Loaded {len(self.projects)} records.")
                    if self.verbosity >= 2:
                        print(f"    📋 Detected Headers: {self.headers}")

                self._ensure_proj_lost()
            else:
                if self.verbosity >= 1: print("    ⚠️ [ExternalKnowledge] Canonical Sheet is empty.")
        except Exception as e:
            if self.verbosity >= 1:
                print(f"    ❌ [ExternalKnowledge] Failed to load from canonical sheet after retries: {e}")
            raise e

    def _ensure_proj_lost(self):
        has_lost = False
        for p in self.projects:
            pid = ""
            for k, v in p.items():
                if any(term in str(k).lower() for term in ["canonical", "projectid", "id"]):
                    pid = str(v).strip().upper()
                    break

            if pid == "PROJ_LOST" or "LOST" in pid:
                has_lost = True
                break

        if not has_lost and self.worksheet and self.headers:
            if self.verbosity >= 1:
                print("    🗂️ 'PROJ_LOST' not found in canonical list. Appending it now...")
            try:
                new_row = []
                for h in self.headers:
                    h_lower = str(h).lower()
                    if "canonical" in h_lower or "id" in h_lower: new_row.append("PROJ_LOST")
                    elif "name" in h_lower: new_row.append("Lost / Incidental Datasets")
                    elif "link" in h_lower or "url" in h_lower: new_row.append("")
                    elif "comment" in h_lower or "desc" in h_lower: new_row.append("Auto-generated sink for orphans and hallucinations.")
                    elif "method" in h_lower or "type" in h_lower: new_row.append("System")
                    else: new_row.append("")

                self.worksheet.append_row(new_row, value_input_option='USER_ENTERED')
                self.projects.append(dict(zip(self.headers, new_row)))
                if self.verbosity >= 1:
                    print("    ✅ 'PROJ_LOST' successfully added to external project list.")
            except Exception as e:
                if self.verbosity >= 1:
                    print(f"    ⚠️ Could not append PROJ_LOST to canonical sheet: {e}")

class ProjectLoader:
    def __init__(self, config, authenticated_gc=None, verbosity=1):
        self.config = config
        self.gc = authenticated_gc
        self.verbosity = verbosity

    def _clean_formula(self, text):
        """Extracts display text from =HYPERLINK formulas so Names/Comments aren't messy."""
        text = str(text).strip()
        m = re.search(r'=HYPERLINK\("([^"]+)"\s*,\s*"([^"]+)"\)', text, re.IGNORECASE)
        if m: return m.group(2)

        m2 = re.search(r'=HYPERLINK\("([^"]+)"\)', text, re.IGNORECASE)
        if m2: return m2.group(1)

        return text

    def resolve_configuration(self):
        raw_id = str(getattr(self.config, 'CURRENT_PROJECT_ID', '')).strip().upper()

        if not raw_id:
            sys.exit("🛑 ABORTING: CURRENT_PROJECT_ID is missing or empty in configuration.")

        if self.verbosity >= 1:
            print(f"🔄 [ProjectLoader] Dynamically loading configuration for ID: {raw_id}...")

        target_id = raw_id if raw_id.startswith("PROJ_") else f"PROJ_{raw_id}"
        self.config.CURRENT_PROJECT_ID = target_id

        ek = ExternalKnowledge(self.config, self.gc, self.verbosity)
        ek.load()

        if not ek.projects:
            sys.exit("🛑 ABORTING: Could not load any projects from the canonical sheet.")

        matched_p = None
        available_ids = []

        for p in ek.projects:
            pid_raw = ""
            for k, v in p.items():
                k_lower = str(k).lower()
                if any(term in k_lower for term in ["canonical", "projectid", "id"]):
                    pid_raw = self._clean_formula(v).upper()
                    break

            if not pid_raw and p:
                pid_raw = self._clean_formula(list(p.values())).upper()

            pname_raw = ""
            for k, v in p.items():
                k_lower = str(k).lower()
                if "name" in k_lower and "canonical" not in k_lower:
                    pname_raw = self._clean_formula(v).upper()
                    break

            pid = f"PROJ_{pid_raw}" if pid_raw and not pid_raw.startswith("PROJ_") else pid_raw
            pname_as_id = f"PROJ_{pname_raw}" if pname_raw else ""

            if pid: available_ids.append(pid)
            elif pname_as_id: available_ids.append(pname_as_id)

            if target_id == pid or target_id == pname_as_id:
                matched_p = p
                break

        if not matched_p:
            print(f"\n📋 Detected Headers in Sheet: {ek.headers}")
            print(f"📋 Available Projects in Sheet: {', '.join(available_ids[:10])}... (Showing first 10)")
            sys.exit(f"\n🛑 ABORTING: Target project '{target_id}' was NOT FOUND in the Canonical Projects Sheet.\n"
                     f"Please ensure it is added to the Google Sheet.")

        # =====================================================================
        # OMNI-URL EXTRACTOR: Scan ALL columns in the row for http/https links OR /content/drive/ paths
        # This completely ignores headers, immune to shifted columns or typos.
        # =====================================================================
        extracted_urls = []
        for k, v in matched_p.items():
            val_str = str(v)
            found_urls = re.findall(r'(https?://[^\s,\">|\]\)]+|/content/drive/[^\s,\">|\]\)]+)', val_str)
            for u in found_urls:
                clean_u = u.strip(';,"\'()[]')
                if clean_u and clean_u not in extracted_urls:
                    extracted_urls.append(clean_u)

        if not extracted_urls and target_id != "PROJ_LOST":
            row_data_str = " | ".join([f"{k}: {v}" for k, v in matched_p.items()])
            print(f"\n⚠️ WARNING: Project '{target_id}' was found, but NO valid http/https URLs or /content/drive/ paths could be extracted from ANY column.\n"
                  f"   Row Data Seen by Agent: {row_data_str}\n"
                  f"   Please ensure URLs start with http://, https://, or /content/drive/")

        self.config.INITIAL_URLS = extracted_urls

        pname_raw = ""
        for k, v in matched_p.items():
            k_lower = str(k).lower()
            if "name" in k_lower and "canonical" not in k_lower:
                pname_raw = self._clean_formula(v)
                break

        self.config.CURRENT_PROJECT_NAME = pname_raw or target_id.replace("PROJ_", "")

        method_str = ""
        for k, v in matched_p.items():
            if "method" in str(k).lower():
                method_str = self._clean_formula(v)
                break

        try:
            clean_val = re.sub(r'[^0-9]', '', method_str)
            method_val = int(clean_val) if clean_val else 1
        except ValueError:
            method_val = 1

        self.config.PROJECT_METHOD = method_val
        self.config.SCRAPING_METHOD = method_val

        desc = ""
        for k, v in matched_p.items():
            if "comment" in str(k).lower() or "description" in str(k).lower() or "summary" in str(k).lower():
                desc = self._clean_formula(v)
                break

        self.config.PROJECT_CONTEXT = f"{self.config.CURRENT_PROJECT_NAME}: {desc}" if desc else self.config.CURRENT_PROJECT_NAME

        if self.verbosity >= 1:
            print(f"    ✅ [ProjectLoader] Configuration loaded. Method: {method_val}, URLs: {len(self.config.INITIAL_URLS)}.")
            for i, u in enumerate(self.config.INITIAL_URLS):
                print(f"    🔗 Source URL [{i+1}]: {u}")

        return True

print("✅ deepcollector/utils/project_loader.py written (Full Restoration + Gspread V5/V6 API Safe Fetch).")

Writing deepcollector/utils/project_loader.py


#### **Cell 10: Base Harvester deepcollector/harvesting/base_harvester.py**

In [ ]:
%%writefile deepcollector/harvesting/base_harvester.py
# =============================================================================
# HDK V1.5: Core (BaseHarvester Interface)
# Matches the definition in Jan24_2026HDKupdatedV2.ipynb
# =============================================================================
from abc import ABC, abstractmethod
from typing import Any

# Import State for type hinting if available
try:
    from deepcollector.core.state import CatalogState
except ImportError:
    CatalogState = Any

class BaseHarvester(ABC):
    """
    Abstract Base Class defining the interface required for all harvesters.
    """
    def __init__(self, config: Any, tools: Any):
        self.config = config
        self.tools = tools
        self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)

    @abstractmethod
    def execute_harvest(self, state: CatalogState) -> bool:
        """
        Executes the main harvesting workflow.
        Must update the provided CatalogState object.
        Returns True on success, False on critical failure.
        """
        pass

print("✅ deepcollector/harvesting/base_harvester.py written (HDK Compatible).")

Writing deepcollector/harvesting/base_harvester.py


#### **Cell 11 UCI Harvester deepcollector/harvesting/uci_harvester.py**

In [ ]:
%%writefile deepcollector/harvesting/uci_harvester.py
# =============================================================================
# HDK V11.2: UCI Harvester (Deep Think Fix: Single Line Enforcer)
# =============================================================================

import requests
import time
import re
import random
import json
import pandas as pd
import traceback
from typing import List, Dict, Any, Optional, Tuple

try:
    import gspread
    from google.colab import auth
    from google.auth import default
except ImportError:
    gspread = None

try:
    import lxml
    BS_PARSER = 'lxml'
except ImportError:
    BS_PARSER = 'html.parser'

from bs4 import BeautifulSoup

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None

from requests.exceptions import HTTPError, RequestException, ConnectionError, Timeout

try:
    from deepcollector.harvesting.base_harvester import BaseHarvester
    from deepcollector.core.state import CatalogState, CellData
    from deepcollector.utils.profiler import profiler
except ImportError:
    BaseHarvester = object
    CatalogState = object
    CellData = dict

    class DummyProfiler:
        def track(self, c):
            return lambda x: x

    profiler = DummyProfiler()


class UCIHarvester(BaseHarvester):
    """
    Comprehensive Embedded JSON Extraction Harvester for the UCI Machine Learning Repository.
    """

    BASE_URL = "https://archive.ics.uci.edu"
    API_LIST_ENDPOINT = f"{BASE_URL}/api/datasets/list"
    WEB_UI_BASE_LIST = f"{BASE_URL}/datasets"
    TARGET_DATA_TYPE = "Time-Series"

    # Export Configurations
    SPREADSHEET_NAME_TEMPLATE = "HDK_UCI_Harvest_Export_{date}"
    CHAR_LIMIT = 49000

    # User agents to rotate to avoid basic scraping blocks
    USER_AGENTS = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.5 Safari/605.1.15',
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/115.0'
    ]

    # Rate Limiting & Backoff
    FETCH_DELAY_MIN = 1.0
    FETCH_DELAY_MAX = 3.0
    ADAPTIVE_BACKOFF_TIME = 60.0
    MAX_CONSECUTIVE_FAILURES = 5

    def __init__(self, config: Any, tools: Any):
        """
        Initializes the UCI Harvester with the application configuration and tools.
        """
        if BaseHarvester != object:
            try:
                super().__init__(config, tools)
            except TypeError:
                self.config = config
                self.tools = tools
                self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)
        else:
             self.config = config
             self.tools = tools
             self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)

        # Initialize a persistent requests session for connection pooling
        if 'requests' in globals():
             self.session = requests.Session()
        else:
             self.session = None

    @profiler.track("Harvester: UCI Execute")
    def execute_harvest(self, state: CatalogState) -> bool:
        """
        The main orchestration method for the UCI Harvest.
        """
        print("\n" + "="*60)
        print("🚜 STARTING HARVEST (UCI V11.2 - Single Line Fix)")
        print("="*60)

        if not self.session or not BeautifulSoup:
            print("❌ [UCI Harvester] CRITICAL: Missing 'requests' or 'BeautifulSoup' libraries.")
            return False

        # --- Stage 1: Discovery ---
        asset_identifiers = self.discover_assets()

        if not asset_identifiers:
            print("🛑 [UCI Harvester] CRITICAL: Discovery failed or no assets found. Aborting harvest.")
            return False

        # --- Stage 2 & 3: Fetch HTML & Screen Metadata ---
        enriched_assets = self.fetch_extract_and_screen_metadata(asset_identifiers)

        if not enriched_assets:
            print("🚜 [UCI Stage 2/3] Finished. No target Time-Series assets identified.")
            return True

        # --- Stage 4: Update Catalog State ---
        self._update_state(state, enriched_assets)

        return True

    @profiler.track("Harvester: UCI Discover Assets (API)")
    def discover_assets(self) -> List[Dict[str, Any]]:
        """
        Fetches the complete roster of all available dataset IDs and Slugs.
        """
        print("🚜 [UCI Stage 1] Fetching Comprehensive API List...")
        all_assets = []

        try:
            if self.session is None:
                raise AttributeError("self.session is None")

            headers = self._get_randomized_headers(referer=self.WEB_UI_BASE_LIST)

            # The UCI API currently ignores skip/take and dumps everything.
            # We make one large request to be safe and avoid infinite loops.
            response = self.session.get(
                self.API_LIST_ENDPOINT,
                headers=headers,
                params={'skip': 0, 'take': 5000},
                timeout=45
            )
            response.raise_for_status()

            data = self._parse_json_response(response)

            batch_assets = []
            if isinstance(data, dict):
                if 'data' in data:
                    if isinstance(data['data'], list):
                        batch_assets = data['data']
            elif isinstance(data, list):
                batch_assets = data

            if not batch_assets:
                print("    ❌ [UCI Stage 1] API returned an empty list.")
                return []

            standardized_batch = self._standardize_keys(batch_assets)

            seen_ids = set()
            for item in standardized_batch:
                item_id = item.get('ID')
                if item_id and item_id not in seen_ids:
                    seen_ids.add(item_id)
                    all_assets.append(item)

            if self.verbosity >= 1:
                print(f"    ✅ [UCI Stage 1] API fetch complete. Retrieved {len(all_assets)} unique datasets.")

            return all_assets

        except Exception as e:
            print(f"    ❌ [UCI Stage 1 Error] {type(e).__name__}: {e}")
            return []

    @profiler.track("Harvester: UCI HTML Fetch")
    def fetch_extract_and_screen_metadata(self, asset_identifiers: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Iterates through discovered assets, visits their specific HTML page,
        extracts the embedded JSON metadata, and screens them for the "Time-Series" tag.
        """
        print("\n🚜 [UCI Stage 2/3] HTML Fetch, Extraction, and Screening...")

        enriched_assets = []
        screened_out_count = 0
        failure_count = 0
        consecutive_failures = 0
        last_referer = self.WEB_UI_BASE_LIST

        # Allow user to artificially limit the run size via configuration
        MAX_ASSETS = getattr(self.config, 'HDK_LIMIT_ASSETS', None)
        if MAX_ASSETS and len(asset_identifiers) > MAX_ASSETS:
            print(f"    ⚠️ [Limit] Processing only the first {MAX_ASSETS} assets as per config.")
            asset_identifiers = asset_identifiers[:MAX_ASSETS]
        else:
             print(f"    💡 Processing {len(asset_identifiers)} assets. (Estimated time: 10-15 minutes)")

        for i, asset in enumerate(asset_identifiers):
            asset_id = asset.get('ID')
            asset_slug = asset.get('Slug')
            asset_name = asset.get('Name', "").strip()

            # Formulate a fallback name for logging
            fallback_name = asset_name
            if not fallback_name:
                if asset_slug:
                    fallback_name = asset_slug
                else:
                    fallback_name = f"ID {asset_id}"

            if not asset_id:
                continue

            # -------------------------------------------------------------
            # STRICT ANTI-GARBAGE FILTER:
            # Skip if API returned broken data or literal placeholder strings
            # -------------------------------------------------------------
            if not asset_name or str(asset_name).strip().lower() in ["null", "none", "nan", "missing", "[missing]", ""]:
                if self.verbosity >= 2:
                    print(f"    ⚠️ [Filter] Skipping ID {asset_id} due to missing or garbage name.")
                continue

            # Polite scraping delay
            time.sleep(random.uniform(self.FETCH_DELAY_MIN, self.FETCH_DELAY_MAX))

            if self.verbosity >= 1:
                print(f"    🔍 [Fetch] ({i+1}/{len(asset_identifiers)}) '{fallback_name}'...")

            slug_part = asset_slug if asset_slug else ""
            html_page_url = f"{self.BASE_URL}/dataset/{asset_id}/{slug_part}".rstrip('/')

            try:
                headers = self._get_randomized_headers(referer=last_referer)
                response = self.session.get(html_page_url, headers=headers, timeout=30)
                response.raise_for_status()
                last_referer = html_page_url

                if self._is_blocked(response.text):
                    consecutive_failures += 1
                    failure_count += 1
                    continue

                soup = BeautifulSoup(response.content, BS_PARSER)

                # 1. Try to get pristine metadata from embedded JSON
                extracted_blocks = self._extract_and_identify_json_blocks(soup)
                raw_metadata = extracted_blocks.get('metadata')
                raw_variables = extracted_blocks.get('variables')

                if raw_metadata:
                    metadata = self._map_embedded_json_to_standard_structure(raw_metadata, raw_variables)

                    # 2. Augment missing variables with Deep HTML Scraping
                    metadata = self._augment_metadata_from_html(metadata, soup)

                    # 3. Screen for "Time-Series"
                    if self._is_target_type(metadata):
                        if self.verbosity >= 1:
                            if raw_variables:
                                var_status = "(+Variables Table)"
                            else:
                                var_status = ""
                            print(f"    ✅ [MATCH] Target '{self.TARGET_DATA_TYPE}' Identified! {var_status}")

                        # Fallback for naming
                        if not metadata.get('name') or metadata.get('name') in ["[missing]", ""]:
                            metadata['name'] = fallback_name

                        metadata['_source_method'] = 'deep_scrape_v11.1'
                        enriched_assets.append(metadata)
                        consecutive_failures = 0
                    else:
                        if self.verbosity >= 2:
                            print(f"    ❌ [Filter] Dropping - Not '{self.TARGET_DATA_TYPE}'")
                        screened_out_count += 1
                        consecutive_failures = 0
                else:
                    # JSON failed completely
                    consecutive_failures += 1
                    failure_count += 1

            except Exception as e:
                if self.verbosity >= 2:
                    print(f"    ⚠️ [Fetch Error] {e}")
                consecutive_failures += 1
                failure_count += 1

            # Adaptive Backoff if we are hitting rate limits
            if consecutive_failures >= self.MAX_CONSECUTIVE_FAILURES:
                print(f"    ⚠️ [Backoff] {self.MAX_CONSECUTIVE_FAILURES} consecutive failures. Pausing {self.ADAPTIVE_BACKOFF_TIME}s to avoid ban.")
                time.sleep(self.ADAPTIVE_BACKOFF_TIME)
                consecutive_failures = 0

        print(f"\n🚜 [UCI Stage 2/3] Finished. Matches: {len(enriched_assets)}, Screened Out: {screened_out_count}, Failures: {failure_count}.")
        return enriched_assets

    def _augment_metadata_from_html(self, metadata: Dict[str, Any], soup: BeautifulSoup) -> Dict[str, Any]:
        """
        Scrapes the physical HTML of the UCI page to find variables and instances
        if the embedded JSON payload forgot to include them.
        """
        if metadata.get('num_instances') and metadata.get('num_features'):
            return metadata

        try:
            headers = soup.find_all(['h1', 'h2', 'h3', 'p', 'span', 'div'])

            # Scrape Features
            if not metadata.get('num_features'):
                for elem in headers:
                    text = elem.get_text(strip=True).lower()
                    if text == "features" or "number of features" in text:
                        parent = elem.find_parent()
                        if parent:
                            parent_text = parent.get_text(separator=" ", strip=True)
                            digits = re.findall(r'\d+', parent_text)
                            if digits:
                                metadata['num_features'] = int(digits[-1])
                                break

            # Scrape Instances (Time points)
            if not metadata.get('num_instances'):
                for elem in headers:
                    text = elem.get_text(strip=True).lower()
                    if text == "instances" or "number of instances" in text:
                        parent = elem.find_parent()
                        if parent:
                            parent_text = parent.get_text(separator=" ", strip=True)
                            digits = re.findall(r'\d+', parent_text)
                            if digits:
                                metadata['num_instances'] = int(digits[-1])
                                break
        except Exception:
            pass

        # If NumFeatures is STILL missing, try to count the rows in the HTML variable table
        if not metadata.get('num_features'):
            try:
                tables = soup.find_all('table')
                for table in tables:
                    th = table.find('th')
                    if th and ('variable' in th.get_text(strip=True).lower() or 'feature' in th.get_text(strip=True).lower()):
                        rows = table.find_all('tr')
                        if len(rows) > 1:
                            metadata['num_features'] = len(rows) - 1
                            break
            except Exception:
                pass

        return metadata

    def _transform_asset(self, metadata: Dict[str, Any]) -> Optional[Dict[str, CellData]]:
        """
        Converts the raw extracted metadata dictionary into the
        standardized DeepCollector CatalogState CellData format.
        """
        if not self._is_target_type(metadata):
            return None

        item: Dict[str, CellData] = {}
        source_method = metadata.get('_source_method', 'unknown')
        telemetry_base = f"Harvested via UCI ({source_method})"

        def create_cell(value: Any, confidence: float, context_detail: str = "") -> CellData:
            if pd.isnull(value) or value is None:
                str_value = "[missing]"
                confidence = 0.0
            else:
                str_value = str(value).strip()

            if not str_value or str_value.lower() == "none" or str_value.lower() == "nan":
                str_value = "[missing]"
                confidence = 0.0

            context = f"{telemetry_base}. {context_detail}".strip()

            if CellData == dict:
                return {"value": str_value, "confidence": confidence, "telemetry_context": context, "anchor_ref_id": None}
            else:
                try:
                    return CellData(value=str_value, confidence=confidence, telemetry_context=context, anchor_ref_id=None)
                except TypeError:
                    return {"value": str_value, "confidence": confidence, "telemetry_context": context, "anchor_ref_id": None}

        name = metadata.get('name')
        if not name:
            return None

        if str(name).lower().strip() in ["null", "none", "nan", "[missing]", "missing", ""]:
            return None

        # --- Base Identifiers (Bumped to 1.00) ---
        item["Dataset Name"] = create_cell(name, 1.00, "Field: Name")
        item["Canonical Name"] = create_cell(name, 1.00, "Field: Name")
        item["Domain"] = create_cell(metadata.get('subject_area'), 1.00, "Field: Area")

        # EXPLICITLY SET THE TYPE
        item["Type"] = create_cell("Dataset", 1.00, "Hardcoded")

        # --- Description Assembly ---
        desc_parts = []
        abstract_text = metadata.get('abstract', '')
        info_text = metadata.get('additional_info', '')

        if abstract_text:
            desc_parts.append(abstract_text)

        if metadata.get('creators'):
            creators_joined = ', '.join(metadata['creators'])
            desc_parts.append(f"Creators: {creators_joined}")

        formatted_vars, time_heur = self._format_variable_info(metadata.get('variable_info'), metadata.get('name'))
        if formatted_vars:
            desc_parts.append(" | Variables: " + formatted_vars.replace('\n', ' '))

        # --- FIX 4: Strip all newlines to maintain Single Line Rule for Google Sheets ---
        final_desc = " | ".join(desc_parts).replace('\n', ' ').replace('\r', '')
        final_desc = re.sub(r'\s{2,}', ' ', final_desc).strip()
        item["Detailed Description"] = create_cell(final_desc, 1.00, "Fields: Abstract, Variables")

        # --- Time Interval Heuristics (Bumped to 1.00 to avoid repair loops) ---
        text_heur, text_conf = self._extract_time_interval_heuristic_text(metadata)
        if time_heur:
            item["Time interval between points"] = create_cell(time_heur, 1.00, "Heuristic: Variables Table")
        elif text_heur:
            item["Time interval between points"] = create_cell(text_heur, 1.00, "Heuristic: Abstract Text Regex")
        else:
            item["Time interval between points"] = create_cell("[missing]", 0.0)

        # --- Dimensionality Heuristics (Bumped to 1.00) ---
        raw_num_instances = metadata.get('num_instances')
        item["Number of Time Points"] = create_cell(raw_num_instances, 1.00, "Field: NumInstances")
        item["Number of Locations/Series"] = create_cell(1, 1.00, "Default (Single Series assumed)")

        num_features = metadata.get('num_features')
        if not num_features:
            var_info = metadata.get('variable_info')
            if var_info:
                if 'data' in var_info:
                    num_features = len(var_info['data'])

        item["Variables per Location"] = create_cell(num_features, 1.00, "Field: NumFeatures")
        item["Total Variables"] = create_cell(num_features, 1.00, "Field: NumFeatures")

        # --- Links & Provenance (Deterministic & 1.00 Confidence) ---
        item["Primary Source Repository"] = create_cell("UCI Machine Learning Repository", 1.00, "Source: UCI")

        uci_id = metadata.get('uci_id')
        doi = metadata.get('doi')

        primary_url = f"https://archive.ics.uci.edu/dataset/{uci_id}" if uci_id else ""
        data_url = f"https://archive.ics.uci.edu/static/public/{uci_id}/data.zip" if uci_id else ""
        other_url = f"https://doi.org/{doi}" if doi else ""

        item["Primary URL"] = create_cell(primary_url, 1.00 if primary_url else 0.0, "Deterministic UCI URL")
        item["Link to Data (Actual Source)"] = create_cell(data_url, 1.00 if data_url else 0.0, "Deterministic UCI URL")
        item["Other URL"] = create_cell(other_url, 1.00 if other_url else 0.0, "Deterministic UCI URL")

        item["Assignment Confidence"] = create_cell(1.00, 1.00, "Time-Series Filter")
        item["Assignment Rationale"] = create_cell(f"Explicitly identified as '{self.TARGET_DATA_TYPE}'", 1.00)

        return item

    def _update_state(self, state: CatalogState, enriched_assets: List[Dict[str, Any]]):
        """Pushes transformed datasets into the active CatalogState."""
        print(f"\n🚜 [UCI Stage 4] State Update ({len(enriched_assets)} items)...")
        transformed_items = []

        for metadata in enriched_assets:
            try:
                item = self._transform_asset(metadata)
                if item:
                    transformed_items.append(item)
            except Exception as e:
                if self.verbosity >= 2:
                    print(f"      ⚠️ [Transform Error] Failed to map asset: {e}")

        if transformed_items:
            if hasattr(state, 'update_catalog_batch'):
                state.update_catalog_batch(transformed_items, allow_new_datasets=True)

        print(f"🚜 [UCI Stage 4] Updated {len(transformed_items)} assets in catalog.")

    def _export_and_summarize(self, state: CatalogState):
        """Export is handled globally by the Agent in Phase 4."""
        pass

    def _standardize_keys(self, assets):
        """Normalizes dictionary keys from the raw API response."""
        standardized = []
        for i in assets:
            new_item = {}
            new_item['ID'] = i.get('id') or i.get('ID')
            new_item['Name'] = i.get('name') or i.get('Name')
            new_item['Slug'] = i.get('slug') or i.get('Slug')
            standardized.append(new_item)
        return standardized

    def _extract_and_identify_json_blocks(self, soup):
        """Fully unrolled JSON extraction from script tags embedded in Next.js page."""
        results = {'metadata': None, 'variables': None}

        next_data_tag = soup.find('script', id='__NEXT_DATA__')
        if next_data_tag and next_data_tag.string:
            try:
                next_data = json.loads(next_data_tag.string)
                props = next_data.get('props', {})
                page_props = props.get('pageProps', {})
                dataset_obj = page_props.get('dataset', {})

                if dataset_obj:
                    results['metadata'] = dataset_obj

                    if 'Variables' in dataset_obj:
                        results['variables'] = {'data': dataset_obj['Variables']}
            except Exception:
                pass

        if not results['metadata']:
            for tag in soup.find_all('script', {'type': 'application/json'}):
                if tag.string:
                    try:
                        data = json.loads(tag.string)
                        if isinstance(data, dict):
                            if 'body' in data:
                                body_str = data['body']
                                body = json.loads(body_str)

                                if isinstance(body, list):
                                    for item in body:
                                        if 'result' in item:
                                            result_obj = item['result']
                                            if 'data' in result_obj:
                                                data_obj = result_obj['data']
                                                if 'json' in data_obj:
                                                    j = data_obj['json']

                                                    if 'ID' in j and 'Abstract' in j:
                                                        results['metadata'] = j

                                                    if 'headers' in j:
                                                        headers_list = j.get('headers', [])
                                                        headers_lower = [str(h).lower() for h in headers_list]
                                                        if 'role' in headers_lower or 'type' in headers_lower:
                                                            results['variables'] = j
                    except Exception:
                        continue

        return results

    def _map_embedded_json_to_standard_structure(self, meta, vars):
        """Maps varying JSON schemas into a consistent dictionary for the Agent."""
        m = {
            'uci_id': meta.get('ID') or meta.get('id'),
            'doi': meta.get('DOI') or meta.get('doi'), # ADDED TO CAPTURE DOI SAFELY
            'name': meta.get('Name') or meta.get('name'),
            'abstract': meta.get('Abstract') or meta.get('abstract'),
            'num_instances': meta.get('NumInstances') or meta.get('numInstances'),
            'num_features': meta.get('NumFeatures') or meta.get('numFeatures'),
            'subject_area': meta.get('Area') or meta.get('area'),
            'additional_info': meta.get('DatasetInfo') or meta.get('datasetInfo'),
            'variable_info': vars
        }

        def parse(field_name):
            val = meta.get(field_name) or meta.get(field_name.lower())
            if isinstance(val, list):
                return [str(v) for v in val if v]
            if val:
                return [v.strip() for v in str(val).split(',')]
            return []

        m['characteristics'] = parse('Types')
        if not m['characteristics']:
            m['characteristics'] = parse('characteristics')

        m['associated_tasks'] = parse('Task')
        if not m['associated_tasks']:
            m['associated_tasks'] = parse('tasks')

        return m

    def _format_variable_info(self, vars_dict, name=None):
        """Builds a markdown table from the variable data and looks for frequency hints."""
        if not vars_dict:
            return None, None

        if not vars_dict.get('data'):
            return None, None

        rows = vars_dict['data']
        headers = vars_dict.get('headers', [])
        heur = None

        for row in rows:
            if isinstance(row, dict):
                row_vals = list(row.values())
            else:
                row_vals = row

            txt = " ".join([str(x) for x in row_vals])
            match = re.search(r'\b(hourly|daily|weekly|monthly|quarterly|yearly|hz|seconds|minutes|days)\b', txt, re.I)
            if match:
                heur = match.group(1).capitalize()
                break

        if tabulate and isinstance(rows[0], list):
            return tabulate(rows, headers=headers), heur

        return str(rows), heur

    def _extract_time_interval_heuristic_text(self, metadata):
        """Searches the abstract and additional info for frequency keywords."""
        abstract = str(metadata.get('abstract', ''))
        additional = str(metadata.get('additional_info', ''))
        text = (abstract + " " + additional).lower()

        patterns = [
            (r'(\d+\s*khz|\d+\s*hz)', 0.95),
            (r'sampled at (\d+\s*hz|\d+\s*khz)', 0.95),
            (r'\b(hourly|daily|weekly|monthly|yearly|quarterly|minutes?|seconds?|ms)\b', 0.90),
            (r'every (\d+\s*(?:minutes?|seconds?|hours?|days?|ms))', 0.90),
            (r'frequency of (\d+\s*(?:hz|khz|minutes?|seconds?|hours?|days?))', 0.85)
        ]

        for p, c in patterns:
            m = re.search(p, text)
            if m:
                return m.group(1).strip().capitalize(), c

        return None, 0.0

    def _is_target_type(self, meta):
        """Confirms the dataset actually has 'Time-Series' in its characteristics list."""
        characteristics = meta.get('characteristics', [])
        char_list_lower = [str(c).lower() for c in characteristics]
        return self.TARGET_DATA_TYPE.lower() in char_list_lower

    def _get_randomized_headers(self, referer=None):
        h = {'User-Agent': random.choice(self.USER_AGENTS)}
        if referer:
            h['Referer'] = referer
        return h

    def _parse_json_response(self, resp):
        try:
            return resp.json()
        except Exception:
            return self._extract_json_robustly_from_text(resp.text)

    def _extract_json_robustly_from_text(self, txt):
        # FIX 6: Use .*? for non-greedy evaluation so it doesn't consume the entire HTML document
        m = re.search(r'{\s*".*?":.*?}', txt, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
        return None

    def _is_blocked(self, txt):
        if "Access denied" in txt:
            return True
        if "Cloudflare" in txt:
            return True
        return False

print("✅ deepcollector/harvesting/uci_harvester.py written (V11.2: Deep Think Fix: Single Line Enforcer).")

Writing deepcollector/harvesting/uci_harvester.py


#### **Cell 12: KnowledgeBaseManager `deepcollector/kb/manager.py`**

In [ ]:
%%writefile deepcollector/kb/manager.py
# =============================================================================
# V86.25: KnowledgeBaseManager (100% Full Un-Truncated + GSpread V5/V6 Fix)
# =============================================================================
import pandas as pd
import re
import uuid
import time
import json
import ast
import random
import traceback
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Tuple
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

try:
    import pytz
except ImportError:
    pytz = None

try:
    import gspread
    from gspread.exceptions import SpreadsheetNotFound, WorksheetNotFound, APIError
except ImportError:
    gspread = None
    SpreadsheetNotFound = Exception
    WorksheetNotFound = Exception
    APIError = Exception

try:
    from deepcollector.utils.profiler import profiler
except ImportError:
    class DummyProfiler:
        def track(self, c):
            return lambda f: f
    profiler = DummyProfiler()

class SheetLock:
    LOCK_PREFIX = "sys_Lock_GLOBAL_MUTEX"
    EXPIRY_MINUTES = 10

    def __init__(self, manager, job_id, verbosity=1):
        self.manager = manager
        self.job_id = job_id
        self.verbosity = verbosity
        # Static name guarantees 400 APIError on concurrent creation attempts
        self.lock_name = self.LOCK_PREFIX

    def acquire(self, timeout_seconds=600, poll_interval=10) -> bool:
        start_time = time.time()

        if self.verbosity >= 1:
            print(f"🔒 [Lock] Job {self.job_id} requesting truly atomic write lock...")

        while (time.time() - start_time) < timeout_seconds:
            try:
                # 1. Atomic Collision: Only the first execution succeeds
                ws = self.manager.spreadsheet.add_worksheet(title=self.lock_name, rows=1, cols=1)
                ws.update_acell('A1', f"{self.job_id}_{int(time.time())}")
                if self.verbosity >= 1:
                    print(f"    🔒 [Lock] Acquired safely by {self.job_id}.")
                return True
            except Exception:
                try:
                    # 2. If locked, evaluate for zombie expiry safely
                    ws = self.manager.spreadsheet.worksheet(self.lock_name)
                    val = ws.acell('A1').value
                    if val:
                        lock_timestamp = int(val.split("_")[-1])
                        if time.time() - lock_timestamp > (self.EXPIRY_MINUTES * 60):
                            if self.verbosity >= 1:
                                print(f"    ⚠️ [Lock] Breaking expired zombie lock: {ws.title}")
                            self.manager.spreadsheet.del_worksheet(ws)
                except Exception:
                    pass

                if self.verbosity >= 1:
                    print(f"    ⏳ [Lock] Collision or API error. Waiting {poll_interval}s...")
                time.sleep(poll_interval)

        print("❌ [Lock] Failed to acquire lock (Timeout).")
        return False

    def release(self):
        try:
            ws = self.manager.spreadsheet.worksheet(self.lock_name)
            self.manager.spreadsheet.del_worksheet(ws)
            if self.verbosity >= 1:
                print(f"    🔓 [Lock] Released safely.")
        except Exception:
            pass


class KnowledgeBaseManager:
    def __init__(self, config: Any):
        self.config = config
        self.sheet_id = getattr(config, 'GOOGLE_SHEET_KB_ID', None)
        self.kb_data: Dict[str, pd.DataFrame] = {}
        self.gc = None
        self.spreadsheet = None
        self.is_connected = False
        self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)
        self.KB_SCHEMA = getattr(config, 'KB_SCHEMA', {})
        self.CATALOG_SCHEMA = getattr(config, 'CATALOG_SCHEMA', {})
        self.EXTRACTED_FIELDS = getattr(config, 'EXTRACTED_FIELDS', [])
        self.MISSING_DATA_PLACEHOLDERS = getattr(config, 'MISSING_DATA_PLACEHOLDERS', set())

        self.FIELD_CONFIDENCE_MAP = {
            "Domain": "Domain (C)",
            "Frequency": "Frequency (C)",
            "Num Time Points": "Num Time Points (C)",
            "Num Locations/Series": "Num Locations/Series (C)",
            "Variables per Location": "Variables per Location (C)",
            "Total Variables": "Total Variables (C)",
            "Primary URL": "Primary URL (C)",
            "Link to Data (Actual Source)": "Link to Data (Actual Source) (C)",
            "Other URL": "Other URL (C)"
        }

    def initialize_connection(self, authenticated_gc: Any):
        if not self.sheet_id or not authenticated_gc:
            if self.verbosity >= 1:
                print("⚠️ [KB Manager] Missing ID or Client. Disabled.")
            return

        self.gc = authenticated_gc
        try:
            self.spreadsheet = self.gc.open_by_key(self.sheet_id)
            self.is_connected = True
            if self.verbosity >= 1:
                print(f"🌐 [KB Manager] Connected to '{self.spreadsheet.title}'")
        except Exception as e:
            print(f"❌ [KB Manager] Connection failed: {e}")

    @profiler.track("KB: Read")
    def read_and_validate_kb(self) -> bool:
        if not self.is_connected:
            return False

        valid = True
        for tab, cols in self.KB_SCHEMA.items():
            try:
                ws = self.spreadsheet.worksheet(tab)

                # =================================================================
                # CRITICAL FIX: Safe handling for gspread v6+
                # =================================================================
                try:
                    if hasattr(ws, 'get_values'):
                        data = ws.get_values(value_render_option='FORMULA')
                    else:
                        try:
                            data = ws.get_all_values(value_render_option='FORMULA')
                        except TypeError:
                            data = ws.get_all_values()
                except Exception:
                    data = ws.get_all_values()

                if data:
                    best_match = 0
                    header_idx = 0
                    for i, row in enumerate(data[:10]):
                        match_count = sum(1 for c in cols if c in row)
                        if match_count > best_match:
                            best_match = match_count
                            header_idx = i

                    if best_match == 0:
                        if self.verbosity >= 1:
                            print(f"    ⚠️ [KB] Header row not found in '{tab}'. Falling back to empty DataFrame.")
                        self.kb_data[tab] = pd.DataFrame(columns=cols)
                        continue

                    df = pd.DataFrame(data[header_idx+1:], columns=data[header_idx])

                    if "DatasetID" in cols and "DatasetID" not in df.columns:
                        print(f"    🚨 [CRITICAL] 'DatasetID' column missing from {tab} headers! Check Google Sheet for structural corruption.")

                    for col in cols:
                        if col not in df.columns:
                            df[col] = ""

                    def extract_url(val):
                        if isinstance(val, str) and val.startswith('=HYPERLINK'):
                            import re
                            urls = re.findall(r'(https?://[^\s|\"]+)', val)
                            unique_urls = []
                            for u in urls:
                                if u not in unique_urls:
                                    unique_urls.append(u)
                            return ", ".join(unique_urls) if unique_urls else val
                        return val

                    for col in df.columns:
                        if "URL" in col or "Link" in col:
                            df[col] = df[col].apply(extract_url)

                    df = df.reindex(columns=cols).fillna("").astype(str)
                    self.kb_data[tab] = df
                    if self.verbosity >= 1:
                        print(f"    ✅ [KB] Loaded '{tab}' ({len(df)} rows).")
                else:
                    self.kb_data[tab] = pd.DataFrame(columns=cols)
            except WorksheetNotFound:
                if self.verbosity >= 1:
                    print(f"    ⚠️ [KB] Tab '{tab}' not found. Initializing empty.")
                self.kb_data[tab] = pd.DataFrame(columns=cols)
            except Exception as e:
                print(f"    ❌ [KB] Error reading '{tab}': {e}")
                valid = False
        return valid

    def get_kb_data(self, tab_name: str) -> pd.DataFrame:
        return self.kb_data.get(tab_name, pd.DataFrame())

    def wipe_kb(self):
        if not self.is_connected:
            return
        print("⚠️ [KB] Wiping all data tabs...")
        for tab, cols in self.KB_SCHEMA.items():
            try:
                try:
                    ws = self.spreadsheet.worksheet(tab)
                except WorksheetNotFound:
                    ws = self.spreadsheet.add_worksheet(title=tab, rows=100, cols=20)
                ws.clear()
                ws.append_row(cols)
                self.kb_data[tab] = pd.DataFrame(columns=cols)
                print(f"    🗑️ Wiped '{tab}'.")
            except Exception as e:
                print(f"    ❌ Failed to wipe '{tab}': {e}")

    def wipe_project(self, project_id: str, job_id: str):
        if not self.is_connected:
            return
        print(f"\n⚠️ [KB] Executing Targeted Wipe for Project: {project_id}...")
        lock = SheetLock(self, job_id, self.verbosity)
        if not lock.acquire():
            print("    ❌ Failed to acquire lock for project wipe.")
            return

        try:
            self.read_and_validate_kb()
            df_links = self.get_kb_data("Project_Dataset_Link")
            df_ds = self.get_kb_data("Datasets")

            if df_links.empty or df_ds.empty:
                print("    ℹ️ No data to wipe.")
                return

            proj_links = df_links[df_links["ProjectID"] == project_id]
            if proj_links.empty:
                print("    ℹ️ Project has no linked datasets to wipe.")
                return

            ds_ids_to_check = proj_links["DatasetID"].tolist()
            df_links_clean = df_links[df_links["ProjectID"] != project_id]
            remaining_links = df_links_clean["DatasetID"].tolist()
            ds_ids_to_delete = [ds for ds in ds_ids_to_check if ds not in remaining_links]

            df_ds_clean = df_ds[~df_ds["DatasetID"].isin(ds_ids_to_delete)]

            self.write_dataframe_to_tab("Project_Dataset_Link", df_links_clean)
            self.write_dataframe_to_tab("Datasets", df_ds_clean)
            self.kb_data["Project_Dataset_Link"] = df_links_clean
            self.kb_data["Datasets"] = df_ds_clean

            print(f"    🗑️ Safely Deleted {len(ds_ids_to_delete)} exclusive datasets and {len(proj_links)} links for {project_id}.")
        except Exception as e:
            print(f"    ❌ Wipe Failed: {e}")
        finally:
            lock.release()

    @staticmethod
    def _col_num_to_letter(n):
        string = ""
        while n > 0:
            n, remainder = divmod(n - 1, 26)
            string = chr(65 + remainder) + string
        return string

    @profiler.track("KB: Write")
    @retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=2, min=2, max=20), retry=retry_if_exception_type((APIError, Exception)))
    def write_dataframe_to_tab(self, tab_name: str, df: pd.DataFrame):
        if not self.is_connected:
            return

        try:
            try:
                ws = self.spreadsheet.worksheet(tab_name)
            except WorksheetNotFound:
                if tab_name in self.KB_SCHEMA:
                    ws = self.spreadsheet.add_worksheet(title=tab_name, rows=100, cols=20)
                else:
                    return

            cols = self.KB_SCHEMA.get(tab_name, df.columns.tolist())
            df_write = df.reindex(columns=cols)

            def safe_stringify(val):
                # 1. Process iterables BEFORE pd.isna evaluates
                if isinstance(val, (list, dict, tuple, set)):
                    try:
                        # Join lists to ensure val.startswith("http") triggers downstream
                        if isinstance(val, list):
                            val = ", ".join(str(v) for v in val)
                        else:
                            val = json.dumps(val)
                    except Exception:
                        val = str(val)

                # 2. Now guaranteed scalar/string, pd.isna evaluation is safe
                if pd.isna(val) or val is None:
                    return "[missing]"

                s = str(val).strip()
                if not s:
                    return "[missing]"
                return s

            for col in df_write.columns:
                df_write[col] = df_write[col].apply(safe_stringify)

            def sanitize_cell(col_name, val):
                val = str(val).strip()
                if ("URL" in col_name or "Link" in col_name) and val.startswith("http") and not val.startswith("="):
                    urls = [u.strip() for u in re.split(r'[,|]', val) if u.strip()]
                    if not urls: return f"'{val}"

                    primary_url = str(urls)
                    safe_url = primary_url.replace('"', '""').replace('\n', '').replace('\r', '')

                    if len(urls) == 1:
                        return f'=HYPERLINK("{safe_url}", "{safe_url}")'
                    else:
                        secondary_text = " | ".join([f"Alt: {str(u).replace('\"', '\"\"').replace('\\n', '').replace('\\r', '')}" for u in urls[1:]])
                        return f'=HYPERLINK("{safe_url}", "{safe_url}") & " | {secondary_text}"'
                else:
                    if val.lstrip().startswith(('=', '+', '-', '@')):
                        return f"'{val}"
                    return val

            for col in df_write.columns:
                df_write[col] = df_write[col].apply(lambda x: sanitize_cell(col, x))

            data = [df_write.columns.values.tolist()] + df_write.values.tolist()
            ws.clear()

            num_rows = len(data)
            num_cols = len(data) if num_rows > 0 else 1

            try:
                ws.resize(rows=max(100, num_rows + 5), cols=max(20, num_cols + 2))
            except Exception:
                pass

            range_addr = "A1"

            try:
                ws.update(values=data, range_name=range_addr, value_input_option='USER_ENTERED')
            except TypeError:
                try:
                    ws.update(range_addr, data, value_input_option='USER_ENTERED')
                except TypeError:
                    ws.update(data, value_input_option='USER_ENTERED')

            if self.verbosity >= 1:
                print(f"    📝 [KB] Wrote {num_rows - 1} records to '{tab_name}'.")

        except Exception as e:
            if self.verbosity >= 1:
                print(f"    ⚠️ [KB Retry] Write to '{tab_name}' failed: {type(e).__name__} - {e}")
                print(f"    🐛 [DEBUG TRACE]:\n{traceback.format_exc()}")
            raise e

    def log_job_execution(self, job_data: Dict[str, Any]):
        if not self.is_connected:
            return

        try:
            tab_name = "Jobs"
            try:
                ws = self.spreadsheet.worksheet(tab_name)
            except WorksheetNotFound:
                ws = self.spreadsheet.add_worksheet(title=tab_name, rows=100, cols=20)
                ws.append_row(self.KB_SCHEMA["Jobs"])

            row = []
            for col in self.KB_SCHEMA["Jobs"]:
                val = job_data.get(col, "")
                if isinstance(val, (dict, list)):
                    val = json.dumps(val)
                row.append(str(val))

            ws.append_row(row)
            if self.verbosity >= 1:
                print(f"    📋 [Job Log] Recorded Job {job_data.get('JobID')} to KB.")
        except Exception as e:
            print(f"    ⚠️ [Job Log Error] Failed to log job: {e}")

    @profiler.track("KB: Reconcile & Write")
    def reconcile_and_write(self, state, current_project_name, job_id=None):
        if not self.is_connected:
            return

        job_id = job_id or f"JOB_{uuid.uuid4().hex[:6]}"

        lock = SheetLock(self, job_id, self.verbosity)
        if not lock.acquire():
            raise RuntimeError("Failed to acquire KB Lock for write-back.")

        try:
            if self.verbosity >= 1:
                print(f"🔄 [KB] JIT Refresh: Downloading latest KB state under lock...")
            if not self.read_and_validate_kb():
                raise RuntimeError("Failed to refresh KB data under lock.")

            if self.verbosity >= 1:
                print(f"🔄 [KB Reconciliation] Processing {len(state.catalog)} items (Job: {job_id})...")

            kb_dfs = {name: df.copy() for name, df in self.kb_data.items()}
            current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            self._reconcile_projects(state, kb_dfs, current_project_name, current_time)
            citation_ids = self._reconcile_citations(state, kb_dfs)
            weblink_ids = self._reconcile_weblinks(state, kb_dfs)
            link_info = self._reconcile_datasets(state, kb_dfs, current_time, job_id)
            self._manage_linkages(state, kb_dfs, link_info, citation_ids, weblink_ids, current_time, job_id)

            for tab, df in kb_dfs.items():
                self.write_dataframe_to_tab(tab, df)

            self.kb_data = kb_dfs

        except Exception as e:
            print(f"❌ [KB Write Error] Critical Failure: {e}")
            raise e
        finally:
            lock.release()

    def _reconcile_projects(self, state, kb_dfs, project_name, time_str):
        df = kb_dfs.get("Projects")
        if state.project_id in df["ProjectID"].values:
            mask = df["ProjectID"] == state.project_id
            df.loc[mask, "Last Analyzed"] = time_str
            df.loc[mask, "Name"] = project_name
        else:
            new_row = {
                "ProjectID": state.project_id,
                "Name": project_name,
                "Last Analyzed": time_str,
                "Type": "Project",
                "Source URL": getattr(self.config, 'INITIAL_URLS', ['']) if getattr(self.config, 'INITIAL_URLS', []) else ""
            }
            kb_dfs["Projects"] = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    def _reconcile_datasets(self, state, kb_dfs, time_str, job_id):
        df_ds = kb_dfs.get("Datasets")
        link_info = []

        for item in state.catalog:
            name = item.get("Dataset Name", {}).get("value")
            if not name or name == "[missing]":
                continue

            record, field_confs, assign_conf = self._normalize_dataset_entry(state, item, time_str)

            canon = record["Canonical Name"]
            variant = record["Variant Name"]
            mask = (df_ds["Canonical Name"].str.lower() == canon.lower()) & \
                   (df_ds["Variant Name"].str.lower() == variant.lower())

            ds_id = None
            if mask.any():
                idx = mask.idxmax()
                ds_id = df_ds.loc[idx, "DatasetID"]
                row_updated = False

                for field_col, conf_col in self.FIELD_CONFIDENCE_MAP.items():
                    new_val = record.get(field_col, "")
                    new_conf = field_confs.get(field_col, 0.0)

                    try:
                        old_conf = float(df_ds.loc[idx, conf_col])
                    except Exception:
                        old_conf = 0.0

                    if new_conf > old_conf and new_val not in self.MISSING_DATA_PLACEHOLDERS:
                        df_ds.loc[idx, field_col] = new_val
                        df_ds.loc[idx, conf_col] = str(new_conf)
                        row_updated = True

                if len(record["Description"]) > len(str(df_ds.loc[idx, "Description"])) + 10:
                    df_ds.loc[idx, "Description"] = record["Description"]
                    row_updated = True

                current_aliases = str(df_ds.loc[idx, "Aliases"])
                new_alias = record["Variant Name"]
                merged_aliases = self._merge_aliases(current_aliases, new_alias)
                if merged_aliases != current_aliases:
                    df_ds.loc[idx, "Aliases"] = merged_aliases
                    row_updated = True

                if row_updated:
                    df_ds.loc[idx, "Job_Updated"] = job_id
                    df_ds.loc[idx, "Project_Updated"] = state.project_id
                    df_ds.loc[idx, "Date_Updated"] = time_str

            else:
                ds_id = self._generate_new_id("DS_", df_ds, "DatasetID")
                record["DatasetID"] = ds_id
                record["Job_Created"] = job_id
                record["Project_Created"] = state.project_id
                record["Date_Created"] = time_str
                record["Job_Updated"] = job_id
                record["Project_Updated"] = state.project_id
                record["Date_Updated"] = time_str

                for field_col, conf_col in self.FIELD_CONFIDENCE_MAP.items():
                    record[conf_col] = str(field_confs.get(field_col, 0.0))

                df_ds = pd.concat([df_ds, pd.DataFrame([record])], ignore_index=True)

            link_info.append({
                "DatasetID": ds_id,
                "Actual Data URL Used": item.get("Link to Data (Actual Source)", {}).get("value", ""),
                "Assignment Confidence": f"{assign_conf:.4f}"
            })

        kb_dfs["Datasets"] = df_ds
        return link_info

    def _merge_aliases(self, current_str: str, new_val: str) -> str:
        if not current_str or current_str == "[missing]":
            current_str = ""
        if not new_val or new_val == "[missing]":
            return current_str

        existing = []
        for x in current_str.split(','):
            x = x.strip()
            if x and x.lower() not in self.MISSING_DATA_PLACEHOLDERS and "[missing]" not in x.lower():
                existing.append(x)

        existing_lower = {x.lower() for x in existing}

        if new_val.lower() not in existing_lower and new_val.lower() not in self.MISSING_DATA_PLACEHOLDERS and "[missing]" not in new_val.lower():
            existing.append(new_val)

        return ", ".join(existing) if existing else "[missing]"

    def _sanitize_dimensionality(self, time_val, var_val, dataset_name=None):
        val_str = str(var_val).strip()

        if (val_str.startswith("{") and val_str.endswith("}")) or (val_str.count(":") > 2 and "{" in val_str):
            try:
                start = val_str.find("{")
                end = val_str.rfind("}")
                if start != -1 and end != -1:
                    candidate = val_str[start:end+1]
                    data = ast.literal_eval(candidate)
                    if isinstance(data, dict):
                        if dataset_name:
                            target = dataset_name.lower()
                            if dataset_name in data:
                                return str(data[dataset_name])
                            for k, v in data.items():
                                if str(k).lower() in target or target in str(k).lower():
                                    return str(v)
                        return "[extraction_error: dict_returned]"
            except Exception:
                return "[extraction_error: complex_string]"

        if re.search(r'\d+\s*(?:-|to)\s*\d+', val_str, re.IGNORECASE):
            return val_str

        try:
            t_clean = "".join(filter(str.isdigit, str(time_val)))
            v_clean = "".join(filter(str.isdigit, str(val_str)))

            if not t_clean or not v_clean:
                return val_str

            t_int = int(t_clean)
            v_int = int(v_clean)
        except Exception:
            pass

        return val_str

    def _normalize_dataset_entry(self, state, item, time_str):
        name = item.get("Dataset Name", {}).get("value")

        field_confs = {}
        total_conf = 0.0
        count = 0

        CATALOG_TO_SCHEMA = {
            "Domain": "Domain",
            "Time interval between points": "Frequency",
            "Number of Time Points": "Num Time Points",
            "Number of Locations/Series": "Num Locations/Series",
            "Variables per Location": "Variables per Location",
            "Total Variables": "Total Variables",
            "Primary URL": "Primary URL",
            "Link to Data (Actual Source)": "Link to Data (Actual Source)",
            "Other URL": "Other URL",
            "Primary Source Repository": "Primary Creator"
        }

        for cat_field, schema_col in CATALOG_TO_SCHEMA.items():
            data = item.get(cat_field, {})
            val = data.get("value", "")
            conf = data.get("confidence", 0.0)
            if val in self.MISSING_DATA_PLACEHOLDERS:
                conf = 0.0

            field_confs[schema_col] = conf
            total_conf += conf
            count += 1

        avg_cell_conf = (total_conf / count) if count else 0
        try:
            assign_conf = float(item.get("Assignment Confidence", {}).get("value", 0))
        except Exception:
            assign_conf = 0.0

        overall_conf = assign_conf * avg_cell_conf

        def get_val(f):
            return item.get(f, {}).get("value", "")

        raw_time = get_val("Number of Time Points")
        raw_vars = get_val("Total Variables")
        raw_vars_loc = get_val("Variables per Location")

        clean_vars = self._sanitize_dimensionality(raw_time, raw_vars, name)
        clean_vars_loc = self._sanitize_dimensionality(raw_time, raw_vars_loc, name)

        ds_type = get_val("Type")
        if not ds_type or ds_type == "[missing]":
             ds_type = "Dataset"

        record = {
            "Canonical Name": get_val("Canonical Name") or name,
            "Variant Name": name,
            "Aliases": get_val("Aliases"),
            "Description": get_val("Detailed Description"),
            "Domain": get_val("Domain"),
            "Frequency": get_val("Time interval between points"),
            "Num Time Points": raw_time,
            "Num Locations/Series": get_val("Number of Locations/Series"),
            "Variables per Location": clean_vars_loc,
            "Total Variables": clean_vars,
            "Primary Creator": get_val("Primary Source Repository"),
            "Primary URL": get_val("Primary URL"),
            "Link to Data (Actual Source)": get_val("Link to Data (Actual Source)"),
            "Other URL": get_val("Other URL"),
            "Overall Confidence": f"{overall_conf:.4f}",
            "Type": ds_type
        }
        return record, field_confs, assign_conf

    def _manage_linkages(self, state, kb_dfs, link_info, cite_ids, web_ids, time_str, job_id):
        pid = state.project_id

        df_pdl = kb_dfs.get("Project_Dataset_Link")
        df_pdl = df_pdl[df_pdl["ProjectID"] != pid]

        for info in link_info:
            new = {
                **info,
                "LinkID": self._generate_new_id("PDL_", df_pdl, "LinkID"),
                "ProjectID": pid,
                "Link_Date": time_str,
                "Linked_By_Job": job_id
            }
            df_pdl = pd.concat([df_pdl, pd.DataFrame([new])], ignore_index=True)

        kb_dfs["Project_Dataset_Link"] = df_pdl

    def _reconcile_citations(self, state, kb_dfs):
        df = kb_dfs.get("Citations")
        ids = {}
        for c in state.discovered_citations:
            title = c.get("Title", "").strip()
            if not title: continue

            match = df[df["Title"].str.lower() == title.lower()]
            if not match.empty:
                ids[title] = match.iloc["CitationID"]
            else:
                new_id = self._generate_new_id("CITE_", df, "CitationID")
                row = {
                    "CitationID": new_id,
                    "Title": title,
                    "Authors": c.get("Authors", ""),
                    "Venue": c.get("Venue", ""),
                    "Year": c.get("Year", ""),
                    "DOI": c.get("DOI", ""),
                    "URL": c.get("URL", ""),
                    "Full Citation Text": c.get("Full Citation Text", "")
                }
                df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
                ids[title] = new_id

        kb_dfs["Citations"] = df
        return ids

    def _reconcile_weblinks(self, state, kb_dfs):
        df = kb_dfs.get("WebLinks")
        ids = {}
        for w in state.discovered_weblinks:
            url = w.get("URL", "").strip()
            if not url: continue

            match = df[df["URL"] == url]
            if not match.empty:
                ids[url] = match.iloc["WebLinkID"]
            else:
                new_id = self._generate_new_id("WEB_", df, "WebLinkID")
                row = {
                    "WebLinkID": new_id,
                    "URL": url,
                    "Resource Type": w.get("Resource Type", ""),
                    "Description": w.get("Description", "")
                }
                df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
                ids[url] = new_id

        kb_dfs["WebLinks"] = df
        return ids

    def _generate_new_id(self, prefix, df, col):
        if df.empty or col not in df.columns:
            return f"{prefix}001"
        try:
            nums = df[col].astype(str).str.extract(rf"{prefix}(\d+)").dropna().astype(int)
            return f"{prefix}{int(nums.max().iloc) + 1:03d}" if not nums.empty else f"{prefix}001"
        except Exception:
            return f"{prefix}{uuid.uuid4().hex[:6]}"

print("✅ deepcollector/kb/manager.py written (100% Full Un-Truncated + GSpread V5/V6 Safe Fetch).")

Writing deepcollector/kb/manager.py


#### **Cell 13: `deepcollector/core/state.py`**

In [ ]:
%%writefile deepcollector/core/state.py
# =============================================================================
# V33: CatalogState (LlamaIndex Metadata Crash Fix + Search Memory Loop Killer)
# =============================================================================
import pandas as pd
import re
import copy
import math
import time
import asyncio
import os
from typing import Dict, List, TypedDict, Optional, Any, Union

try:
    from llama_index.core import VectorStoreIndex, Document, Settings
    from llama_index.core.retrievers import VectorIndexRetriever, BaseRetriever
    from llama_index.core.schema import NodeWithScore
    from llama_index.core.node_parser import SentenceSplitter
    from llama_index.retrievers.bm25 import BM25Retriever
except ImportError:
    VectorStoreIndex = None; Document = None; VectorIndexRetriever = None; BaseRetriever = object
    NodeWithScore = object; BM25Retriever = None; Settings = None; SentenceSplitter = None
    print("⚠️ [State] LlamaIndex components missing.")

try:
    from deepcollector.utils.profiler import profiler
except ImportError:
    class DummyProfiler:
        def track(self, c):
            return lambda f: f
        def update_stats(self, *a, **k):
            pass
    profiler = DummyProfiler()

class CellData(TypedDict):
    value: str
    confidence: float
    telemetry_context: Optional[str]
    anchor_ref_id: Optional[str]

CatalogItem = Dict[str, CellData]

class RAGResult(TypedDict):
    cell_data: CellData
    potential_sources: List[str]

if VectorIndexRetriever and BM25Retriever:
    class HybridRetriever(BaseRetriever):
        def __init__(self, vr, br):
            self.vr = vr
            self.br = br
            super().__init__()

        def _retrieve(self, q, **kwargs):
            bn = self.br.retrieve(q, **kwargs)
            vn = self.vr.retrieve(q, **kwargs)
            seen = set()
            final = []
            for n in vn + bn:
                if n.node.node_id not in seen:
                    final.append(n)
                    seen.add(n.node.node_id)
            return final[:self.vr.similarity_top_k]

class CatalogState:
    def __init__(self, config: Any, context: str, project_id: str):
        self.config = config
        self.context = context
        self.project_id = project_id
        self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)
        self.catalog: List[CatalogItem] = []
        self.history: List[str] = []
        self.current_phase: str = "INITIALIZATION"
        self.iteration: int = 0
        self.confidence_history: List[Dict[str, Any]] = []

        self.discovered_citations: List[Dict[str, Any]] = []
        self.discovered_weblinks: List[Dict[str, Any]] = []
        self.discovered_composites: List[Dict[str, Any]] = []

        self.index: Optional[VectorStoreIndex] = None
        self.documents: List[Document] = []
        self.all_nodes: List[Any] = []
        self.indexed_urls: set = set()

        # 🧠 FIX: Search Memory prevents infinite RAG extraction loops
        self.past_searches: set = set()

        self.bm25_retriever: Optional[Any] = None
        self.BM25RetrieverClass = getattr(config, 'BM25Retriever', None)

        if VectorStoreIndex and getattr(config, 'LLAMA_INDEX_AVAILABLE', False):
            try:
                self.index = VectorStoreIndex.from_documents([])
            except Exception as e:
                if self.verbosity >= 2: print(f"⚠️ [State] LlamaIndex Init Failed: {e}")
                self.index = None

        self.CATALOG_SCHEMA = getattr(config, 'CATALOG_SCHEMA', {})
        self.MISSING_DATA_PLACEHOLDERS = getattr(config, 'MISSING_DATA_PLACEHOLDERS', set(["[missing]"]))
        self.CONFIDENCE_LOCK_THRESHOLD = getattr(config, 'CONFIDENCE_LOCK_THRESHOLD', 0.95)
        self.INDEXING_BATCH_SIZE = getattr(config, 'INDEXING_BATCH_SIZE', 10)
        self.EXTRACTED_FIELDS = getattr(config, 'EXTRACTED_FIELDS', [])
        self.GROUNDING_FIELDS = getattr(config, 'GROUNDING_FIELDS', [])
        self.MIN_ASSIGNMENT_CONFIDENCE_GATE = getattr(config, 'MIN_ASSIGNMENT_CONFIDENCE_GATE', 0.70)

        self.INVALID_NAMES = {
            "", "[missing]", "unknown", "n/a", "nan", "none",
            "dataset", "time series", "time series dataset", "various", "varies", "null"
        }

    def _is_valid_dataset_scope(self, name: str) -> bool:
        if not name: return False
        n = name.lower()
        if n.endswith(('.py', '.xlsx', '.csv', '.md', '.txt')): return False
        invalid_static = ['pascal voc', 'titanic', 'shopee', 'imagenet', 'cifar', 'mnist']
        if any(inv in n for inv in invalid_static): return False
        return True

    def add_history(self, message: str):
        self.history.append(f"[{self.current_phase} Iter {self.iteration}] {message}")

    @profiler.track("State: Add Data and Index")
    def add_data_and_index(self, data: List[Dict[str, str]]):
        if not self.index: return

        new_documents = []
        for item in data:
            url = str(item.get("url", "N/A"))[:150]
            title = str(item.get("title", "N/A"))[:150]
            m_type = str(item.get("type", "Web"))[:50]
            content = item.get("content", "")

            ref_id = item.get("ref_id", url)
            if not content or ref_id in self.indexed_urls: continue

            text = f"Title: {title}\nURL: {url}\nContext: {self.context}\nContent:\n{content}"
            if Document:
                doc = Document(
                    text=text,
                    metadata={"url": url, "title": title, "type": m_type},
                    id_=ref_id
                )
                new_documents.append(doc)
                self.indexed_urls.add(ref_id)

        if not new_documents: return
        self.documents.extend(new_documents)

        try:
            node_parser = Settings.node_parser if Settings and hasattr(Settings, 'node_parser') else SentenceSplitter(chunk_size=1024)
            new_nodes = node_parser.get_nodes_from_documents(new_documents)
            self.all_nodes.extend(new_nodes)
            self.index.insert_nodes(new_nodes)
            if self.verbosity >= 1: print(f"    ✅ [Vector Index] Inserted {len(new_documents)} docs ({len(new_nodes)} nodes).")
        except Exception as e:
            if self.verbosity >= 2: print(f"    ❌ [Vector Index Error] {e}")

        if self.BM25RetrieverClass and self.all_nodes:
            try:
                self.bm25_retriever = self.BM25RetrieverClass.from_defaults(nodes=self.all_nodes, similarity_top_k=10)
                if self.verbosity >= 1: print(f"    ✅ [BM25 Index] Rebuilt with {len(self.all_nodes)} nodes.")
            except Exception as e:
                if self.verbosity >= 2: print(f"    ❌ [BM25 Index Error] {e}")

    def get_retriever(self, similarity_top_k=8, mode="HYBRID") -> Any:
        if not self.index or not VectorIndexRetriever: return None

        actual_k = min(similarity_top_k, len(self.all_nodes))
        if actual_k <= 0: actual_k = 1

        vr = VectorIndexRetriever(index=self.index, similarity_top_k=actual_k)
        if mode == "VECTOR": return vr

        if self.bm25_retriever and mode in ["BM25", "HYBRID"]:
            self.bm25_retriever.similarity_top_k = actual_k
            if mode == "BM25": return self.bm25_retriever
            if 'HybridRetriever' in globals():
                return HybridRetriever(vr, self.bm25_retriever)
        return vr

    def _initialize_new_item(self, name):
        item = {k: {"value": "[missing]", "confidence": 0.0, "telemetry_context": None, "anchor_ref_id": None} for k in self.CATALOG_SCHEMA}
        item["Dataset Name"] = {"value": name, "confidence": 0.5, "telemetry_context": "Init", "anchor_ref_id": None}
        return item

    def find_item_by_name(self, name):
        return next((i for i in self.catalog if i["Dataset Name"]["value"] == name), None)

    def get_cell_data(self, dataset_name, field, item_override=None):
        item = item_override or self.find_item_by_name(dataset_name)
        return item.get(field, {"value": "[missing]", "confidence": 0.0}) if item else {"value": "[missing]", "confidence": 0.0}

    def update_cell_data(self, dataset_name, field, new_data, allow_new_datasets=False, item_override=None):
        if not self._is_valid_dataset_scope(dataset_name):
            return False

        item = item_override or self.find_item_by_name(dataset_name)
        if not item:
            if not allow_new_datasets: return False
            item = self._initialize_new_item(dataset_name); self.catalog.append(item)

        if field not in item: return False
        curr = item[field]
        curr_val = curr.get("value", "[missing]")
        is_curr_missing = curr_val in self.MISSING_DATA_PLACEHOLDERS

        try: new_conf = float(new_data.get("confidence", 0.0))
        except: new_conf = 0.0

        if curr["confidence"] >= self.CONFIDENCE_LOCK_THRESHOLD and not is_curr_missing:
             return False

        new_val = new_data.get("value", "[missing]")
        is_new_missing = new_val in self.MISSING_DATA_PLACEHOLDERS

        should_update = False
        if is_curr_missing and not is_new_missing: should_update = True
        elif not is_curr_missing and new_conf > curr["confidence"]: should_update = True

        if should_update:
            curr.update({k: v for k, v in new_data.items() if k in curr})
            return True
        return False

    def update_catalog_batch(self, updates: List[CatalogItem], allow_new_datasets: bool = True):
        count = 0
        for up in updates:
            name = up.get("Dataset Name", {}).get("value")
            if not name or not self._is_valid_dataset_scope(name):
                continue
            for field, data in up.items():
                if self.update_cell_data(name, field, data, allow_new_datasets): count += 1
        if self.verbosity >= 1: print(f"    📊 [Batch Update] Updated {count} fields.")

    def capture_confidence_metrics(self, stage_name: str = "Unknown Stage") -> Dict[str, float]:
        if not self.catalog: return {"avg_conf": 0.0, "completeness": 0.0, "count": 0}
        total_conf = 0.0; total_cells = 0; filled_cells = 0
        for item in self.catalog:
            for field in self.EXTRACTED_FIELDS:
                data = self.get_cell_data(None, field, item)
                val = data.get("value", "[missing]")
                total_conf += data.get("confidence", 0.0)
                total_cells += 1
                if val not in self.MISSING_DATA_PLACEHOLDERS: filled_cells += 1
        avg_conf = (total_conf / total_cells) if total_cells > 0 else 0.0
        completeness = (filled_cells / total_cells) if total_cells > 0 else 0.0
        stats = {"Stage": stage_name, "Iteration": self.iteration, "Avg Confidence": avg_conf, "Completeness": completeness, "Catalog Size": len(self.catalog)}
        self.confidence_history.append(stats)
        if self.verbosity >= 1: print(f"    📊 [Metrics: {stage_name}] Catalog Size: {len(self.catalog)} | Avg Conf: {avg_conf:.2f} | Completeness: {completeness:.1%}")
        return {"avg_conf": avg_conf, "completeness": completeness, "count": len(self.catalog)}

    def get_average_confidence(self, fields=None):
        if not self.catalog: return 0.0
        fields = fields or self.CATALOG_SCHEMA.keys()
        total = 0.0; count = 0
        for item in self.catalog:
            for f in fields: total += self.get_cell_data(None, f, item)["confidence"]; count += 1
        return total / count if count else 0.0

    def inject_structured_knowledge(self, data): pass

    def update_project_resources(self, citations: List[Dict[str, Any]], weblinks: List[Dict[str, Any]]):
        if self.verbosity >= 1: print(f"    🔄 [State] Updating project resources (Citations: {len(citations)}, WebLinks: {len(weblinks)})...")
        self.discovered_citations.extend(citations)
        self.discovered_weblinks.extend(weblinks)

    def get_effective_name(self, item_or_name: Union[Dict, str]) -> str:
        item = item_or_name if isinstance(item_or_name, dict) else self.find_item_by_name(item_or_name)
        if not item: return str(item_or_name)

        def is_valid(val):
            s = str(val).strip().lower()
            if not s or s in self.MISSING_DATA_PLACEHOLDERS or s in self.INVALID_NAMES or s == "unknown": return False
            if s.startswith("ds_") and len(s) < 10: return False
            return True

        variant = item.get("Variant Name", {}).get("value", "")
        if is_valid(variant): return str(variant)

        ds_name = item.get("Dataset Name", {}).get("value", "")
        if is_valid(ds_name): return str(ds_name)

        aliases = item.get("Aliases", {}).get("value", "")
        if aliases and isinstance(aliases, str):
            for alias in aliases.split(','):
                clean_alias = alias.strip()
                if is_valid(clean_alias):
                    return clean_alias

        canon = item.get("Canonical Name", {}).get("value", "")
        if is_valid(canon): return str(canon)

        return "Unknown Dataset"

print("✅ deepcollector/core/state.py written (Added Search Memory Set).")

Writing deepcollector/core/state.py


#### **Cell 14: `deepcollector/core/rag_engine.py` (The "Frozen Cellular RAG" Module)**

This module encapsulates the RAG logic, separated from the Agent orchestration.

In [ ]:
%%writefile deepcollector/core/rag_engine.py
# =============================================================================
# V183: RAG Engine (JSON Markdown Parse Fix Natively Baked In - Fully Uncompressed)
# =============================================================================
import json
import re
import asyncio
import time
import sys
import gc
import traceback
import pandas as pd
import math
import ast
from typing import List, Dict, Optional, Tuple, Any, TYPE_CHECKING
from datetime import datetime

try:
    from google.genai import types
except ImportError:
    types = None

try:
    from deepcollector.utils.profiler import profiler
except ImportError:
    class DummyProfiler:
        def track(self, c):
            return lambda f: f
        def update_stats(self, *a, **k):
            pass
    profiler = DummyProfiler()

try:
    from deepcollector.core.state import RAGResult, CatalogState, CellData
    try:
        from llama_index.core.schema import NodeWithScore
    except ImportError:
        from llama_index.core import NodeWithScore
except ImportError:
    CatalogState = object
    CellData = dict

if TYPE_CHECKING:
    from deepcollector.tools.research import ResearchTools
    from deepcollector.config.settings import AppConfig

# Safe markdown constants to prevent UI truncation issues
MD_JSON = chr(96) * 3 + "json\n"
MD_END = "\n" + chr(96) * 3


class RAGEngine:
    def __init__(self, config: 'AppConfig', tools: 'ResearchTools'):
        self.config = config
        self.tools = tools
        self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)

        self.CATALOG_SCHEMA = getattr(config, 'CATALOG_SCHEMA', {})
        self.PLAUSIBILITY_THRESHOLDS = getattr(config, 'PLAUSIBILITY_THRESHOLDS', {})
        self.MISSING_DATA_PLACEHOLDERS = getattr(config, 'MISSING_DATA_PLACEHOLDERS', set())
        self.CELLULAR_RAG_BATCH_SIZE = getattr(config, 'CELLULAR_RAG_BATCH_SIZE', 50)
        self.CELLULAR_RAG_THROTTLE_DELAY = getattr(config, 'CELLULAR_RAG_THROTTLE_DELAY', 0.5)
        self.PARALLEL_CONCURRENCY_LIMIT = getattr(config, 'PARALLEL_CONCURRENCY_LIMIT', 4)

        # Configurable RAG Limits pulled dynamically from settings
        self.RAG_DISCOVERY_TOP_K = getattr(config, 'RAG_DISCOVERY_TOP_K', 15)
        self.RAG_DISCOVERY_MAX_CHARS = getattr(config, 'RAG_DISCOVERY_MAX_CHARS', 45000)
        self.RAG_CELLULAR_TOP_K = getattr(config, 'RAG_CELLULAR_TOP_K', 10)
        self.RAG_CELLULAR_MAX_CHARS = getattr(config, 'RAG_CELLULAR_MAX_CHARS', 35000)
        self.RAG_CELLULAR_FALLBACK_CHARS = getattr(config, 'RAG_CELLULAR_FALLBACK_CHARS', 15000)

    @profiler.track("RAGEngine: Discover Datasets")
    def discover_datasets_from_index(self, state: CatalogState) -> int:
        if not state.index:
            if self.verbosity >= 2:
                print("    ⚠️ [Discovery] No active vector index found. Skipping discovery.")
            return 0

        is_local = getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]
        retriever = state.get_retriever(similarity_top_k=self.RAG_DISCOVERY_TOP_K, mode="HYBRID")

        if not retriever:
            return 0

        query = f"List ALL datasets, archives, benchmarks, and repositories directly related to: '{state.context}'."

        try:
            nodes = retriever.retrieve(query)
            if not nodes:
                return 0

            context = self._format_context(nodes)[:self.RAG_DISCOVERY_MAX_CHARS]

            if getattr(self.config, 'ENABLE_VARIANT_MAPPING', False):
                variant_instruction = "2. **VARIANT MAPPING (Parent-Child):** If you find a dataset collection, list specific variants as SEPARATE dataset entries.\n"
            else:
                variant_instruction = "2. **EXTRACT LEAF DATASETS:** Your primary goal is to find specific individual datasets.\n"

            json_example = (
                MD_JSON +
                '{ "discovered_datasets": [{"dataset_name": "Name", "type": "Real-World Dataset", "confidence": 0.95, "rationale": "Is a leaf dataset in..."}] }' +
                MD_END
            )

            prompt = (
                f"Analyze the target project: '{state.context}'.\n\n"
                "--- Instructions ---\n"
                "1. **RUTHLESS SCOPE RULE:** ONLY extract 'Time-Series' datasets that are the PRIMARY FOCUS of the target project. Absolutely IGNORE datasets that are mentioned as 'related work', or static image datasets (e.g. Pascal VOC, ImageNet), NLP corpora, tabular ML sets (e.g. Titanic), or python scripts.\n"
                f"{variant_instruction}"
                "3. **UNPACK COLLECTIONS:** If a collection/archive is mentioned, list the specific datasets contained within it.\n"
                "4. **CLASSIFY TYPE / ENTITY TAXONOMY:** Explicitly classify each entry strictly as one of: [Real-World Dataset | Synthetic Dataset | Collection | Provider | Synthetic Generator | Augmentation Tool | Evaluation Script].\n"
                "5. **Format:** You MUST respond ONLY with a valid JSON block. Do NOT add extra conversational text.\n"
                f"{json_example}\n"
                "6. **Default Confidence:** 0.95.\n\n"
                f"--- Context ---\n{context}\n"
            )

            model = self.tools.models.MODEL_SYNTHESIZER
            max_new = 1536 if is_local else 1536
            response = self.tools.generate_content_synthesizer(model, prompt, max_new_tokens=max_new)

            if response.text == "[missing]" or response.text == "{}" or response.text == "[]":
                return 0

            return self._parse_discovery_response(state, response.text)

        except Exception as e:
            if getattr(self.config, '_CUDA_OOM_ABORT', False):
                raise e
            if self.verbosity >= 1:
                print(f"    ❌ [Discovery Error] Failed to extract from index: {str(e)[:150]}")
            return 0

    def _parse_discovery_response(self, state: CatalogState, text: str) -> int:
        added = 0
        try:
            data = self.tools._extract_json_robustly(text)

            if isinstance(data, dict):
                datasets = data.get("discovered_datasets", [])
            elif isinstance(data, list):
                datasets = data
            else:
                datasets = []

            for d in datasets:
                if not isinstance(d, dict):
                    continue

                name = d.get("dataset_name", "").strip()
                if not name or len(name) < 2 or name.lower() in ["varies", "n/a", "unknown", "dataset", "time series"] or name.endswith(('.py', '.xlsx')):
                    continue

                raw_conf = float(d.get("confidence", 0.5))
                conf = max(raw_conf, 0.95)
                raw_type = str(d.get("type", "Real-World Dataset")).strip().title()

                if getattr(self.config, 'ENABLE_ENTITY_TAXONOMY', False):
                    invalid_types = ["Synthetic Generator", "Augmentation Tool", "Evaluation Script"]
                    if any(inv.lower() in raw_type.lower() for inv in invalid_types):
                        continue

                    if "Real" in raw_type or "Synthetic" in raw_type or "Dataset" in raw_type:
                        raw_type = "Dataset"
                    elif "Collection" in raw_type:
                        raw_type = "Collection"
                    elif "Provider" in raw_type:
                        raw_type = "Provider"
                    else:
                        raw_type = "Dataset"
                else:
                    if raw_type not in ["Dataset", "Collection", "Provider"]:
                        raw_type = "Dataset"

                existing = state.find_item_by_name(name)

                if not existing:
                    new_item = state._initialize_new_item(name)
                    new_item["Assignment Confidence"] = {
                        "value": str(conf),
                        "confidence": 1.0,
                        "telemetry_context": "Discovery RAG",
                        "anchor_ref_id": None
                    }
                    new_item["Assignment Rationale"] = {
                        "value": d.get("rationale", "Extracted from context"),
                        "confidence": 1.0,
                        "telemetry_context": "Discovery RAG",
                        "anchor_ref_id": None
                    }
                    new_item["Type"] = {
                        "value": raw_type,
                        "confidence": 0.95,
                        "telemetry_context": "Discovery RAG",
                        "anchor_ref_id": None
                    }
                    state.catalog.append(new_item)
                    added += 1
                else:
                    try:
                        curr_val = float(state.get_cell_data(name, "Assignment Confidence")['value'])
                    except ValueError:
                        curr_val = 0.0

                    if conf > curr_val:
                        state.update_cell_data(
                            name,
                            "Assignment Confidence",
                            {"value": str(conf), "confidence": 1.0}
                        )

                    curr_type = state.get_cell_data(name, "Type").get("value", "[missing]")
                    if curr_type == "[missing]":
                        state.update_cell_data(
                            name,
                            "Type",
                            {"value": raw_type, "confidence": 0.95}
                        )

        except Exception as e:
            if getattr(self.config, '_CUDA_OOM_ABORT', False):
                raise e
            pass

        return added

    @profiler.track("RAGEngine: Plan Discovery")
    def plan_discovery_search(self, state: CatalogState) -> List[Dict[str, str]]:
        if not state.index:
            return []

        known_datasets = [i["Dataset Name"]["value"] for i in state.catalog]
        known_str = ", ".join(known_datasets[:20])

        json_example_queries = MD_JSON + '["query 1", "query 2"]' + MD_END

        prompt = (
            f"You are the strategic planner for a data cataloging agent. Target Project Context: {state.context}\n"
            f"Currently Discovered Datasets: {known_str}\n\nInstructions:\n"
            "1. Analyze the context and identified datasets.\n"
            "2. Determine what is missing. Are there foundational datasets likely used but not yet found?\n"
            "3. **RUTHLESS SCOPE RULE:** Do NOT generate searches for 'related' datasets, later versions, or competitors.\n"
            "4. Generate 3 targeted Google Search queries to find these missing data sources.\n"
            "5. Format STRICTLY as a JSON list of strings ONLY inside a markdown block:\n"
            f"{json_example_queries}\n"
        )

        try:
            model = self.tools.models.MODEL_PLANNER
            is_local = getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]
            response = self.tools.generate_content_planner(model, prompt, max_new_tokens=256 if is_local else 512)

            if response.text in ["[]", "[missing]"]:
                return []

            data = self.tools._extract_json_robustly(response.text)

            if isinstance(data, list):
                return [{"type": "search", "query": q} for q in data if isinstance(q, str)]
            else:
                return []

        except Exception as e:
            if getattr(self.config, '_CUDA_OOM_ABORT', False):
                raise e
            pass
            return []

    @profiler.track("RAGEngine: Execute Cellular RAG")
    def execute_cellular_rag(self, state: CatalogState, target_fields: List[str], retrieval_mode="HYBRID") -> Tuple[int, int, int]:
        is_local = getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]

        if not self.tools or (not getattr(self.tools.models, 'CLIENT', None) and not is_local):
            return 0, 0, 0

        candidate_cells = self._identify_candidate_cells(state, target_fields)
        if not candidate_cells:
            return 0, 0, 0

        retriever = state.get_retriever(similarity_top_k=self.RAG_CELLULAR_TOP_K, mode=retrieval_mode)
        if not retriever:
            return 0, 0, 0

        loop = asyncio.get_event_loop()
        results = loop.run_until_complete(self._run_rag_batches(state, candidate_cells, retriever))
        fills, refinements, confirmed = self._process_rag_results(state, results)

        if self.verbosity >= 1:
            print(f"    📊 [Cellular RAG] Execution complete. Updates applied: {fills + refinements}")

        return fills, refinements, confirmed

    async def _run_rag_batches(self, state, candidate_cells, retriever):
        results = []
        total_cells = len(candidate_cells)
        is_local = getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]
        batch_size = 1 if is_local else self.CELLULAR_RAG_BATCH_SIZE

        for i in range(0, total_cells, batch_size):
            batch = candidate_cells[i:i+batch_size]
            start_time = time.time()

            if self.verbosity >= 1:
                print(f"    🔄 [Cellular RAG Batch] Processing batch {i//batch_size + 1} of {math.ceil(total_cells/batch_size)} ({len(batch)} cells)...")

            tasks = []
            valid_batch = []

            for cell_info in batch:
                dataset_name = cell_info["dataset_name"]
                field_name = cell_info["field_name"]

                query_template = self.CATALOG_SCHEMA.get(field_name, {}).get("query")
                if not query_template:
                    continue

                verified_url = self._get_cell_value(cell_info['item'], "Link to Data (Actual Source)")
                if verified_url == "[missing]":
                    verified_url = None

                task = self._extract_cell_data_rag(
                    dataset_name,
                    effective_name=state.get_effective_name(cell_info['item']),
                    field_name=field_name,
                    query_template=query_template,
                    verified_url=verified_url,
                    retriever=retriever
                )
                tasks.append(task)
                valid_batch.append(cell_info)

            batch_results = []
            if is_local:
                import torch
                for t in tasks:
                    try:
                        res = await t
                        batch_results.append(res)
                    except Exception as e:
                        batch_results.append(RuntimeError(str(e)))
                    gc.collect()
                    if torch is not None and torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        torch.cuda.ipc_collect()
            else:
                batch_results = await self._run_async_tasks(tasks)

            for cell_info, rag_result in zip(valid_batch, batch_results):
                if getattr(self.config, '_CUDA_OOM_ABORT', False):
                    raise RuntimeError("CUDA OOM Abort")

                if isinstance(rag_result, Exception):
                    continue

                if rag_result:
                    results.append({
                        "dataset_name": cell_info["dataset_name"],
                        "field_name": cell_info["field_name"],
                        "rag_result": rag_result
                    })

            duration = time.time() - start_time
            profiler.update_stats("LLM: Cellular RAG", duration, count=len(batch))

            if i + batch_size < total_cells:
                await asyncio.sleep(self.CELLULAR_RAG_THROTTLE_DELAY)

        return results

    async def _run_async_tasks(self, tasks):
        semaphore = asyncio.Semaphore(self.PARALLEL_CONCURRENCY_LIMIT)

        async def semaphore_wrapper(task):
            async with semaphore:
                try:
                    return await task
                except Exception as e:
                    return RuntimeError(str(e))

        wrapped_tasks = [semaphore_wrapper(task) for task in tasks]
        return await asyncio.gather(*wrapped_tasks, return_exceptions=True)

    async def _extract_cell_data_rag(self, dataset_name, effective_name, field_name, query_template, verified_url, retriever) -> Optional[RAGResult]:
        is_local = getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]
        base_query = query_template.format(name=effective_name)
        queries = [base_query]

        numeric_instruction = ""
        f_lower = field_name.lower()

        if "time points" in f_lower or "variables" in f_lower or "locations" in f_lower:
            numeric_instruction = " WARNING: The value MUST be ONLY the raw integer number."
        elif "frequency" in f_lower or "interval" in f_lower:
            numeric_instruction = " WARNING: The value MUST be a concise phrase."
        elif "domain" in f_lower:
            numeric_instruction = " WARNING: The value MUST be a concise category."

        if getattr(self.config, 'ENABLE_MULTI_QUERY_RAG', False):
            if "Variables" in field_name:
                queries.append(f"How many features, dimensions, or variables does the {effective_name} dataset have?")
            elif "Time Points" in field_name:
                queries.append(f"What is the total length, number of rows, or number of time points for the {effective_name} dataset?")
            elif "Frequency" in field_name or "interval" in field_name.lower():
                queries.append(f"What is the sampling rate, frequency, or time interval for the {effective_name} dataset?")
            elif "url" in field_name.lower() or "link" in field_name.lower():
                queries.append(f"What is the official website or download link for the {effective_name} dataset?")

        nodes = []
        seen_node_ids = set()

        try:
            for q in queries:
                if hasattr(retriever, 'aretrieve'):
                    q_nodes = await retriever.aretrieve(q)
                else:
                    q_nodes = retriever.retrieve(q)

                for n in q_nodes:
                    nid = getattr(n.node, 'node_id', getattr(n, 'node_id', None))
                    if nid not in seen_node_ids:
                        seen_node_ids.add(nid)
                        nodes.append(n)
        except Exception as e:
            if getattr(self.config, '_CUDA_OOM_ABORT', False):
                raise e
            del e

        if nodes and hasattr(nodes, 'score'):
            nodes.sort(key=lambda x: getattr(x, 'score', 0.0), reverse=True)

        nodes = nodes[:self.RAG_CELLULAR_TOP_K]

        context_snippets = []
        for node in nodes:
            meta = node.node.metadata if hasattr(node, 'node') else getattr(node, 'metadata', {})
            url = meta.get("url", "N/A")

            if verified_url and url == verified_url:
                prefix = "Verified Source"
            else:
                prefix = f"Source ({meta.get('type', 'Unknown')})"

            if hasattr(node, 'get_content'):
                content = node.get_content()
            else:
                content = str(node)

            context_snippets.append(f"--- [{prefix}: {meta.get('title', 'N/A')}] ---\n{content}")

        context = self._sanitize_context("\n\n".join(context_snippets))

        arbitration_instruction = ""
        if getattr(self.config, 'ENABLE_ARBITRATION_PROMPT', False):
            arbitration_instruction = "**DISCREPANCY ARBITRATION:** Review the context chunks carefully. PRIORITIZE repository file structures and technical appendices over high-level abstract mentions."

        kwargs = {}
        if types and not is_local:
            try:
                kwargs["config"] = types.GenerateContentConfig(response_mime_type="application/json")
            except Exception:
                pass

        for attempt in range(2):
            try:
                if attempt == 0:
                    max_len = self.RAG_CELLULAR_MAX_CHARS
                else:
                    max_len = self.RAG_CELLULAR_FALLBACK_CHARS

                curr_context = context[:max_len]
                if len(context) > max_len:
                    curr_context += "... [TRUNCATED]"

                citation_instruction = ""
                if "citation" in field_name.lower():
                    citation_instruction = "IMPORTANT: Return the FULL academic citation if available."

                json_example = '{"value": "Extracted Text", "confidence": 0.95, "rationale": "Found in context"}'
                json_instructions = f"\nFormat EXACTLY as raw JSON. Do NOT wrap in markdown blocks. Example:\n{json_example}\n"

                prompt = (
                    f"Query: {base_query}\nTarget Field: {field_name}\nContext:\n{curr_context}\n\n"
                    f"Instructions: Answer strictly based on context. {numeric_instruction} {citation_instruction} {arbitration_instruction} "
                    f"If not found, set value='[missing]'. {json_instructions}"
                )

                resp = await self.tools.generate_content_rag_async(prompt, max_new_tokens=256, **kwargs)

                if not resp or not resp.text:
                    raise ValueError("Empty response")

                if resp.text in ["[missing]", "{}", "[]"]:
                    if attempt == 1:
                        return None
                    continue

                val, conf, rat = self._parse_response(resp.text, True, field_name)

                is_plausible, reason = self._validate_plausibility(field_name, val)

                if not is_plausible:
                    val = "[implausible]"
                    conf = 0.2
                    rat = rat + f" [VETO: {reason}]"

                return Auditor(self.config).format_result(val, conf, rat, nodes)

            except Exception as e:
                if getattr(self.config, '_CUDA_OOM_ABORT', False):
                    raise e
                if attempt == 1:
                    return None

        return None


    def _process_rag_results(self, state, results) -> Tuple[int, int, int]:
        fills = 0
        refinements = 0
        confirmed = 0

        for res in results:
            d_name = res["dataset_name"]
            f_name = res["field_name"]
            new_data = res["rag_result"]["cell_data"]

            old_data = state.get_cell_data(d_name, f_name)
            old_val = old_data.get("value", "[missing]")
            old_conf = old_data.get("confidence", 0.0)

            new_val = new_data.get("value", "[missing]")
            new_conf = new_data.get("confidence", 0.0)

            is_fill = False
            if old_val in self.MISSING_DATA_PLACEHOLDERS and new_val not in self.MISSING_DATA_PLACEHOLDERS:
                is_fill = True

            is_same = False
            if old_val.lower().strip() == str(new_val).lower().strip():
                is_same = True

            if is_fill:
                if state.update_cell_data(d_name, f_name, new_data):
                    fills += 1
            elif is_same:
                 if new_conf > old_conf:
                     state.update_cell_data(d_name, f_name, new_data)
                 confirmed += 1
            elif not is_same and new_conf >= old_conf:
                 if state.update_cell_data(d_name, f_name, new_data):
                     refinements += 1

        return fills, refinements, confirmed


    def _parse_response(self, text, is_json, field_name=""):
        text = self._clean_json_text(text)
        val = "[missing]"
        conf = 0.0
        rat = "JSON Parse Error"

        if is_json:
            data = self.tools._extract_json_robustly(text)
            if isinstance(data, dict) and "value" in data:
                val = data.get('value', '[missing]')

                if isinstance(val, list):
                    val = ", ".join([str(v).strip() for v in val if v])
                elif isinstance(val, dict):
                    val = str(val)

                val = re.sub(r'[\r\n]+', ' ', str(val).strip())

                conf = float(data.get('confidence', 0.0))
                if conf > 10.0:
                    conf = conf / 100.0
                elif conf > 1.0:
                    conf = conf / 10.0

                conf = min(conf, 1.0)
                rat = data.get('rationale', '')
            else:
                val_match = re.search(r'"value"\s*:\s*"([^"]+)"', text, re.I)
                c_match = re.search(r'"confidence"\s*:\s*([\d.]+)', text, re.IGNORECASE)

                if val_match:
                    val = val_match.group(1).strip()
                if c_match:
                    try:
                        conf = min(float(c_match.group(1)), 1.0)
                    except:
                        pass

        val = str(val).strip()
        if val.startswith("['") and val.endswith("']"):
            val = val[2:-2]
        if val.startswith('["') and val.endswith('"]'):
            val = val[2:-2]

        f_lower = field_name.lower()
        if "url" in f_lower or "link" in f_lower:
            m = re.search(r'(https?://[^\s\'"\]\)\>\,]+)', val)
            if m:
                val = m.group(1)

        return val, conf, rat


    def _clean_json_text(self, text):
        tb = chr(96) * 3
        if text.strip().startswith(tb):
            try:
                text = text.strip()
                if text.startswith(tb + "json"):
                    text = text[7:]
                elif text.startswith(tb):
                    text = text[3:]

                if text.endswith(tb):
                    text = text[:-3]
                return text.strip()
            except Exception:
                return text
        return text


    def _sanitize_context(self, text):
        text = re.sub(r'[ \t]+', ' ', text)
        return re.sub(r'\n{3,}', '\n\n', text)


    def _get_cell_value(self, item, field):
        return item.get(field, {}).get("value", "[missing]")


    def _validate_plausibility(self, field, val) -> Tuple[bool, str]:
        if val == "[missing]":
            return True, ""

        thresholds = self.PLAUSIBILITY_THRESHOLDS.get(field)
        if not thresholds:
            return True, ""

        v_lower = str(val).lower()
        multiplier = 1

        if re.search(r'(?:\b|\d)(billion|b)(?:\b|$)', v_lower):
            multiplier = 1_000_000_000
        elif re.search(r'(?:\b|\d)(million|m)(?:\b|$)', v_lower):
            multiplier = 1_000_000
        elif re.search(r'(?:\b|\d)(thousand|k)(?:\b|$)', v_lower):
            multiplier = 1_000

        nums = re.findall(r'(\d+\.?\d*)', str(val).replace(',', ''))
        if not nums:
            return True, ""

        max_val = max([float(n) for n in nums]) * multiplier

        if "min" in thresholds and max_val < thresholds["min"]:
            return False, "Below Min"
        if "max" in thresholds and max_val > thresholds["max"]:
            return False, "Above Max"

        return True, ""


    def _identify_candidate_cells(self, state, fields):
        cands = []
        for item in state.catalog:
            name = item["Dataset Name"]["value"]
            for f in fields:
                if state.get_cell_data(name, f)["confidence"] < 0.95:
                    cands.append({"dataset_name": name, "field_name": f, "item": item})
        return cands


    def _format_context(self, nodes):
        formatted_nodes = []
        for n in nodes:
            url = getattr(n, 'metadata', {}).get('url','')
            if hasattr(n, 'get_content'):
                content = n.get_content()[:2000]
            else:
                content = str(n)[:2000]
            formatted_nodes.append(f"--- Source: {url} ---\n{content}")
        return "\n\n".join(formatted_nodes)


class Auditor:
    def __init__(self, config):
        self.config = config
        self.MISSING = getattr(config, 'MISSING_DATA_PLACEHOLDERS', set())

    def format_result(self, val, conf, rat, nodes) -> RAGResult:
        if val.lower() in self.MISSING:
            val = "[missing]"
            conf = 0.0

        srcs = [getattr(n, 'metadata', {}).get('url') for n in nodes if getattr(n, 'metadata', {}).get('url')]

        if nodes and isinstance(nodes, list) and hasattr(nodes[0], 'score'):
            score = nodes[0].score
        else:
            score = 0.0

        if nodes and isinstance(nodes, list):
            anchor_id = getattr(nodes[0], 'node_id', None)
        else:
            anchor_id = None

        data = {
            "value": val,
            "confidence": conf,
            "telemetry_context": f"Rationale: {rat}\nTop Score: {score:.2f}\nSources: {srcs[:3]}",
            "anchor_ref_id": anchor_id
        }

        if CellData != dict:
            data = CellData(**data)

        return RAGResult(cell_data=data, potential_sources=srcs)

print("✅ deepcollector/core/rag_engine.py written (100% Fully Expanded + Native JSON Prompt Fix).")

Writing deepcollector/core/rag_engine.py


#### **Cell 15: Orchestrator agent.py**

This module handles the workflow (Phases 0-4) and the final reconciliation/reporting.

In [ ]:
%%writefile deepcollector/core/agent.py
# =============================================================================
# V209: CatalogAgent (Strict CSV Folder Export & Clean Naming Convention)
# =============================================================================
import pandas as pd
import time
import json
import re
import io
import uuid
import requests
import asyncio
import difflib
import math
import os
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple
from tabulate import tabulate

try:
    import gspread
    from google.colab import auth
    from google.auth import default
except ImportError:
    gspread = None

try:
    from deepcollector.config.settings import AppConfig
    from deepcollector.core.state import CatalogState, CellData
    from deepcollector.tools.research import ResearchTools
    from deepcollector.tools.ddi import DataInspectionTools
    from deepcollector.kb.manager import KnowledgeBaseManager, SheetLock

    try:
        from deepcollector.kb.merger import ProjectMerger
    except ImportError:
        ProjectMerger = None

    from deepcollector.core.rag_engine import RAGEngine
    from deepcollector.harvesting.uci_harvester import UCIHarvester
    from deepcollector.utils.initialization import initialize_apis
    from deepcollector.kb.maintenance import DatasetDoctor
    from deepcollector.tools.deep_research_runner import DeepResearchRunner
    from deepcollector.utils.analytics import PerformanceAnalyzer
    import deepcollector.utils.profiler as profiler_module

except ImportError as e:
    print(f"❌ [Agent] Critical imports failed: {e}")
    AppConfig = object
    CatalogState = object
    ResearchTools = object
    RAGEngine = object
    PerformanceAnalyzer = object
    profiler_module = None
    ProjectMerger = None

class AdvancedKnowledgeProcessor:
    def __init__(self, doc_url: str, tools: Any, target_project_name: str = None):
        self.doc_url = doc_url
        self.tools = tools
        self.target_project_name = target_project_name
        self.html_content = None
        self.references_map = {}
        self.processed_entries = []
        try:
            import bs4
            self.parser = 'lxml' if 'lxml' in globals() else 'html.parser'
            self.BeautifulSoup = bs4.BeautifulSoup
        except ImportError:
            self.BeautifulSoup = None

    def process(self):
        try:
            if not self.BeautifulSoup:
                return []

            if not self._fetch_content():
                return []

            self._extract_references()
            self._extract_and_augment_entries()
            return self.processed_entries
        except Exception as e:
            print(f"      ⚠️ [Knowledge Injection Error] Failed to parse Master Registry: {e}. Skipping injection.")
            return []

    def _fetch_content(self):
        try:
            import requests
            resp = requests.get(self.doc_url, timeout=20)
            if resp.status_code == 200:
                self.html_content = resp.content
                return True
        except Exception:
            pass
        return False

    def _safe_get_text(self, element):
        if not element: return ""
        try:
            if hasattr(element, 'get_text'):
                return element.get_text(strip=True)
            elif hasattr(element, 'text'):
                return str(element.text).strip()
        except Exception:
            pass
        return str(element).strip()

    def _extract_references(self):
        soup = self.BeautifulSoup(self.html_content, self.parser)
        for elem in soup.find_all(['p', 'li']):
            text = self._safe_get_text(elem)
            match = re.match(r'^\[(\d+)\]\s*(.+)', text)
            if match:
                self.references_map[f"[{match.group(1)}]"] = match.group(2)

    def _extract_and_augment_entries(self):
        soup = self.BeautifulSoup(self.html_content, self.parser)
        tables = soup.find_all('table')
        if not tables:
            return

        try:
            main_table = max(tables, key=lambda t: len(t.find_all('tr')))
        except Exception:
            return

        for row in main_table.find_all('tr'):
            cells = row.find_all(['td', 'th'])
            if len(cells) >= 3:
                name = self._safe_get_text(cells[0])
                desc = self._safe_get_text(cells[2])

                if not name or "Name" in name:
                    continue

                if self.target_project_name:
                    n_clean = name.lower().strip()
                    t_clean = self.target_project_name.lower().strip()
                    if n_clean not in t_clean and t_clean not in n_clean:
                        continue

                aug_desc = desc
                citations = [f"{rid}: {rtext}" for rid, rtext in self.references_map.items() if rid in desc]
                if citations:
                    aug_desc += "\n\nReferences:\n" + "\n".join(citations)

                self.processed_entries.append({
                    "Name": name,
                    "Project summary with references": desc,
                    "Augmented Description": aug_desc
                })

class CatalogAgent:
    VERSION = "V209"

    def __init__(self, config: AppConfig, authenticated_gc: Any, keys: Any = None, models: Any = None):
        self.config = config
        self.authenticated_gc = authenticated_gc
        self.verbosity = getattr(config, 'VERBOSITY_LEVEL', 1)

        self.job_id = f"JOB_{uuid.uuid4().hex[:6].upper()}"
        self.stop_workflow_flag = False
        self.deep_research_failed = False

        self.state = CatalogState(
            config,
            getattr(config, 'PROJECT_CONTEXT', ''),
            getattr(config, 'CURRENT_PROJECT_ID', '')
        )

        if keys and models:
            self.keys = keys
            self.models = models
        else:
            if 'initialize_apis' in globals() and initialize_apis:
                self.keys, self.models = initialize_apis(config)
            else:
                self.keys, self.models = None, None

        self.kb_manager = KnowledgeBaseManager(config)
        self.tools = ResearchTools(config, self.keys, self.models)
        self.ddi_tools = DataInspectionTools(config)
        self.rag_engine = RAGEngine(config, self.tools)
        self.doctor = DatasetDoctor(self.kb_manager, self.tools, self.verbosity)

        if ProjectMerger:
            self.merger = ProjectMerger(self.kb_manager, verbosity=self.verbosity, tools=self.tools, models=self.models)
        else:
            self.merger = None

        self.deep_researcher = None
        is_local = getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]
        has_client = getattr(self.models, 'CLIENT', None) is not None

        if has_client or is_local:
            self.deep_researcher = DeepResearchRunner(
                client=getattr(self.models, 'CLIENT', None),
                config=self.config,
                verbosity=self.verbosity,
                tools=self.tools
            )

        self.analyzer = PerformanceAnalyzer()

        method = getattr(config, 'PROJECT_METHOD', 1)
        if method == 2:
            self.harvester = UCIHarvester(config, self.tools)
        else:
            self.harvester = None

        self.MAX_DISCOVERY_ITERATIONS = getattr(config, 'MAX_DISCOVERY_ITERATIONS', 3)
        self.MAX_GROUNDING_ITERATIONS = getattr(config, 'MAX_GROUNDING_ITERATIONS', 5)
        self.MAX_EXTRACTION_ITERATIONS = getattr(config, 'MAX_EXTRACTION_ITERATIONS', 15)
        self.MIN_ASSIGNMENT_CONFIDENCE_GATE = getattr(config, 'MIN_ASSIGNMENT_CONFIDENCE_GATE', 0.70)
        self.CONFIDENCE_THRESHOLD = getattr(config, 'CONFIDENCE_THRESHOLD', 0.80)
        self.GROUNDING_FIELDS = getattr(config, 'GROUNDING_FIELDS', [])
        self.EXTRACTED_FIELDS = getattr(config, 'EXTRACTED_FIELDS', [])
        self.DATA_INSPECTION_ENABLED = getattr(config, 'DATA_INSPECTION_ENABLED', True)

    @staticmethod
    def _track(category):
        if profiler_module and hasattr(profiler_module, 'profiler'):
            return profiler_module.profiler.track(category)
        return lambda f: f

    def execute_workflow(self, mode="AGENT", job_comment="", merge_dry_run=False):
        @self._track("Agent: Workflow Wall-Clock Time")
        def _exec():
            start_time = datetime.now()
            print(f"\n🚀 Starting Job {self.job_id} ({mode}) for: '{self.state.context}'\n" + "="*60)
            if job_comment:
                print(f"📝 Job Comment: {job_comment}")

            job_status = "FAILED"
            try:
                if mode == "MERGE":
                    self.kb_manager.initialize_connection(self.authenticated_gc)
                    if self.merger:
                        self.merger.execute_merge(
                            self.config.CURRENT_PROJECT_ID,
                            self.job_id,
                            dry_run=merge_dry_run,
                            models_verifier=self.models
                        )
                    else:
                        print("❌ Merger module not available.")
                    job_status = "COMPLETED"
                    self._print_llm_summary()
                    return

                elif mode == "REPAIR":
                    self.execute_repair_workflow()

                elif mode == "HARVEST":
                    self.analyzer.start_phase("Bootstrapping", len(self.state.catalog))
                    if getattr(self.config, 'GSPREAD_AVAILABLE', False):
                        self.kb_manager.initialize_connection(self.authenticated_gc)
                        if getattr(self.config, 'WIPE_CURRENT_PROJECT_ONLY', False):
                            self.kb_manager.wipe_project(self.config.CURRENT_PROJECT_ID, self.job_id)
                    self.analyzer.end_phase(len(self.state.catalog))
                    self.execute_harvester_workflow()

                else:
                    self.analyzer.start_phase("Bootstrapping", len(self.state.catalog))
                    success = self.phase_0_bootstrapping()
                    self.analyzer.end_phase(len(self.state.catalog))

                    if not success:
                        print("\n🛑 Bootstrapping failed (Sources inaccessible). Terminating to prevent poor quality results.")
                        job_status = "ABORTED (Source Fetch Failed)"
                        self.stop_workflow_flag = True
                        return
                    else:
                        self.execute_standard_workflow()

                if getattr(self.config, '_CUDA_OOM_ABORT', False):
                    print("\n🛑 Execution stopped gracefully due to insufficient local GPU Memory (CUDA OOM).")
                    job_status = "ABORTED (CUDA OOM)"
                    self.stop_workflow_flag = True
                elif self.stop_workflow_flag:
                    print("\n🛑 Execution stopped to prevent inferior answers (Deep Research or Source Fetch Failed).")
                    if job_status == "FAILED":
                        job_status = "ABORTED (Validation Failed)"

                if not getattr(self.config, '_CUDA_OOM_ABORT', False):
                    if mode != "REPAIR":
                        self.analyzer.start_phase("Phase 3.5: Maintenance", len(self.state.catalog))
                        if self.state.catalog:
                            pre_count = len(self.state.catalog)
                            self.state.catalog = self.doctor.execute_maintenance(self.state.catalog)
                            post_count = len(self.state.catalog)
                            self.analyzer.record_deletions(pre_count - post_count)
                        self.analyzer.end_phase(len(self.state.catalog))

                    if mode == "REPAIR":
                        self.phase_4_repair_write_back()
                    else:
                        self.phase_4_write_back()

                    if getattr(self.config, 'EXPORT_TO_NEW_SHEET', True):
                        self._export_run_data()

                    self.analyzer.print_report()
                    self._print_llm_summary()

                    df_final_report = self.get_catalog_report(full_details=False)
                    if not df_final_report.empty:
                        print("\n" + "="*90)
                        print("📊 FINAL DISCOVERED DATASETS CATALOG")
                        print("="*90)
                        print(tabulate(df_final_report, headers="keys", tablefmt="pipe", showindex=False))

                    job_status = "COMPLETED"

            except Exception as e:
                if getattr(self.config, '_CUDA_OOM_ABORT', False):
                    print(f"\n🛑 [Workflow Interrupted] Graceful halt due to CUDA Out of Memory.")
                    job_status = "ABORTED (CUDA OOM)"
                else:
                    print(f"\n❌ [Workflow Crash] {e}")
                    import traceback
                    print(traceback.format_exc())
                    job_status = f"ERROR: {str(e)[:40]}"
            finally:
                duration = (datetime.now() - start_time).total_seconds()

                if not self.kb_manager.is_connected and getattr(self.config, 'GSPREAD_AVAILABLE', False):
                    try:
                        self.kb_manager.initialize_connection(self.authenticated_gc)
                    except:
                        pass

                if self.kb_manager.is_connected:
                    job_data = {
                        "JobID": self.job_id,
                        "ProjectID": self.config.CURRENT_PROJECT_ID,
                        "Mode": mode,
                        "Start_Time": start_time.strftime("%Y-%m-%d %H:%M:%S"),
                        "End_Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        "Duration_Sec": f"{duration:.2f}",
                        "Status": job_status,
                        "Items_Found": len(self.state.catalog),
                        "Operational_Parameters": self.config.get_operational_report(),
                        "JOB_COMMENT": job_comment
                    }
                    self.kb_manager.log_job_execution(job_data)

                if mode != "MERGE":
                    print(f"\n⏱️ Workflow Wall-Clock Time: {duration:.2f}s")

        return _exec()

    def _format_stats(self, stats):
        count = stats.get('count', 0)
        t = stats.get('time', 0.0)
        t_sq = stats.get('time_sq', 0.0)

        if count > 1:
            mean = t / count
            variance = max(0, (t_sq / count) - (mean ** 2))
            std = math.sqrt(variance)
            return f"{count:>4} calls | {t:>6.2f}s cum. CPU time | Mean: {mean:>5.2f}s | SD: {std:>5.2f}s"
        elif count == 1:
            return f"{count:>4} calls | {t:>6.2f}s cum. CPU time | Mean: {t:>5.2f}s | SD:   N/A "
        return "   0 calls"

    def _print_llm_summary(self):
        print("\n🤖 UNIFIED MODEL USAGE & TIMING STATS (Includes Parallel Cumulative CPU Time):")

        total_gemini_calls = 0
        total_gemma_calls = 0

        print("\n  [RAG & Search Operations]")
        if hasattr(self.tools, 'model_usage_stats') and self.tools.model_usage_stats:
            for model_name, stats in sorted(self.tools.model_usage_stats.items()):
                if stats['count'] > 0:
                    print(f"   {model_name:<35}: {self._format_stats(stats)}")
                    if "gemini" in model_name.lower():
                        total_gemini_calls += stats['count']
                    elif "gemma" in model_name.lower():
                        total_gemma_calls += stats['count']
        else:
            print("   No RAG/Search calls made.")

        print("\n  [Oracle Arbitrations (Merge/Review)]")
        if self.merger and hasattr(self.merger, 'oracle'):
            if hasattr(self.merger.oracle, 'model_stats') and self.merger.oracle.model_stats:
                for model_name, stats in sorted(self.merger.oracle.model_stats.items()):
                    if stats['count'] > 0:
                        print(f"   {model_name:<35}: {self._format_stats(stats)}")
                        if "gemini" in model_name.lower():
                            total_gemini_calls += stats['count']
                        elif "gemma" in model_name.lower():
                            total_gemma_calls += stats['count']
            else:
                print("   No Oracle LLM calls made.")

            print(f"   -------------------------------------------------------------")
            print(f"   Oracle Merges Approved:     {self.merger.oracle.stats['approved']}")
            print(f"   Oracle Merges Rejected:     {self.merger.oracle.stats['rejected']}")
            print(f"   Hard Discriminator Blocks:  {self.merger.oracle.stats['hard_blocks']}")
            print(f"   Rate Limits Hit (Cascaded): {self.merger.oracle.stats['rate_limits_hit']}")
            if self.merger.oracle.stats['errors'] > 0:
                print(f"   🛑 Fatal API Errors:        {self.merger.oracle.stats['errors']}")

        print("\n  [Aggregated LLM Call Totals]")
        print(f"   Total Gemini (Cloud) Calls: {total_gemini_calls}")
        print(f"   Total Gemma (Local) Calls:  {total_gemma_calls}")

    def _checkpoint_save(self, phase_name):
        try:
            df = self.get_catalog_report(full_details=False)
            if not df.empty:
                filename = f"checkpoint_{self.config.CURRENT_PROJECT_ID}_{self.job_id}.csv"
                df.to_csv(filename, index=False)
                if self.verbosity >= 2:
                    print(f"    💾 Auto-Checkpoint saved to {filename} ({phase_name})")
        except Exception:
            pass

    def execute_harvester_workflow(self):
        print(f"\n🚀 HARVESTER WORKFLOW")
        if self.harvester:
            try:
                success = self.harvester.execute_harvest(self.state)
                if not success:
                    print("🛑 [Agent] Harvester failed. Aborting workflow to prevent empty write-backs.")
                    self.stop_workflow_flag = True
            except Exception as e:
                print(f"❌ [Harvester Error] Caught critical failure: {e}")
                self.stop_workflow_flag = True

    def execute_repair_workflow(self):
        print(f"\n🛠️ REPAIR WORKFLOW (Fixing [missing] & low confidence values)")
        self.state.current_phase = "REPAIR"

        self.analyzer.start_phase("Repair: KB Hydration", len(self.state.catalog))
        self.kb_manager.initialize_connection(self.authenticated_gc)
        if not self.kb_manager.read_and_validate_kb():
            print("    ❌ Failed to load KB for repair.")
            return

        self._load_catalog_from_kb(self.config.CURRENT_PROJECT_ID)
        print(f"    📥 Loaded {len(self.state.catalog)} datasets from KB for targeted repair.")
        self.analyzer.end_phase(len(self.state.catalog))

        if not self.state.catalog:
            print("    ✅ No data found requiring repair based on specified thresholds.")
            return

        pre_stats = self.state.capture_confidence_metrics("Pre-Repair")

        urls = getattr(self.config, 'INITIAL_URLS', [])
        if urls:
            loaded_any = False
            for url in urls:
                fetched_data = self.tools.tool_load_url(url)
                if fetched_data:
                    self.state.add_data_and_index(fetched_data)
                    loaded_any = True

            if not loaded_any:
                print("    🛑 [CRITICAL] Failed to load ANY of the provided INITIAL_URLS. Aborting repair to prevent hallucination.")
                self.stop_workflow_flag = True
                return
        else:
             print("    ⚠️ No INITIAL_URLS found. The agent will attempt to search based on dataset names.")

        self.analyzer.start_phase("Phase 2: Grounding (Repair)", len(self.state.catalog))
        self.phase_2_grounding()
        self.analyzer.end_phase(len(self.state.catalog))

        if getattr(self.config, '_CUDA_OOM_ABORT', False):
            return

        self.analyzer.start_phase("Phase 3: Extraction (Repair)", len(self.state.catalog))
        self.phase_3_extraction()
        self.analyzer.end_phase(len(self.state.catalog))

        post_stats = self.state.capture_confidence_metrics("Post-Repair")
        self._print_repair_report(pre_stats, post_stats)

    def _print_repair_report(self, pre, post):
        print("\n" + "="*50)
        print("📊 REPAIR IMPACT REPORT")
        print("="*50)
        print(f"{'Metric':<20} | {'Before':<10} | {'After':<10} | {'Delta':<10}")
        print("-" * 60)

        c_pre = pre.get('completeness', 0.0) * 100
        c_post = post.get('completeness', 0.0) * 100
        print(f"{'Completeness':<20} | {c_pre:.1f}%      | {c_post:.1f}%      | {c_post-c_pre:+.1f}%")

        a_pre = pre.get('avg_conf', 0.0)
        a_post = post.get('avg_conf', 0.0)
        print(f"{'Avg Confidence':<20} | {a_pre:.2f}        | {a_post:.2f}        | {a_post-a_pre:+.2f}")
        print("-" * 60)

    def _load_catalog_from_kb(self, project_id=None):
        df_ds = self.kb_manager.get_kb_data("Datasets")
        if df_ds.empty:
            return

        if project_id and project_id not in ["PROJ_UNKNOWN", "PROJ_GLOBAL_MAINTENANCE", "GLOBAL"]:
            df_links = self.kb_manager.get_kb_data("Project_Dataset_Link")
            if not df_links.empty and "DatasetID" in df_links.columns:
                linked_ds = df_links[df_links["ProjectID"] == project_id]["DatasetID"].tolist()
                df_ds = df_ds[df_ds["DatasetID"].isin(linked_ds)]

        schema_mapping = {
            "Time interval between points": "Frequency",
            "Number of Time Points": "Num Time Points",
            "Number of Locations/Series": "Num Locations/Series",
            "Variables per Location": "Variables per Location",
            "Total Variables": "Total Variables",
            "Primary URL": "Primary URL",
            "Link to Data (Actual Source)": "Link to Data (Actual Source)",
            "Other URL": "Other URL",
            "Detailed Description": "Description",
            "Primary Source Repository": "Primary Creator"
        }

        missing_set = self.kb_manager.MISSING_DATA_PLACEHOLDERS

        for _, row in df_ds.iterrows():
            name = row.get("Variant Name", "")
            if not name or name == "[missing]":
                name = row.get("Canonical Name", "Unknown")

            needs_repair = False
            for c_f, k_f in schema_mapping.items():
                val_str = str(row.get(k_f, "")).strip().lower()

                if val_str in missing_set or val_str in ["nan", "none", ""]:
                    needs_repair = True
                    break

                conf_col = f"{k_f} (C)"
                try:
                    conf = float(str(row.get(conf_col, "0.0")))
                except:
                    conf = 0.0

                threshold = 0.90 if c_f in self.GROUNDING_FIELDS else self.CONFIDENCE_THRESHOLD
                if conf < threshold:
                    needs_repair = True
                    break

            if not needs_repair:
                continue

            item = self.state._initialize_new_item(name)
            for col in self.state.CATALOG_SCHEMA.keys():
                kb_col = schema_mapping.get(col, col)

                val_raw = str(row.get(kb_col, "[missing]")).strip()
                if val_raw.lower() in ["nan", "none", ""]:
                    val_raw = "[missing]"

                conf_col = f"{kb_col} (C)"
                try:
                    conf = float(str(row.get(conf_col, "0.0")))
                except:
                    conf = 0.0

                if val_raw.lower() in missing_set:
                    conf = 0.0
                elif conf == 0.0:
                    conf = 1.0

                item[col] = {"value": val_raw, "confidence": conf, "telemetry_context": "Loaded for Repair", "anchor_ref_id": None}

            item["Dataset Name"] = {"value": name, "confidence": 1.0, "telemetry_context": "KB", "anchor_ref_id": None}
            item["Canonical Name"] = {"value": str(row.get("Canonical Name", name)), "confidence": 1.0, "telemetry_context": "KB", "anchor_ref_id": None}
            item["Type"] = {"value": str(row.get("Type", "Dataset")), "confidence": 1.0, "telemetry_context": "KB", "anchor_ref_id": None}
            item["Assignment Confidence"] = {"value": str(row.get("Overall Confidence", "1.0")), "confidence": 1.0, "telemetry_context": "KB", "anchor_ref_id": None}
            item["DatasetID"] = {"value": str(row.get("DatasetID", "")), "confidence": 1.0, "telemetry_context": "KB", "anchor_ref_id": None}
            item["Aliases"] = {"value": str(row.get("Aliases", "")), "confidence": 1.0, "telemetry_context": "KB", "anchor_ref_id": None}

            self.state.catalog.append(item)

    def _apply_golden_kb_fastpath(self):
        print(f"\n{'='*20} PHASE 1.5: GOLDEN KB FAST-PATH {'='*20}")
        if not getattr(self.config, 'ENABLE_GOLDEN_FASTPATH', True):
            print("    ⏭️ Golden Fast-Path is disabled in config.")
            return

        df_ds = self.kb_manager.get_kb_data("Datasets")
        if df_ds is None or df_ds.empty:
            print("    ℹ️ KB is empty. Skipping fast-path.")
            return

        schema_mapping = {
            "Domain": "Domain",
            "Time interval between points": "Frequency",
            "Number of Time Points": "Num Time Points",
            "Number of Locations/Series": "Num Locations/Series",
            "Variables per Location": "Variables per Location",
            "Total Variables": "Total Variables",
            "Primary URL": "Primary URL",
            "Link to Data (Actual Source)": "Link to Data (Actual Source)",
            "Other URL": "Other URL",
            "Primary Source Repository": "Primary Creator",
            "Detailed Description": "Description",
            "Comments on Data Preparation": "Comments on Data Preparation"
        }

        def norm(n):
            return re.sub(r'[^a-z0-9]', '', str(n).lower())

        kb_name_map = {}
        for idx, row in df_ds.iterrows():
            try:
                conf = float(row.get("Overall Confidence", 0.0))
            except:
                conf = 0.0

            if conf >= 0.85:
                v_name = norm(row.get("Variant Name", ""))
                c_name = norm(row.get("Canonical Name", ""))
                aliases = [norm(x) for x in str(row.get("Aliases", "")).split(",") if x.strip()]

                keys = [v_name, c_name] + aliases
                for k in keys:
                    if k and (k not in kb_name_map or conf > kb_name_map[k]['conf']):
                        row_copy = row.copy()
                        row_copy['_num_conf'] = conf
                        kb_name_map[k] = {'row': row_copy, 'conf': conf}

        fast_path_count = 0
        skipped_fields_total = 0
        for item in self.state.catalog:
            name = item.get("Dataset Name", {}).get("value", "")
            if not name or name == "[missing]":
                continue

            clean_target = norm(name)
            match = kb_name_map.get(clean_target)

            if match:
                row = match['row']
                hydrated = False

                for cat_field in list(self.state.CATALOG_SCHEMA.keys()):
                    if cat_field in ["Dataset Name", "Assignment Confidence", "Assignment Rationale", "Type", "Canonical Name", "Aliases"]:
                        continue

                    kb_col = schema_mapping.get(cat_field, cat_field)
                    val = str(row.get(kb_col, "[missing]")).strip()

                    if val and val.lower() not in ["nan", "none", ""] and val not in self.kb_manager.MISSING_DATA_PLACEHOLDERS:
                        self.state.update_cell_data(name, cat_field, {"value": val, "confidence": 1.0, "telemetry_context": "KB Golden Fast-Path"})
                        hydrated = True
                        skipped_fields_total += 1

                for id_field in ["Canonical Name", "Type"]:
                    curr_val = item.get(id_field, {}).get("value", "[missing]")
                    if curr_val in self.kb_manager.MISSING_DATA_PLACEHOLDERS:
                        kb_val = str(row.get(id_field, "")).strip()
                        if kb_val and kb_val.lower() != "nan" and kb_val not in self.kb_manager.MISSING_DATA_PLACEHOLDERS:
                            self.state.update_cell_data(name, id_field, {"value": kb_val, "confidence": 1.0, "telemetry_context": "KB Golden Fast-Path"})

                if hydrated:
                    fast_path_count += 1
                    if self.verbosity >= 2:
                        print(f"    🌟 Fast-Path Matched: '{name[:30]}' -> KB Entity '{row.get('Variant Name')[:30]}' (Conf: {match['conf']:.2f})")

        print(f"    ⚡ Fast-Patched {fast_path_count} datasets! Pre-filled {skipped_fields_total} fields to skip RAG.")
        self.analyzer.record_cell_change('FILL', skipped_fields_total)

    def execute_standard_workflow(self):
        print(f"\n🚀 STANDARD RAG WORKFLOW (Method 1)")
        self.analyzer.start_phase("Phase 1a: Deep Research", len(self.state.catalog))
        self.phase_1_deep_research()
        self.analyzer.end_phase(len(self.state.catalog))

        if self.stop_workflow_flag:
            print("    🛑 Workflow halted securely at Phase 1a.")
            return

        self.analyzer.start_phase("Phase 1b: Standard Discovery", len(self.state.catalog))
        self.phase_1_standard_loop()
        self.analyzer.end_phase(len(self.state.catalog))

        if self.stop_workflow_flag:
            return

        self.analyzer.start_phase("Phase 1.5: Golden KB Fast-Path", len(self.state.catalog))
        self._apply_golden_kb_fastpath()
        self.analyzer.end_phase(len(self.state.catalog))

        self.analyzer.start_phase("Phase 2: Grounding", len(self.state.catalog))
        self.phase_2_grounding()
        self.analyzer.end_phase(len(self.state.catalog))

        if getattr(self.config, '_CUDA_OOM_ABORT', False):
            return

        self.analyzer.start_phase("Phase 3: Extraction", len(self.state.catalog))
        self.phase_3_extraction()
        self.analyzer.end_phase(len(self.state.catalog))

    def _run_preflight_crawler(self, text_content: str):
        if not text_content or not isinstance(text_content, str):
            return

        print(f"    🕷️ [Pre-Flight Crawler] Scanning context for outbound repositories...")
        found_urls = re.findall(r'(https?://(?:github\.com|huggingface\.co|zenodo\.org|kaggle\.com|archive\.ics\.uci\.edu)[^\s,\">|\)\]]+|https?://[^\s,\">|\)\]]+\.pdf)', text_content)
        unique_urls = list(set([u.rstrip('.,;:') for u in found_urls]))

        if unique_urls:
            print(f"      🔗 Found {len(unique_urls)} secondary URLs.")
            for u in unique_urls[:5]:
                print(f"        🕸️ Fetching & Indexing: {u}")
                self.state.add_data_and_index(self.tools.tool_load_url(u))

    def phase_0_bootstrapping(self) -> bool:
        self.state.current_phase = "BOOTSTRAPPING"
        print("\n=== PHASE 0: BOOTSTRAPPING ===")

        if getattr(self.config, 'GSPREAD_AVAILABLE', False):
            self.kb_manager.initialize_connection(self.authenticated_gc)
            if self.kb_manager.read_and_validate_kb():
                ds_data = self.kb_manager.get_kb_data("Datasets")
                if ds_data is not None and not ds_data.empty:
                    self.state.add_data_and_index(ds_data.to_dict('records'))

        urls = getattr(self.config, 'INITIAL_URLS', [])
        if urls:
            print(f"🌐 Loading {len(urls)} initial URLs...")
            loaded_any = False
            for url in urls:
                fetched_data = self.tools.tool_load_url(url)
                if fetched_data and isinstance(fetched_data, list) and len(fetched_data) > 0:
                    loaded_any = True
                    self.state.add_data_and_index(fetched_data)
                    if getattr(self.config, 'ENABLE_PREFLIGHT_CRAWLER', False):
                        for f_item in fetched_data:
                            if isinstance(f_item, dict):
                                content = f_item.get("content", "")
                                if content:
                                    self._run_preflight_crawler(content)

            if not loaded_any:
                print("    🛑 [CRITICAL] Could not access ANY of the provided initial URLs.")
                return False

        return True

    def phase_1_deep_research(self):
        if not getattr(self.config, 'ENABLE_DEEP_RESEARCH', False):
            return

        print("\n--- Phase 1a: Deep Research (Agentic) ---")
        citation_text = self.state.context

        if hasattr(self.config, 'INITIAL_URLS') and self.config.INITIAL_URLS:
            citation_text += f" (Source URL: {self.config.INITIAL_URLS})"
        else:
            citation_text += " (No specific URL provided, rely on context)"

        print(f"    ℹ️ Deep Research Context: {citation_text[:100]}...")

        required_columns = ["Dataset Name", "Entity Type", "Domain", "Number of Variables", "Number of Time Points", "Time interval", "Primary Source", "Primary Home Page URL", "Link to Data (Actual Source)", "Other URLs", "Detailed Description", "Comments"]

        prompt = (f"Act as a Data Archivist. You are tasked with mapping the SPECIFIC project/collection: '{self.state.context}'.\n"
                  f"There is a collection of Time series datasets described in the following context/citation:\n{citation_text}\n\n"
                  f"YOUR TASK: Produce a Dataset Catalog table describing ONLY the datasets that are officially part of or evaluated in THIS specific project.\n"
                  f"REQUIRED COLUMNS:\n- {', '.join(required_columns)}\n\n"
                  "INSTRUCTIONS:\n"
                  "1. Each dataset should be one row.\n"
                  f"2. RUTHLESS SCOPE RULE: ONLY extract datasets that are the PRIMARY FOCUS of '{self.state.context}'. Absolutely IGNORE datasets that are mentioned as 'related work', 'other competitions', or navigation links (e.g., if target is M1, ignore M3, M4, M5 entirely).\n"
                  "3. Entity Type MUST be exactly one of: 'Dataset', 'Collection', or 'Provider'.\n"
                  "4. Datasets should be specific and never have generic names like Dataset, Dataset name, Time Series.\n"
                  "5. **CONTENT RULE:** Do NOT use generic labels, for any table cells. Use specific names.\n"
                  "6. Be exhaustive for the target project ONLY.\n"
                  "7. VERY IMPORTANT URL RULES: 'Primary Home Page URL' must be the dataset's official site. 'Link to Data (Actual Source)' MUST be the direct data repository (HuggingFace, Zenodo, Kaggle, GitHub) or raw file endpoints (.zip, .csv). 'Other URLs' should hold academic papers (e.g. arXiv). DO NOT use Google Scholar or Vertex AI search tracking links for any of these.\n"
                  "8. IMPORTANT: Use '|||' as a clean separator between columns.\n"
                  "9. **CRITICAL ANTI-LOOP WARNING:** Limit your deep exploration. If you find yourself repeating the same search query or repeating the same chain of thought, IMMEDIATELY HALT EXPLORATION. Break out of the research phase and synthesize your report based strictly on the data you have found up to that point. Do not get stuck in infinite loops."
                  )

        if self.deep_researcher:
            dr_items = self.deep_researcher.execute_research(prompt)
            if not dr_items:
                if getattr(self.config, 'ABORT_ON_DEEP_RESEARCH_FAILURE', True):
                    print("    🛑 [STRICT ABORT] Deep Research failed, timed out, or returned 0 results.")
                    print("    🛑 User Policy Enforcement: Aborting workflow to prevent inferior answers.")
                    self.stop_workflow_flag = True
                else:
                    print("    ⚠️ [FALLBACK ACTIVATED] Deep Research failed or timed out.")
                    print("    ⚠️ User Policy Enforcement: Switching to Rigorous Standard Discovery Mode.")
                    self.deep_research_failed = True
                return

            idx_docs = [i for i in dr_items if isinstance(i, dict) and i.get("is_index_doc")]
            catalog_items = [i for i in dr_items if isinstance(i, dict) and not i.get("is_index_doc")]

            if idx_docs:
                self.state.add_data_and_index(idx_docs)
                print(f"    ✅ Indexed Deep Research Report.")

            if catalog_items:
                print(f"    📥 Bootstrapping {len(catalog_items)} datasets from Deep Research:")
                for item in catalog_items:
                    type_str = item.get("Type", {}).get("value", "Unknown")
                    print(f"      + {item.get('Dataset Name', {}).get('value', 'Unknown')} [{type_str}]")
                self.state.update_catalog_batch(catalog_items, allow_new_datasets=True)
                self.analyzer.record_cell_change('FILL', len(catalog_items) * 8)

    def phase_1_standard_loop(self):
        print(f"\n--- Phase 1b: Standard Discovery Loop ---")

        if getattr(self, 'deep_research_failed', False):
            self.MAX_DISCOVERY_ITERATIONS = max(self.MAX_DISCOVERY_ITERATIONS, 4)
            self.config.SEARCH_NUM_RESULTS = max(getattr(self.config, 'SEARCH_NUM_RESULTS', 8), 12)
            print(f"\n🔥 [RIGOROUS MODE] Increasing Discovery Iterations to {self.MAX_DISCOVERY_ITERATIONS} and Search Limit to {self.config.SEARCH_NUM_RESULTS}")

        self.state.iteration = 0

        # Access KI URL dynamically from config
        ki_url = getattr(self.config, 'KNOWLEDGE_MASTER_DOC_URL', None)
        current_proj_name = getattr(self.config, 'CURRENT_PROJECT_NAME', None)

        if ki_url and current_proj_name:
            print(f"\n--- Discovery Stage 0: Knowledge Injection (Master Registry) ---")
            processor = AdvancedKnowledgeProcessor(ki_url, self.tools, target_project_name=current_proj_name)
            ki_data = processor.process()
            if ki_data and isinstance(ki_data, list) and len(ki_data) > 0:
                idx_data = [{"url": "KI_DOC_MASTER", "content": d['Augmented Description'], "title": d['Name'], "type": "Knowledge Injection"} for d in ki_data if isinstance(d, dict)]
                self.state.add_data_and_index(idx_data)
                print(f"    ✅ Found and Indexed match: {ki_data[0].get('Name', 'Unknown')}")
        else:
             if self.verbosity >= 1:
                 print("    ℹ️  Skipping Knowledge Injection (Missing Master URL or Project Name).")

        while self.state.iteration < self.MAX_DISCOVERY_ITERATIONS:
            self.state.iteration += 1
            print(f"\n--- Discovery Iteration {self.state.iteration}/{self.MAX_DISCOVERY_ITERATIONS} ---")

            added = self.rag_engine.discover_datasets_from_index(self.state)
            self.analyzer.record_cell_change('FILL', added * 2)

            search_plan = self.rag_engine.plan_discovery_search(self.state)
            actions_taken = False
            if search_plan:
                print(f"🧠 [Planner] Generating search strategy...")
                for action in search_plan:
                    if not isinstance(action, dict): continue
                    q = action.get("query")
                    if q:
                        print(f"🛠️ Executing Search: {q}")
                        limit = getattr(self.config, 'SEARCH_NUM_RESULTS', 8)
                        results = self.tools.tool_search_and_fetch(q, num_results=limit)
                        if results:
                            self.state.add_data_and_index(results)
                            actions_taken = True

            if not actions_taken and self.state.iteration > 1:
                print("💡 No new actions. Stopping Discovery.")
                break

        self._validate_discovered_entities(self.state)
        self._apply_relevance_gate()
        self.state.capture_confidence_metrics("End of Phase 1")

    def _validate_discovered_entities(self, state):
        print(f"\n--- Entity Validation (Collection Awareness & Naming) ---")

        leaf_regex = [
            r"\bett[hm]\d\b", r"\bpems[-_ ]?(sf|bay|0)\b", r"san francisco traffic",
            r"traffic \(pems\)", r"\becl\b", r"electricity consuming load",
            r"exchange rate", r"illness", r"weather \(mpi", r"taxi", r"solar"
        ]

        exact_collections = {"pems", "ett", "monash", "utsd", "tslib", "m3", "m4", "m5", "gefcom", "lotsa", "timebench", "chronos", "gluonts", "timeseriesclassification"}
        collection_keywords = ["archive", "repository", "library", "collection", "benchmark", "corpus", "unified time series", "benchmark suite", "competition", "dataset suite"]
        provider_keywords = ["ecmwf", "bank", "reserve", "authority", "commission", "physionet", "kaggle", "caltrans", "nrel", "noaa", "google", "microsoft", "uci"]

        current_project_name = getattr(self.config, 'CURRENT_PROJECT_NAME', "").lower().strip()
        context_str = state.context.lower()
        subject_match = re.match(r"^([^:-]+)", context_str)
        context_subject = subject_match.group(1).strip() if subject_match else ""

        self_ref_terms = set()
        if current_project_name:
            self_ref_terms.add(current_project_name)
        if context_subject:
            self_ref_terms.add(context_subject)

        self_ref_count = 0

        for item in state.catalog:
            name_obj = item.get("Dataset Name", {})
            name = name_obj.get("value", "") if isinstance(name_obj, dict) else str(name_obj)
            name_lower = name.lower().strip()

            curr_type = state.get_cell_data(name, "Type").get("value", "Dataset")

            is_collection = False
            is_provider = False
            is_leaf = False

            if any(re.search(p, name_lower) for p in leaf_regex):
                is_leaf = True
            elif name_lower in exact_collections:
                is_collection = True
            else:
                if any(k in name_lower for k in collection_keywords):
                    is_collection = True
                elif any(k in name_lower for k in provider_keywords):
                    is_provider = True

            if "pems" in name_lower and not is_collection:
                is_leaf = True
            if "ett" in name_lower and not is_collection:
                is_leaf = True
            if re.search(r"\bm\b", name_lower) and not is_collection:
                if len(name_lower) > 4:
                    is_leaf = True

            if is_leaf:
                if curr_type != "Dataset":
                    state.update_cell_data(name, "Type", {"value": "Dataset", "confidence": 0.99})
            elif is_collection:
                if curr_type != "Collection":
                    state.update_cell_data(name, "Type", {"value": "Collection", "confidence": 0.95})
            elif is_provider:
                if curr_type != "Provider":
                    state.update_cell_data(name, "Type", {"value": "Provider", "confidence": 0.95})

            is_self_ref = False
            if name_lower in self_ref_terms:
                is_self_ref = True
            else:
                for term in self_ref_terms:
                    if name_lower == f"{term} dataset" or (len(term) > 3 and difflib.SequenceMatcher(None, name_lower, term).ratio() > 0.9):
                        is_self_ref = True
                        break

            if is_self_ref:
                state.update_cell_data(name, "Assignment Confidence", {"value": "0.0", "confidence": 1.0})
                self_ref_count += 1
                continue

        print(f"    📊 Classification complete. Validated Types and Deprecated {self_ref_count} items.")

    def _apply_relevance_gate(self):
        print(f"\n--- Relevance Gate (Threshold: {self.MIN_ASSIGNMENT_CONFIDENCE_GATE}) ---")
        kept = []
        for item in self.state.catalog:
            try:
                val = float(self.state.get_cell_data(None, "Assignment Confidence", item).get("value", 0))
            except:
                val = 0.0

            if val >= self.MIN_ASSIGNMENT_CONFIDENCE_GATE:
                kept.append(item)

        self.state.catalog = kept
        print(f"    📉 Gate applied. Retained {len(kept)} items.")

    def phase_2_grounding(self):
        self.state.current_phase = "GROUNDING"
        self.state.iteration = 0
        print(f"\n{'='*20} PHASE 2: GROUNDING {'='*20}")
        current_gaps = self._audit_catalog(threshold=0.90, specific_fields=self.GROUNDING_FIELDS)

        while self.state.iteration < self.MAX_GROUNDING_ITERATIONS:
            if not current_gaps:
                break

            self.state.iteration += 1
            print(f"\n--- Grounding Iteration {self.state.iteration} ---")

            fills, refs, confs = self.rag_engine.execute_cellular_rag(self.state, self.GROUNDING_FIELDS)
            self.analyzer.record_cell_change('FILL', fills)
            self.analyzer.record_cell_change('REFINE', refs)
            self.analyzer.record_cell_change('CONFIRM', confs)

            if self.DATA_INSPECTION_ENABLED:
                self._execute_ddi_sweep(current_gaps)

            current_gaps = self._audit_catalog(threshold=0.90, specific_fields=self.GROUNDING_FIELDS)

            if current_gaps:
                self._plan_and_execute_extraction_search(current_gaps)

            self._checkpoint_save(f"Grounding_Iter_{self.state.iteration}")

    def phase_3_extraction(self):
        self.state.current_phase = "EXTRACTION"
        self.state.iteration = 0
        print(f"\n{'='*20} PHASE 3: EXTRACTION {'='*20}")

        fields = [f for f in self.EXTRACTED_FIELDS if f not in self.GROUNDING_FIELDS]
        self._lock_collections_fields(fields)
        current_gaps = self._audit_catalog(threshold=self.CONFIDENCE_THRESHOLD, specific_fields=fields, skip_collections=True)

        stall_counter = 0
        prev_gap_count = len(current_gaps)
        STALL_THRESHOLD = 3

        while self.state.iteration < self.MAX_EXTRACTION_ITERATIONS:
            if not current_gaps:
                break

            self.state.iteration += 1
            print(f"\n--- Extraction Iteration {self.state.iteration} ---")

            fills, refs, confs = self.rag_engine.execute_cellular_rag(self.state, fields)
            self.analyzer.record_cell_change('FILL', fills)
            self.analyzer.record_cell_change('REFINE', refs)
            self.analyzer.record_cell_change('CONFIRM', confs)

            if current_gaps:
                self._plan_and_execute_extraction_search(current_gaps)

            self._checkpoint_save(f"Extraction_Iter_{self.state.iteration}")

            current_gaps = self._audit_catalog(threshold=self.CONFIDENCE_THRESHOLD, specific_fields=fields, skip_collections=True)
            gap_reduction = prev_gap_count - len(current_gaps)

            if gap_reduction < 2:
                stall_counter += 1
            else:
                stall_counter = 0

            prev_gap_count = len(current_gaps)

            if stall_counter >= STALL_THRESHOLD:
                print("🛑 [Early Termination] Extraction stalled. Stopping.")
                break

    def _lock_collections_fields(self, fields):
        dim_fields = {
            "Number of Time Points", "Number of Locations/Series",
            "Variables per Location", "Total Variables", "Time interval between points"
        }
        fields_to_lock = [f for f in fields if f in dim_fields]

        for item in self.state.catalog:
            if item.get("Type", {}).get("value") in ["Collection", "Provider"]:
                for f in fields_to_lock:
                    if self.state.get_cell_data(None, f, item).get("value") == "[missing]":
                        self.state.update_cell_data(item["Dataset Name"]["value"], f, {"value": "[Skipped]", "confidence": 1.0})

    def phase_4_write_back(self):
        print(f"\n{'='*20} PHASE 4: WRITE-BACK {'='*20}")
        if not self.kb_manager.is_connected:
            print("⚠️ [Write-back] KB Manager not connected.")
            return

        try:
            self.kb_manager.reconcile_and_write(
                self.state,
                getattr(self.config, 'CURRENT_PROJECT_NAME', 'Unknown Project'),
                job_id=self.job_id
            )
        except Exception as e:
            print(f"❌ [Write-back Error] {e}")

    def phase_4_repair_write_back(self):
        print(f"\n{'='*20} PHASE 4: REPAIR WRITE-BACK {'='*20}")
        if not self.kb_manager.is_connected:
            print("⚠️ [Write-back] KB Manager not connected.")
            return

        from deepcollector.kb.manager import SheetLock
        lock = SheetLock(self.kb_manager, self.job_id, self.verbosity)

        if not lock.acquire(timeout_seconds=300):
            print("    ❌ Failed to acquire KB Lock for repair write-back.")
            return

        try:
            if not self.kb_manager.read_and_validate_kb():
                raise RuntimeError("Failed to refresh KB.")

            df_ds = self.kb_manager.get_kb_data("Datasets")
            time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            updates = 0
            for item in self.state.catalog:
                name = item.get("Dataset Name", {}).get("value")
                if not name or name == "[missing]":
                    continue

                record, field_confs, _ = self.kb_manager._normalize_dataset_entry(self.state, item, time_str)

                mask = (df_ds["Variant Name"].astype(str).str.strip().str.lower() == str(name).strip().lower()) | \
                       (df_ds["Canonical Name"].astype(str).str.strip().str.lower() == str(name).strip().lower())

                if mask.any():
                    idx = mask.idxmax()
                    row_updated = False

                    for field_col, conf_col in self.kb_manager.FIELD_CONFIDENCE_MAP.items():
                        new_val = record.get(field_col, "")
                        new_conf = field_confs.get(field_col, 0.0)

                        try:
                            old_conf = float(df_ds.loc[idx, conf_col])
                        except:
                            old_conf = 0.0

                        old_val_str = str(df_ds.loc[idx, field_col]).strip().lower()
                        is_missing = old_val_str in self.kb_manager.MISSING_DATA_PLACEHOLDERS or old_val_str in ["nan", "none", ""]

                        if (new_conf > old_conf or is_missing) and new_val not in self.kb_manager.MISSING_DATA_PLACEHOLDERS:
                            df_ds.loc[idx, field_col] = new_val
                            df_ds.loc[idx, conf_col] = str(new_conf)
                            row_updated = True

                    if len(str(record.get("Description", ""))) > len(str(df_ds.loc[idx, "Description"])) + 10:
                        df_ds.loc[idx, "Description"] = record["Description"]
                        row_updated = True

                    if row_updated:
                        df_ds.loc[idx, "Job_Updated"] = self.job_id
                        df_ds.loc[idx, "Date_Updated"] = time_str
                        df_ds.loc[idx, "Project_Updated"] = self.config.CURRENT_PROJECT_ID
                        updates += 1

            if updates > 0:
                self.kb_manager.write_dataframe_to_tab("Datasets", df_ds)
                print(f"    ✅ Repair Write-back complete. Updated {updates} rows in 'Datasets' tab.")
            else:
                print("    ℹ️ No updates required in KB.")

        except Exception as e:
            print(f"❌ [Write-back Error] {e}")
        finally:
            lock.release()

    # =========================================================================
    # CRITICAL EXPORT OVERRIDE: Strict CSV Export
    # Replaces standalone Google Sheet creation with a direct Google Drive
    # API upload to the specified target folder dynamically via AppConfig.
    # =========================================================================
    def _export_run_data(self):
        print(f"\n{'='*20} RUN EXPORT {'='*20}")
        try:
            df = self.get_catalog_report(full_details=True)
            if df.empty:
                print("    ℹ️ No data to export.")
                return

            # Access the globally configured folder ID safely (This is for the DATA)
            folder_id = getattr(self.config, 'GOOGLE_DRIVE_SHEET_FOLDER_ID', None)
            if not folder_id:
                print("    ⚠️ Export Folder ID not configured in settings. Skipping Drive upload.")
                return

            import numpy as np
            # Python 3.11 Safe Infinity & NaN stripping
            df = df.replace([np.inf, -np.inf], np.nan).fillna("")
            df = df.astype(str)
            df = df.replace(r'(?i)^(nan|inf|-inf|none|<na>)$', '', regex=True)

            # -------------------------------------------------------------
            # Enforce Naming Convention: ProjectName_YYYYMMDD_HHMM_JobID.csv
            # -------------------------------------------------------------
            proj_name = str(getattr(self.config, 'CURRENT_PROJECT_NAME', 'UNKNOWN'))
            safe_proj_name = re.sub(r'[^A-Za-z0-9_\-]', '_', proj_name).strip('_')
            timestamp = time.strftime('%Y%m%d_%H%M')
            filename = f"{safe_proj_name}_{timestamp}_{self.job_id}.csv"

            # Save Locally
            df.to_csv(filename, index=False)
            print(f"🎉 [Export] Data saved to local CSV: {filename}")

            # Upload to Google Drive natively
            if hasattr(self, 'authenticated_gc') and self.authenticated_gc:
                try:
                    from googleapiclient.discovery import build
                    from google.auth import default
                    from googleapiclient.http import MediaFileUpload

                    print(f"    ☁️ Uploading {filename} to target Google Drive Folder...")

                    creds, _ = default()
                    drive_service = build('drive', 'v3', credentials=creds)

                    file_metadata = {
                        'name': filename,
                        'parents': [folder_id]
                    }

                    # Force text/csv to prevent Google Drive from converting to a Sheet
                    media = MediaFileUpload(filename, mimetype='text/csv')
                    file = drive_service.files().create(
                        body=file_metadata,
                        media_body=media,
                        fields='id'
                    ).execute()

                    print(f"    ✅ Success! CSV cleanly uploaded to Drive with ID: {file.get('id')}")
                except ImportError:
                    print("    ⚠️ googleapiclient not installed. Skipping Drive upload.")
                except Exception as e:
                    print(f"    ❌ Failed to upload to Google Drive folder: {e}")

        except Exception as e:
            print(f"❌ [Export Error] Failed: {e}")
            import traceback
            print(traceback.format_exc())

    def _audit_catalog(self, threshold, specific_fields, skip_collections=False):
        gaps = []
        fields = specific_fields or self.EXTRACTED_FIELDS
        for item in self.state.catalog:
            if skip_collections and item.get("Type", {}).get("value") in ["Collection", "Provider"]:
                continue
            try:
                if float(item.get("Assignment Confidence", {}).get("value", 0)) <= 0.0:
                    continue
            except:
                pass
            name = item["Dataset Name"]["value"]
            for f in fields:
                data = self.state.get_cell_data(name, f)
                if data["confidence"] < threshold:
                    gaps.append({"Dataset": name, "Field": f, "Confidence": data["confidence"]})
        return gaps

    def _execute_ddi_sweep(self, gaps):
        print("    🔍 Running DDI Sweep...")
        inspected = set()
        for gap in gaps:
            name = gap['Dataset']
            if name in inspected:
                continue
            url = self.state.get_cell_data(name, "Link to Data (Actual Source)").get("value")
            if url and (url.lower().endswith(('.csv', '.txt', '.zip', '.npz')) or "raw.githubusercontent" in url.lower()):
                res = self.tools.tool_inspect_data_file(url)
                if res.get("status") == "success":
                    self.state.update_cell_data(name, "Variables per Location", {"value": str(res.get("column_count")), "confidence": 1.0})
                    self.analyzer.record_cell_change('FILL', 1)
                    inspected.add(name)

    def _plan_and_execute_extraction_search(self, gaps):
        ds_gaps = {}
        for g in gaps:
            ds_gaps.setdefault(g['Dataset'], []).append(g['Field'])
        for ds, fields in list(ds_gaps.items())[:15]:
            clean_fields = [f.replace(" (C)", "") for f in fields]
            eff_name = self.state.get_effective_name(ds)
            field_str = str(clean_fields).lower()
            if "url" in field_str or "source" in field_str or "link" in field_str:
                query = f"official download url repository github dataset '{eff_name}'"
            else:
                query = f"{eff_name} dataset {', '.join(clean_fields)}"
            print(f"🛠️ Executing Search: {query}")
            res = self.tools.tool_search_and_fetch(query, num_results=getattr(self.config, 'SEARCH_NUM_RESULTS', 8))
            if res:
                self.state.add_data_and_index(res)

    def get_catalog_report(self, full_details=False):
        if not self.state.catalog:
            return pd.DataFrame()
        data = []
        for item in self.state.catalog:
            try:
                if float(item.get("Assignment Confidence", {}).get("value", 0)) <= 0.0:
                    continue
            except:
                pass
            row = {}
            high_conf = 0
            total = 0
            item_type = item.get("Type", {}).get("value", "[missing]")
            if item_type == "[missing]":
                row["Type"] = "Dataset"
            else:
                row["Type"] = item_type
            for k, v in item.items():
                if not k.startswith("_"):
                    val = v.get("value", "[missing]")
                    conf = v.get("confidence", 0.0)
                    if k == "Type" and val == "[missing]":
                        val = "Dataset"
                    row[k] = val
                    if conf >= 0.80:
                        high_conf += 1
                    total += 1
                    if full_details:
                        row[f"{k} (Telemetry)"] = v.get("telemetry_context", "")
            row["Completeness (High Conf %)"] = f"{(high_conf/total)*100:.1f}%" if total else "0%"
            data.append(row)
        return pd.DataFrame(data)

print("✅ deepcollector/core/agent.py written (Strict CSV Folder Export & Clean Naming Convention Fix).")

Writing deepcollector/core/agent.py


####Cell 16 Clean Up Dataset Doctor

In [ ]:
%%writefile deepcollector/kb/maintenance.py
# =============================================================================
# V166: Dataset Doctor (SyntaxWarning Fix & Aggressive Artifact Cleanser)
# =============================================================================
import pandas as pd
import time
import re
import json
from typing import List, Dict, Any

class DatasetDoctor:
    def __init__(self, kb_manager, research_tools, verbosity=1):
        self.kb_manager = kb_manager
        self.tools = research_tools
        self.verbosity = verbosity

        self.INVALID_NAMES = {
            "", "[missing]", "missing", "unknown", "n/a", "nan", "none",
            "dataset", "time series", "time series dataset", "various", "varies", "null"
        }

    def _get_effective_name(self, item):
        def clean(s): return str(s).strip()
        def is_valid(s): return s.lower() not in self.INVALID_NAMES and not s.startswith("DS_")

        variant = clean(item.get("Dataset Name", {}).get("value", ""))
        if is_valid(variant): return variant

        variant_alt = clean(item.get("Variant Name", {}).get("value", ""))
        if is_valid(variant_alt): return variant_alt

        aliases = clean(item.get("Aliases", {}).get("value", ""))
        if aliases:
            for a in aliases.split(','):
                if is_valid(a.strip()): return a.strip()

        canon = clean(item.get("Canonical Name", {}).get("value", ""))
        if is_valid(canon): return canon

        return variant

    def execute_maintenance(self, catalog: List[Dict]) -> List[Dict]:
        print("\n=== PHASE 3.5: MAINTENANCE & REPAIR (V166) ===")
        if not catalog: return []

        catalog = self._clean_names_regex(catalog)
        catalog = self._clean_urls(catalog)
        catalog = self._exact_deduplicate(catalog)
        catalog = self._repair_zombies(catalog)
        catalog = self._sanitize_uci_links(catalog)

        return catalog

    def _clean_names_regex(self, catalog: List[Dict]) -> List[Dict]:
        print("    🧹 [Doctor] Standardizing names...")

        patterns = [
            r'^thuml/', r'^google/', r'^microsoft/', r'^facebook/', r'^ucr/',
            r'\s*\((?:.*\bUTSD\b.*)\)', r'\s*\((?:.*\bTimer\b.*)\)',
            r'\s*\(Component\)', r'\s*\(Subset\)', r'\s*\(Collection\)',
            r'[-_ ](small|medium|large|tiny|full|subset|complete|12g|1g|4g)(?=[-_ ]|$)'
        ]

        count = 0
        for item in catalog:
            name_obj = item.get("Dataset Name", {})
            old_name = name_obj.get("value", "")
            new_name = old_name

            for p in patterns:
                new_name = re.sub(p, '', new_name, flags=re.IGNORECASE).strip()

            new_name = new_name.strip("-_ ")

            if new_name != old_name and len(new_name) > 2:
                item["Dataset Name"]["value"] = new_name
                item["Variant Name"] = {"value": old_name, "confidence": 1.0}
                count += 1

        if count > 0 and self.verbosity >= 1:
            print(f"    ✨ Standardized {count} names.")

        return catalog

    def _clean_urls(self, catalog: List[Dict]) -> List[Dict]:
        print("    🧹 [Doctor] Cleaning URL formatting and purging banned domains...")
        url_cols = ["Link to Data (Actual Source)", "Primary URL", "Other URL", "URL"]
        banned_domains = ['vertexaisearch.cloud.google.com', 'grounding-api-redirect', 'scholar.google']
        cleaned_count = 0
        purged_count = 0

        for item in catalog:
            for col in url_cols:
                if col in item:
                    val = str(item[col].get("value", "")).strip()
                    if not val or val in self.INVALID_NAMES: continue

                    if any(banned in val.lower() for banned in banned_domains):
                        item[col] = {"value": "[missing]", "confidence": 0.0}
                        purged_count += 1
                        continue

                    old_val = val

                    # 1. Strip exact outer match brackets and quotes
                    val = re.sub(r"^\[\]\'\"]+|\[\]\'\"]+$", "", val)

                    # 2. Re-format if it gave a comma-separated list of links inside a string
                    val = val.replace("', '", ", ").replace('", "', ', ')

                    # 3. Pull strictly the URL out if it got wrapped in markdown after all
                    match = re.search(r'\[.*?\]\((https?://.*?)\)', val)
                    if match:
                        val = match.group(1)
                    elif val.startswith("http") and " " in val:
                        match = re.search(r'(https?://[^\s]+)', val)
                        if match:
                            val = match.group(1)

                    # 4. Final safety strip of trailing punctuation
                    # CRITICAL FIX: Removed the backslash completely. rstrip takes a set of characters, so "]", ">" etc are perfectly valid
                    val = val.rstrip(".,;:'\"]>)")

                    if val != old_val:
                        item[col]["value"] = val
                        cleaned_count += 1

        if cleaned_count > 0 or purged_count > 0:
            if self.verbosity >= 1:
                print(f"    ✨ Cleaned {cleaned_count} URLs, Purged {purged_count} banned links.")

        return catalog

    def _exact_deduplicate(self, catalog: List[Dict]) -> List[Dict]:
        print(f"    🧠 [Doctor] Running STRICT Exact Deduplication on {len(catalog)} items...")
        seen = {}
        cleaned = []
        for item in catalog:
            name = self._get_effective_name(item).lower().strip()
            if not name or name in self.INVALID_NAMES: continue

            if name in seen: self._merge_data(seen[name], item)
            else:
                seen[name] = item
                cleaned.append(item)

        dropped = len(catalog) - len(cleaned)
        if dropped > 0: print(f"    📉 Deduplication finished. Safely merged {dropped} exact duplicates.")
        else: print("    ✅ No exact duplicates found.")
        return cleaned

    def _merge_data(self, target: Dict, source: Dict):
        for field, source_cell in source.items():
            if field.startswith("_"): continue
            target_cell = target.get(field)
            source_val = source_cell.get("value", "")
            target_val = target_cell.get("value", "") if target_cell else "[missing]"

            if target_val in ["[missing]", "", None] and source_val not in ["[missing]", "", None]:
                target[field] = source_cell.copy()
            if isinstance(target_cell, dict) and isinstance(source_cell, dict):
                if source_cell.get("confidence", 0) > target_cell.get("confidence", 0):
                    target[field] = source_cell.copy()

    def _sanitize_uci_links(self, catalog):
        print("    🧹 [Sanitizer] Checking for broken UCI zip links...")
        fixed_count = 0
        for item in catalog:
            url_obj = item.get("Link to Data (Actual Source)", {})
            url = url_obj.get("value", "")
            if "archive.ics.uci.edu" in url and url.endswith(".zip"):
                name = self._get_effective_name(item)
                try:
                    query = f'site:archive.ics.uci.edu/dataset/ "{name}"'
                    results = self.tools.tool_search_and_fetch(query, num_results=1)
                    if results and "/dataset/" in results.get('url', ''):
                        new_url = results['url']
                        item["Link to Data (Actual Source)"] = {"value": new_url, "confidence": 0.98}
                        fixed_count += 1
                        if self.verbosity >= 1: print(f"      ✨ Fixed Link: {new_url}")
                except Exception: pass

        print(f"    🧹 Sanitization complete. Fixed {fixed_count} bad links.")
        return catalog

    def _repair_zombies(self, catalog):
        print(f"    🩺 [Doctor] Scanning {len(catalog)} items for missing or redirect URLs...")
        fixed_count = 0
        url_cols = ["Link to Data (Actual Source)", "Primary URL", "URL"]
        invalid_data_domains = [
            'arxiv.org', 'doi.org', 'researchgate.net', 'ieeexplore.ieee.org',
            'sciencedirect.com', 'springer.com', 'nature.com', 'acm.org', 'semanticscholar.org',
            'vertexaisearch.cloud.google.com', 'grounding-api-redirect'
        ]

        for item in catalog:
            name = self._get_effective_name(item)
            if not name or name.lower().strip() in self.INVALID_NAMES: continue
            if item.get("Type", {}).get("value") == "Provider": continue

            has_valid_url = False
            for col in url_cols:
                val = item.get(col, {}).get("value", "")
                if val and val not in ["[missing]", "n/a", ""] and "http" in val:
                    if not any(bad_dom in val.lower() for bad_dom in invalid_data_domains):
                        has_valid_url = True
                        break

            if not has_valid_url:
                if self.verbosity >= 1: print(f"    🧟 Zombie / Paper URL Found (Needs real URL): {name}")
                found_url = None
                try:
                    query = f"official download url repository github dataset '{name}'"
                    res = self.tools.tool_search_and_fetch(query, num_results=2)
                    if res:
                        for r in res:
                            candidate_url = r.get('url', '')
                            if candidate_url and not any(bad_dom in candidate_url.lower() for bad_dom in invalid_data_domains):
                                found_url = candidate_url
                                break
                except Exception: pass

                if found_url:
                    item["Link to Data (Actual Source)"] = {"value": found_url, "confidence": 0.95}
                    print(f"      ✨ Healed with Real URL: {found_url}")
                    fixed_count += 1
                else:
                    if self.verbosity >= 1: print("      ⚠️ Could not recover true data URL. Retaining original fallback link.")

        print(f"    🩺 Repair complete. Fixed {fixed_count} items.")
        return catalog

print("✅ deepcollector/kb/maintenance.py written (SyntaxWarning Fix).")

Writing deepcollector/kb/maintenance.py


#### **Cell 17 Deep research**

In [ ]:
%%writefile deepcollector/tools/deep_research_runner.py
# =============================================================================
# V23: Deep Research Runner (Configurable Agent Model & Timeout Protection)
# =============================================================================
import time
import json
import re
import warnings
import pandas as pd
import io
import csv
from typing import List, Dict, Any, Optional

try:
    from deepcollector.utils.profiler import profiler
except ImportError:
    class DummyProfiler:
        def track(self, c): return lambda f: f
    profiler = DummyProfiler()

class DeepResearchRunner:
    def __init__(self, client: Any, config: Any = None, verbosity: int = 1, tools: Any = None):
        self.client = client
        self.config = config
        self.verbosity = verbosity
        self.tools = tools

        # Pull model name from config, fallback to default if missing
        self.AGENT_NAME = getattr(self.config, 'DEEP_RESEARCH_AGENT_MODEL', "deep-research-pro-preview-12-2025") if self.config else "deep-research-pro-preview-12-2025"

        warnings.filterwarnings("ignore", category=UserWarning, module="google.genai.interactions")

    @profiler.track("Tool: Deep Research Agent")
    def execute_research(self, prompt: str) -> List[Dict[str, Any]]:
        is_local = getattr(self.config, 'LLM_BACKEND', '') in ["LOCAL_PRO", "LOCAL_CLASSROOM"]

        # --- LOCAL GEMMA DEEP RESEARCH ---
        if is_local and getattr(self.config, 'ENABLE_LOCAL_DEEP_RESEARCH', True) and self.tools:
            return self._execute_local_research(prompt)

        # --- GOOGLE CLOUD DEEP RESEARCH ---
        if not self.client:
            if self.verbosity >= 1: print("    ❌ [Deep Research] Client not initialized.")
            return []

        max_retries = getattr(self.config, 'DEEP_RESEARCH_MAX_RETRIES', 6) if self.config else 6
        force_async = getattr(self.config, 'FORCE_ASYNC_POLLING', False) if self.config else False
        poll_interval = getattr(self.config, 'DEEP_RESEARCH_POLLING_INTERVAL_SECONDS', 15) if self.config else 15

        timeout_mins = 60
        max_wait_time = 3600

        for attempt in range(1, max_retries + 1):
            print(f"\n🧠 [Deep Research] Submitting job to '{self.AGENT_NAME}' (Attempt {attempt}/{max_retries}): '{prompt[:60]}...'")

            task_id = None
            final_report_text = ""
            last_log_count = 0
            t0_deep = time.time()
            attempt_failed = False

            try:
                if force_async:
                    print("    ⚡ Forcing Async Polling (Bypassing fragile stream)...")
                    response = self.client.interactions.create(
                        agent=self.AGENT_NAME,
                        input=prompt,
                        agent_config={
                            "type": "deep-research",
                            "thinking_summaries": "auto"
                        },
                        background=True,
                        stream=False
                    )
                    task_id = response.id
                    print(f"    🚀 Task ID: {task_id} | Status: BACKGROUND INITIATED")
                else:
                    response_stream = self.client.interactions.create(
                        agent=self.AGENT_NAME,
                        input=prompt,
                        agent_config={
                            "type": "deep-research",
                            "thinking_summaries": "auto"
                        },
                        background=True,
                        stream=True
                    )

                    for chunk in response_stream:
                        if hasattr(chunk, 'interaction') and chunk.interaction:
                            if not task_id:
                                task_id = chunk.interaction.id
                                print(f"    🚀 Task ID: {task_id} | Status: STREAMING")

                        if hasattr(chunk, 'logs') and chunk.logs:
                            for log in chunk.logs:
                                last_log_count += 1
                                if self.verbosity >= 1:
                                    print(f"    🔎 {log.message}")

                        if hasattr(chunk, 'delta') and chunk.delta:
                            delta = chunk.delta
                            dtype = getattr(delta, 'type', '')
                            content = getattr(delta, 'content', '')
                            text_val = getattr(content, 'text', str(content)) if content else getattr(delta, 'text', '')

                            if dtype == "thought_summary" and text_val:
                                if self.verbosity >= 1:
                                    clean_thought = text_val.replace('\n', ' ').strip()
                                    print(f"    🧠 Thinking: {clean_thought}")
                            elif dtype == "text" and text_val:
                                final_report_text += text_val

            except Exception as e:
                print(f"\n    ⚠️ Stream connection closed or interrupted ({e}). Switching to Async Monitor...")

            if not task_id:
                print(f"\n    ❌ [Deep Research] Failed to initialize task ID on Google's end.")
                attempt_failed = True
            else:
                print(f"    ⏳ Monitoring Task ID: {task_id} (Max wait: {timeout_mins}m)")

                print("    ", end="", flush=True)

                while True:
                    elapsed = time.time() - t0_deep
                    if elapsed > max_wait_time:
                        print(f"\n    🛑 [TIMEOUT] Job hung for over {timeout_mins} minutes.")
                        attempt_failed = True
                        break

                    try:
                        job = self.client.interactions.get(id=task_id)
                        status = str(job.status).upper()

                        print(".", end="", flush=True)

                        if hasattr(job, 'logs') and job.logs:
                            current_logs = job.logs
                            if len(current_logs) > last_log_count:
                                print()
                                for log in current_logs[last_log_count:]:
                                    if self.verbosity >= 1:
                                        print(f"    🔎 (Async) {log.message}")
                                last_log_count = len(current_logs)
                                print("    ", end="", flush=True)

                        if status == "COMPLETED":
                            print(f"\n    ✅ [Deep Research] Job Completed in {int(elapsed)}s on Attempt {attempt}.")

                            if not final_report_text and hasattr(job, 'outputs'):
                                for out in job.outputs:
                                    if getattr(out, 'type', '') == 'text':
                                        final_report_text += getattr(out, 'text', '')

                            class MockInteraction:
                                pass
                            mock_int = MockInteraction()

                            class MockOutput:
                                def __init__(self, text):
                                    self.text = text
                            mock_int.outputs = [MockOutput(final_report_text)]

                            return self._parse_output_to_catalog(mock_int)

                        elif status in ["FAILED", "CANCELLED"]:
                            print(f"\n    ❌ [Deep Research] Job instantly failed ({status}) on attempt {attempt}.")
                            attempt_failed = True
                            break

                        time.sleep(poll_interval)

                    except Exception as poll_err:
                        print("?", end="", flush=True)
                        time.sleep(poll_interval)

            if attempt_failed and attempt < max_retries:
                print(f"    🔄 Brute-force retry in 10 seconds...")
                time.sleep(60)

        print(f"\n    🛑 Exhausted all {max_retries} Deep Research attempts. Giving up.")
        return []

    def _execute_local_research(self, prompt: str) -> List[Dict[str, Any]]:
        print(f"\n🧠 [Local Deep Research] Initiating Gemma-based Agentic Research Loop...")
        query_prompt = (
            "You are the strategic planner for a data cataloging agent. "
            "Based on the task below, generate exactly 3 highly specific Google Search queries "
            "to find the official datasets and repositories.\n\n"
            f"TASK:\n{prompt}\n\n"
            "Return ONLY a JSON list of 3 strings. Do not use markdown blocks."
        )
        print("    🧠 Generating search strategy...")
        try:
            response = self.tools.generate_content_standard("LOCAL", query_prompt)
            if response.text == "[missing]":
                print("    ⚠️ Search strategy failed due to local LLM context limits.")
                queries = []
            else:
                queries = self.tools._extract_json_robustly(response.text)
        except Exception as e:
            if self.verbosity >= 2: print(f"    ⚠️ Strategy generation failed: {e}")
            queries = []

        if not isinstance(queries, list) or len(queries) == 0:
            context = getattr(self.config, 'PROJECT_CONTEXT', 'Target Project')
            queries = [f"{context} official dataset time series", f"{context} benchmark datasets", f"{context} repository github data"]

        aggregated_context = ""
        index_docs = []
        visited_urls = set()

        for q in queries[:3]:
            if not isinstance(q, str): continue
            print(f"    🔎 Searching: {q}")
            results = self.tools.tool_search_and_fetch(q, num_results=3)
            for res in results:
                url = res.get("url", "")
                if not url or url in visited_urls or "youtube.com" in url.lower(): continue
                visited_urls.add(url)
                content = res.get("content", "")
                title = res.get("title", "")
                print(f"    🕸️ Fetching: {url}")
                page_data = self.tools.tool_load_url(url)
                if page_data and len(page_data) > 0: content = page_data[0].get("content", content)
                aggregated_context += f"--- Source: {url} ---\n{content[:4000]}\n\n"
                index_docs.append({"title": title, "url": url, "content": content, "type": "Local Deep Research Grounding", "is_index_doc": True})

        if not aggregated_context.strip(): print("    ⚠️ No web context gathered. Falling back to zero-shot extraction.")
        print("    🧠 Synthesizing findings into Catalog format...")

        is_pro = self.config.LLM_BACKEND == "LOCAL_PRO"
        safe_context_len = 10000 if is_pro else 6000
        safe_context = aggregated_context[:safe_context_len]

        extraction_prompt = (
            f"TASK: {prompt}\n\n"
            "INSTRUCTIONS:\n"
            "You are extracting discovered_datasets. "
            "Based strictly on the gathered context below, extract the datasets into a JSON format. "
            "Format EXACTLY as: {\"discovered_datasets\": [{\"dataset_name\": \"Name\", \"Type\": \"Dataset\", \"Domain\": \"Finance\", \"Total Variables\": \"5\", \"Number of Time Points\": \"100\", \"Time interval between points\": \"1 hour\", \"Primary Source Repository\": \"Github\", \"Primary URL\": \"http...\", \"Link to Data (Actual Source)\": \"http...\", \"Other URL\": \"\", \"Detailed Description\": \"Desc\"}]} "
            "Output ONLY valid JSON.\n\n"
            f"=== GATHERED WEB CONTEXT ===\n"
            f"{safe_context}\n"
            f"============================\n"
        )

        try:
            final_response = self.tools.generate_content_standard("LOCAL", extraction_prompt, max_new_tokens=2048)
            if final_response.text == "[missing]":
                print("    ⚠️ Local Deep Research Extraction aborted due to LLM Context OOM or Safety Halt.")
                return index_docs

            data = self.tools._extract_json_robustly(final_response.text)
            parsed_items = []
            datasets = data.get("discovered_datasets", []) if isinstance(data, dict) else []
            if not datasets and isinstance(data, list): datasets = data

            valid_items = 0
            for d in datasets:
                if not isinstance(d, dict): continue
                raw_name = d.get("dataset_name", d.get("Dataset Name", ""))
                if not raw_name: continue

                item = {"Dataset Name": {"value": raw_name, "confidence": 0.80, "telemetry_context": "Local Deep Research"}}
                for col in ["Type", "Domain", "Total Variables", "Number of Time Points", "Time interval between points", "Primary Source Repository", "Primary URL", "Link to Data (Actual Source)", "Other URL", "Detailed Description"]:
                    val = d.get(col, "")
                    if val: item[col] = {"value": str(val), "confidence": 0.80, "telemetry_context": "Local Deep Research"}

                item["Assignment Confidence"] = {"value": "0.85", "confidence": 1.0}
                item["Assignment Rationale"] = {"value": "Identified by Local Gemma Deep Research Loop", "confidence": 1.0}

                parsed_items.append(item)
                valid_items += 1

            print(f"    🧠 Successfully parsed {valid_items} datasets from Local Deep Research.")
            if index_docs:
                for idx_doc in reversed(index_docs): parsed_items.insert(0, idx_doc)
            return parsed_items
        except Exception as e:
            print(f"    ⚠️ Local Deep Research Extraction Failed: {e}")
            return index_docs

    def _clean_text(self, text: str) -> str:
        if not text: return ""
        clean = text.replace("**", "").replace("__", "").strip("*").strip("_").strip()
        patterns = [
            r'\s*\((?:.*\bUTSD\b.*)\)', r'\s*\((?:.*\bTimer\b.*)\)',
            r'\s*\(Component\)', r'\s*\(Subset\)', r'\s*\(Collection\)'
        ]
        for p in patterns:
            clean = re.sub(p, '', clean, flags=re.IGNORECASE).strip()
        return clean

    def _clean_url_field(self, text: str) -> str:
        if not text: return ""
        import re
        md_links = re.findall(r'\[.*?\]\((https?://.*?)\)', text)
        raw_links = re.findall(r'(?<!\()(https?://[^\s>\"\'\)]+)', text)

        all_links = []
        if md_links:
            all_links.extend([m.strip() for m in md_links])

        for link in raw_links:
            clean_link = link.rstrip('.,;:')
            if clean_link not in all_links:
                all_links.append(clean_link)

        return ", ".join(all_links) if all_links else text.strip()

    def _parse_output_to_catalog(self, interaction: Any) -> List[Dict[str, Any]]:
        if not interaction.outputs: return []
        text = interaction.outputs[-1].text

        meta_item = {
            "title": "Deep Research Report", "url": "DEEP_RESEARCH_API",
            "content": text, "type": "Deep Research Report", "is_index_doc": True
        }
        parsed_items = [meta_item]

        try:
            lines = text.split('\n')
            table_lines = []
            capture = False

            for line in lines:
                if "Dataset Name" in line and "|" in line:
                    capture = True
                    line = line.replace("|||", "|")

                if capture:
                    if not line.strip():
                        capture = False
                        continue
                    if "---" in line: continue

                    line = line.replace("|||", "|")
                    line = line.strip().strip('|')
                    table_lines.append(line)

            if table_lines:
                reader = csv.DictReader(table_lines, delimiter='|')
                reader.fieldnames = [f.strip() for f in reader.fieldnames] if reader.fieldnames else []

                col_map = {
                    "Dataset Name": "Dataset Name", "Entity Type": "Type", "Type": "Type",
                    "Domain": "Domain", "Number of Variables": "Total Variables",
                    "Number of Time Points": "Number of Time Points",
                    "Time interval": "Time interval between points",
                    "Primary Source": "Primary Source Repository",
                    "Primary Home Page URL": "Primary URL",
                    "Link to Data (Actual Source)": "Link to Data (Actual Source)",
                    "Link to Data": "Link to Data (Actual Source)",
                    "Other URLs": "Other URL",
                    "Detailed Description": "Detailed Description"
                }

                valid_items = 0
                for row in reader:
                    row = {k: v.strip() for k, v in row.items() if v}
                    raw_name = self._clean_text(row.get("Dataset Name", ""))
                    if not raw_name or "---" in raw_name: continue

                    item = {"Dataset Name": {"value": raw_name, "confidence": 0.70, "telemetry_context": "Deep Research"}}

                    for dr_col, schema_col in col_map.items():
                        matched_key = next((k for k in row.keys() if dr_col in k), None)
                        if matched_key:
                            if "URL" in schema_col or "Link" in schema_col:
                                val = self._clean_url_field(str(row[matched_key]))
                            else:
                                val = self._clean_text(str(row[matched_key]))

                            if val and val.lower() != "nan":
                                if schema_col == "Type":
                                    if "collection" in val.lower() or "archive" in val.lower(): val = "Collection"
                                    elif "provider" in val.lower(): val = "Provider"
                                    else: val = "Dataset"
                                item[schema_col] = {"value": val, "confidence": 0.70, "telemetry_context": "Deep Research"}

                    item["Assignment Confidence"] = {"value": "0.80", "confidence": 1.0}
                    item["Assignment Rationale"] = {"value": "Identified by Deep Research Agent", "confidence": 1.0}
                    if "Type" not in item or not item["Type"].get("value"):
                        item["Type"] = {"value": "Dataset", "confidence": 0.8}

                    parsed_items.append(item)
                    valid_items += 1

                print(f"    🧠 Successfully parsed {valid_items} datasets from Deep Research.")

        except Exception as e:
            print(f"    ⚠️ Error parsing Deep Research table: {e}")

        return parsed_items

print("✅ deepcollector/tools/deep_research_runner.py written (Configurable Agent Model).")

Writing deepcollector/tools/deep_research_runner.py


#### **Cell 18 Progress Analytics**

In [ ]:
%%writefile deepcollector/utils/analytics.py
# =============================================================================
# V8: Performance Analytics (Workload Aggregation Fix)
# =============================================================================
import time
import pandas as pd
from tabulate import tabulate

class PerformanceAnalyzer:
    """
    Calculates ROI for each phase.
    Tracks:
    - Net Datasets Added
    - Deletions (Deduplication)
    - Cells Filled (Value from [missing] -> Data)
    - Cells Refined (Value changed)
    - Cells Confirmed (Value verified/unchanged)
    """
    def __init__(self):
        self.phases = []
        self.current_phase_start = None
        self.current_items_start = 0
        self.current_phase_name = ""

        self.fill_counts = {}
        self.refine_counts = {}
        self.confirm_counts = {}
        self.deletion_counts = {}

    def start_phase(self, phase_name: str, current_item_count: int):
        self.current_phase_start = time.time()
        self.current_items_start = current_item_count
        self.current_phase_name = phase_name

        self.fill_counts[phase_name] = 0
        self.refine_counts[phase_name] = 0
        self.confirm_counts[phase_name] = 0
        self.deletion_counts[phase_name] = 0

    def record_cell_change(self, change_type: str, count: int = 1):
        if not self.current_phase_name: return

        if change_type == 'FILL':
            self.fill_counts[self.current_phase_name] += count
        elif change_type == 'REFINE':
            self.refine_counts[self.current_phase_name] += count
        elif change_type == 'CONFIRM':
            self.confirm_counts[self.current_phase_name] += count

    def record_deletions(self, count: int):
        if self.current_phase_name:
            self.deletion_counts[self.current_phase_name] += count

    def end_phase(self, current_item_count: int):
        if not self.current_phase_start: return

        duration = time.time() - self.current_phase_start
        items_added = max(0, current_item_count - self.current_items_start)

        filled = self.fill_counts.get(self.current_phase_name, 0)
        refined = self.refine_counts.get(self.current_phase_name, 0)
        confirmed = self.confirm_counts.get(self.current_phase_name, 0)
        deletions = self.deletion_counts.get(self.current_phase_name, 0)

        minutes = duration / 60 if duration > 0 else 0.001

        self.phases.append({
            "Phase": self.current_phase_name,
            "Time (s)": duration,
            "Net Added": items_added,
            "Deletions": deletions,
            "Filled": filled,
            "Refined": refined,
            "Confirmed": confirmed,
            "Total Cell Ops": filled + refined + confirmed,
            "Velocity": items_added / minutes
        })
        self.current_phase_start = None

    def print_report(self):
        if not self.phases: return

        df = pd.DataFrame(self.phases)
        total_time = df["Time (s)"].sum()

        print("\n" + "="*100)
        print("💰 VALUE ANALYSIS REPORT (Tri-State Updates)")
        print("="*100)

        df_display = df.copy()
        df_display["Time %"] = (df_display["Time (s)"] / total_time * 100).map('{:.1f}%'.format)
        df_display["Time (s)"] = df_display["Time (s)"].map('{:.1f}'.format)
        df_display["Velocity"] = df_display["Velocity"].map('{:.2f}'.format)

        cols = ["Phase", "Time (s)", "Time %", "Net Added", "Deletions", "Filled", "Refined", "Confirmed", "Total Cell Ops", "Velocity"]
        print(tabulate(df_display[cols], headers="keys", tablefmt="simple", showindex=False))

        if not df.empty:
            best_phase = df.loc[df["Net Added"].idxmax()]
            if best_phase["Net Added"] > 0:
                print(f"\n💡 Insight: '{best_phase['Phase']}' contributed the most net datasets ({best_phase['Net Added']}).")

            # Evaluate based on Total Operations (Fills + Confirms) rather than just Fills
            most_work_idx = df["Total Cell Ops"].idxmax()
            most_work = df.loc[most_work_idx]

            most_fills_idx = df["Filled"].idxmax()
            most_fills = df.loc[most_fills_idx]

            if most_fills["Filled"] > 0:
                print(f"💡 Insight: '{most_fills['Phase']}' populated the most missing data ({most_fills['Filled']} Fills).")

            if most_work["Total Cell Ops"] > 0 and most_work_idx != most_fills_idx:
                print(f"💡 Insight: '{most_work['Phase']}' handled the heaviest workload, evaluating {most_work['Total Cell Ops']} cells (mostly Confirms).")

print("✅ deepcollector/utils/analytics.py written (V8: Workload Aggregation Fix).")

Writing deepcollector/utils/analytics.py


#### **Cell 19: deepcollector/kb/quality.py (The Quality Auditor Module)**

This module contains the logic for Identity-Based Deduplication and Quality Checks.

In [ ]:
%%writefile deepcollector/kb/quality.py
# =============================================================================
# V59.9: Quality Auditor (Description Bloat Fix)
# =============================================================================
import pandas as pd
import uuid
import time
import re
from datetime import datetime
from collections import defaultdict, Counter
from deepcollector.kb.merger import UniversalOracle

try:
    from deepcollector.utils.profiler import profiler
except ImportError:
    class DummyProfiler:
        def track(self, c): return lambda f: f
    profiler = DummyProfiler()

class QualityAuditor:
    def __init__(self, kb_manager, models=None, llm_limit=None):
        self.kb_manager = kb_manager
        self.models = models
        self.verbosity = getattr(kb_manager.config, 'VERBOSITY_LEVEL', 1)

        self.oracle = UniversalOracle(models=models, verbosity=self.verbosity)
        if llm_limit is not None:
            self.oracle.llm_limit = llm_limit
        else:
            self.oracle.llm_limit = getattr(kb_manager.config, 'LLM_ARBITRATION_LIMIT', 150)

        self.CONFIDENCE_THRESHOLD = 0.80
        self.FLAKY_THRESHOLD = 0.50

        self.project_stats = defaultdict(lambda: {
            'total': 0, 'low_conf': 0, 'dubious': 0, 'norm_error': 0,
            'missing_counts': Counter(), 'missing_time_points_list': [], 'norm_error_list': []
        })

        self.COL_DATASET_ID = "DatasetID"
        self.COL_PROJECT_ID = "ProjectID"
        self.COL_URL = "Primary URL"
        self.COL_DATA_URL = "Link to Data (Actual Source)"
        self.COL_OTHER_URL = "Other URL"
        self.COL_TYPE = "Type"
        self.EMPTY_MARKERS = self.oracle.MISSING

    def _get_display_name(self, row):
        did = str(row.get(self.COL_DATASET_ID, "UnknownID"))
        eff_name = self.oracle._clean_name(row.get("Variant Name", "") or row.get("Canonical Name", ""))
        return f"{did} ({eff_name})"

    @profiler.track("Quality: Full Audit & Relink")
    def run_audit_and_clean(self, run_quarantine=True, dry_run=False, job_id=None, project_id="GLOBAL", job_comment=""):
        start_time = datetime.now()
        job_id = job_id or f"JOB_{uuid.uuid4().hex[:6].upper()}"

        mode_str = "🚫 DRY RUN" if dry_run else "💾 LIVE"
        print("\n" + "="*60)
        print(f"🔍 STARTING DATA QUALITY AUDIT & RELINKING [{mode_str}] (V59.9)")
        print(f"🚀 Job ID: {job_id}")
        if job_comment:
            print(f"📝 Job Comment: {job_comment}")
        print("="*60)

        from deepcollector.kb.manager import SheetLock
        lock = SheetLock(self.kb_manager, job_id, self.verbosity)
        if not lock.acquire(timeout_seconds=7200):
            print("    ❌ Failed to acquire lock for Quality Audit.")
            return

        try:
            if not self.kb_manager.read_and_validate_kb():
                print("    ❌ Failed to load Knowledge Base.")
                return

            df = self.kb_manager.get_kb_data("Datasets")
            if df.empty:
                print("    ⚠️ Datasets tab is empty. Aborting.")
                return

            print("    🔗 Resolving Project Links...")
            df_links = self.kb_manager.get_kb_data("Project_Dataset_Link")
            if df_links.empty or self.COL_DATASET_ID not in df_links.columns:
                print("    🛑 ABORT: Linking table missing/empty.")
                return

            if run_quarantine:
                df, df_links = self._quarantine_low_confidence(df, df_links, dry_run=dry_run)

            ds_to_proj = dict(zip(df_links[self.COL_DATASET_ID], df_links[self.COL_PROJECT_ID]))
            df['Project_ID'] = df[self.COL_DATASET_ID].map(ds_to_proj).fillna('Unknown')

            if self.models:
                print(f"\n    🧠 Universal Oracle Arbitration: ENABLED (Limit: {self.oracle.llm_limit} calls)")
            else:
                print(f"\n    ⚪ Universal Oracle Arbitration: DISABLED")

            print(f"\n🧠 Running Universal Oracle Deduplication...")

            df_clean, dupes_log = self._advanced_deduplication(df)
            duplicates_removed = len(df) - len(df_clean)

            for pid in df_clean['Project_ID'].unique():
                self.project_stats[str(pid)]['total'] = len(df_clean[df_clean['Project_ID'] == pid])

            print("\n🤖 ORACLE USAGE SUMMARY:")
            print(f"   Total LLM Calls: {self.oracle.stats['llm_calls']} / {self.oracle.llm_limit}")
            print(f"   ✅ Merges Approved: {self.oracle.stats['approved']}")
            print(f"   ❌ Merges Rejected: {self.oracle.stats['rejected']}")
            if self.oracle.stats['errors'] > 0:
                print(f"   ⚠️ API Errors/Quota: {self.oracle.stats['errors']}")

            print("\n--- Deduplication Results ---")
            print(f"    Original Count (Post-Quarantine): {len(df)}")
            print(f"    Clean Count:    {len(df_clean)}")
            print(f"    Duplicates Removed: {duplicates_removed}")

            if not dupes_log.empty and not df_links.empty:
                print(f"\n🔗 RELINKING MODE: Reassigning {len(dupes_log)} merged datasets into 'Project_Dataset_Link' tab...")
                df_links_clean = df_links.copy()
                drop_to_keep = dict(zip(dupes_log['DatasetID (Dropped)'], dupes_log['DatasetID (Kept)']))
                before_len = len(df_links_clean)
                relinked_count = 0

                for idx, row in df_links_clean.iterrows():
                    ds_id = row.get(self.COL_DATASET_ID)
                    if ds_id in drop_to_keep:
                        curr_id = ds_id
                        visited = set()
                        while curr_id in drop_to_keep and curr_id not in visited:
                            visited.add(curr_id)
                            curr_id = drop_to_keep[curr_id]
                        df_links_clean.at[idx, self.COL_DATASET_ID] = curr_id

                        old_comments = str(row.get('Data Preparation Comments', ''))
                        relink_note = f"[RELINKED from {ds_id}]"
                        if old_comments and old_comments.lower() != "nan" and old_comments not in self.EMPTY_MARKERS:
                            if relink_note not in old_comments:
                                df_links_clean.at[idx, 'Data Preparation Comments'] = f"{old_comments}; {relink_note}"
                        else:
                            df_links_clean.at[idx, 'Data Preparation Comments'] = relink_note
                        relinked_count += 1

                df_links_clean = df_links_clean.drop_duplicates(subset=[self.COL_PROJECT_ID, self.COL_DATASET_ID])
                after_len = len(df_links_clean)
                print(f"    ✨ Relinking complete. Updated {relinked_count} rows and removed {before_len - after_len} redundant links.")

                if self.kb_manager.is_connected:
                    if dry_run:
                        print("    🚫 [DRY RUN] Skipping write to 'Project_Dataset_Link'.")
                    else:
                        print("    💾 Writing updated links to 'Project_Dataset_Link'...")
                        self.kb_manager.write_dataframe_to_tab("Project_Dataset_Link", df_links_clean)

            print("\n📊 Analyzing Final Catalog Quality...")
            quality_report, _ = self._analyze_quality(df_clean)
            self._print_quality_summary(quality_report, len(df_clean))
            self._print_project_health_report()

            if dry_run:
                print("\n🚫 DRY RUN COMPLETE. No changes were saved to Google Sheets.")
            else:
                if self.kb_manager.is_connected:
                    print("\n💾 Writing to 'Datasets'...")
                    self.kb_manager.write_dataframe_to_tab("Datasets", df_clean)

                    if not dupes_log.empty:
                        print(f"    💾 Writing {len(dupes_log)} records to 'Datasets_Duplicates_Log'...")
                        self.kb_manager.write_dataframe_to_tab("Datasets_Duplicates_Log", dupes_log)
                    print("    ✅ Cleanup & Relinking process complete.")

                duration = (datetime.now() - start_time).total_seconds()
                job_data = {
                    "JobID": job_id,
                    "ProjectID": project_id,
                    "Mode": "REVIEW",
                    "Start_Time": start_time.strftime("%Y-%m-%d %H:%M:%S"),
                    "End_Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "Duration_Sec": f"{duration:.2f}",
                    "Status": "COMPLETED",
                    "Items_Found": len(df_clean),
                    "Operational_Parameters": str({"Quarantine_Threshold": self.FLAKY_THRESHOLD}),
                    "JOB_COMMENT": job_comment
                }
                self.kb_manager.log_job_execution(job_data)

        except Exception as e:
            print(f"    ❌ Critical Error during quality audit: {e}")

        finally:
            lock.release()

    def _quarantine_low_confidence(self, df: pd.DataFrame, df_links: pd.DataFrame, dry_run: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
        print(f"\n🩺 Running Flaky Dataset Quarantine (Threshold < {self.FLAKY_THRESHOLD})...")

        if 'Overall Confidence' not in df.columns:
            print("    ⚠️ 'Overall Confidence' column not found. Skipping quarantine.")
            return df, df_links

        df['_num_conf'] = pd.to_numeric(df['Overall Confidence'], errors='coerce').fillna(1.0)

        flaky_mask = df['_num_conf'] < self.FLAKY_THRESHOLD
        df_flaky = df[flaky_mask].copy()
        df_healthy = df[~flaky_mask].copy()

        df_healthy = df_healthy.drop(columns=['_num_conf'])
        df_flaky = df_flaky.drop(columns=['_num_conf'])

        if df_flaky.empty:
            print("    ✅ No flaky datasets found.")
            return df_healthy, df_links

        flaky_ids = df_flaky[self.COL_DATASET_ID].tolist()
        print(f"    ☢️ Found {len(flaky_ids)} datasets with explicitly low confidence.")

        initial_links = len(df_links)
        df_links_flaky = df_links[df_links[self.COL_DATASET_ID].isin(flaky_ids)].copy()
        df_links_healthy = df_links[~df_links[self.COL_DATASET_ID].isin(flaky_ids)].copy()

        if self.kb_manager.is_connected:
            if dry_run:
                print(f"    🚫 [DRY RUN] Would move {len(flaky_ids)} datasets to 'Datasets_Quarantined'.")
            else:
                print(f"    💾 Archiving flaky datasets to 'Datasets_Quarantined' tab...")
                try:
                    existing_quarantine = self.kb_manager.get_kb_data("Datasets_Quarantined")
                except Exception:
                    existing_quarantine = pd.DataFrame()

                if not existing_quarantine.empty:
                    df_flaky = pd.concat([existing_quarantine, df_flaky], ignore_index=True)
                    df_flaky = df_flaky.drop_duplicates(subset=[self.COL_DATASET_ID])
                self.kb_manager.write_dataframe_to_tab("Datasets_Quarantined", df_flaky)

                print(f"    💾 Preserving provenance in 'Links_Quarantined' tab...")
                try:
                    existing_q_links = self.kb_manager.get_kb_data("Links_Quarantined")
                except Exception:
                    existing_q_links = pd.DataFrame()

                if not existing_q_links.empty:
                    df_links_flaky = pd.concat([existing_q_links, df_links_flaky], ignore_index=True)
                    df_links_flaky = df_links_flaky.drop_duplicates(subset=[self.COL_PROJECT_ID, self.COL_DATASET_ID])
                self.kb_manager.write_dataframe_to_tab("Links_Quarantined", df_links_flaky)

        return df_healthy, df_links_healthy

    def _advanced_deduplication(self, df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        merge_cols = [(c[:-4], c) for c in df.columns if c.endswith('(C)')]
        additive_cols = ["Description", "Assignment Rationale", "Project Citations", "Project WebLinks", "Primary URL", "Link to Data (Actual Source)", "Other URL"]
        av_sort = [c for c in ['Overall Confidence', 'Job_Updated'] if c in df.columns]
        df = df.sort_values(by=av_sort, ascending=[False]*len(av_sort))

        records = df.to_dict('records')
        kept_records = []
        duplicates_log = []
        skip_indices = set()

        for i, row in enumerate(records):
            if i % 20 == 0:
                print(f"    ⏳ Oracle Processing {i}/{len(records)}...")
            if i in skip_indices: continue

            is_duplicate = False
            match_reason = ""
            kept_idx = -1

            for k_idx, kept in enumerate(kept_records):
                try:
                    is_duplicate, match_reason = self.oracle.evaluate_pair(row, kept)
                except Exception as e:
                    if self.verbosity >= 2:
                        print(f"      ⚠️ Oracle error comparing records (Safe Skip): {e}")
                    is_duplicate = False

                if is_duplicate:
                    kept_idx = k_idx
                    break

            if is_duplicate and kept_idx != -1:
                skip_indices.add(i)
                kept_row = kept_records[kept_idx]
                merge_notes = []

                for col in additive_cols:
                    if col in kept_row and col in row:
                        val1, val2 = str(kept_row[col]), str(row[col])

                        if val2 not in self.EMPTY_MARKERS:
                            if val1 in self.EMPTY_MARKERS:
                                kept_row[col] = val2
                                merge_notes.append(f"Filled {col}")
                            elif "URL" in col or "Link" in col:
                                set1 = {v.strip() for v in re.split(r'[,|]', val1) if v.strip()}
                                set2 = {v.strip() for v in re.split(r'[,|]', val2) if v.strip()}
                                new_items = set2 - set1
                                if new_items:
                                    kept_row[col] = val1 + " | " + " | ".join(new_items)
                                    merge_notes.append(f"Added {col}")
                            else:
                                # CRITICAL FIX: Only REPLACE with the longer text, do NOT concatenate descriptions with | anymore
                                if len(val2) > len(val1):
                                    kept_row[col] = val2
                                    merge_notes.append(f"Replaced {col} with longer text")

                for val_col, conf_col in merge_cols:
                    try:
                        c_curr = float(row.get(conf_col, 0)); c_kept = float(kept_row.get(conf_col, 0))
                        v_curr = str(row.get(val_col, ""))
                        if v_curr not in self.EMPTY_MARKERS and c_curr > c_kept:
                            kept_row[val_col] = v_curr; kept_row[conf_col] = c_curr; merge_notes.append(f"Upgraded {val_col}")
                    except: pass

                duplicates_log.append({
                    "DatasetID (Dropped)": row.get('DatasetID', ''),
                    "DatasetID (Kept)": kept_row.get('DatasetID', ''),
                    "Reason": match_reason,
                    "Merge Notes": "; ".join(merge_notes[:5]),
                    "Dropped Name": row.get("Variant Name", ""),
                    "Kept Name": kept_row.get("Variant Name", "")
                })
                kept_records[kept_idx] = kept_row
            else:
                kept_records.append(row)

        df_clean = pd.DataFrame(kept_records)
        df_dupes = pd.DataFrame(duplicates_log)
        return df_clean, df_dupes

    def _analyze_quality(self, df: pd.DataFrame):
        df = df.copy()
        conf_cols = [c for c in df.columns if c.endswith('(C)')]
        stats = {'by_column': {}, 'total_low_conf_cells': 0, 'rows_with_issues': 0}

        for col in conf_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

        for idx, row in df.iterrows():
            pid = str(row.get('Project_ID', 'Unknown'))
            display_name = self._get_display_name(row)
            row_issues = 0

            for col in conf_cols:
                clean_name = col.replace(' (C)', '')
                if "frequency" in clean_name.lower() or "time interval" in clean_name.lower():
                    continue

                if row[col] < self.CONFIDENCE_THRESHOLD:
                    stats['by_column'][clean_name] = stats['by_column'].get(clean_name, 0) + 1
                    stats['total_low_conf_cells'] += 1
                    self.project_stats[pid]['missing_counts'][clean_name] += 1
                    row_issues += 1

                    if clean_name == "Num Time Points":
                        self.project_stats[pid]['missing_time_points_list'].append(display_name)

            if row_issues > 0:
                stats['rows_with_issues'] += 1
                self.project_stats[pid]['low_conf'] += 1

            type_val = str(row.get(self.COL_TYPE, '')).lower().strip()
            url_val = str(row.get(self.COL_URL, '')).lower().strip()
            data_val = str(row.get(self.COL_DATA_URL, '')).lower().strip()

            if type_val in self.EMPTY_MARKERS or (url_val in self.EMPTY_MARKERS and data_val in self.EMPTY_MARKERS):
                self.project_stats[pid]['dubious'] += 1

        return stats, df

    def _print_quality_summary(self, stats, total_rows):
        print("\n--- 📊 Final Quality Summary ---")
        print(f"Total Datasets: {total_rows}")
        pct_issues = (stats['rows_with_issues'] / total_rows) * 100 if total_rows > 0 else 0
        print(f"Datasets with ≥1 Low-Conf Field (excluding Frequency): {stats['rows_with_issues']} ({pct_issues:.1f}%)")
        print("\n--- Low Confidence by Field ---")
        sorted_cols = sorted(stats['by_column'].items(), key=lambda x: x, reverse=True)
        for col, count in sorted_cols:
            pct = (count / total_rows) * 100 if total_rows > 0 else 0
            if count > 0: print(f"  • {col}: {count} ({pct:.1f}%)")

    def _print_project_health_report(self):
        print("\n" + "="*80)
        print("🏗️ PROJECT HEALTH & DIAGNOSTICS")
        print("="*80)
        print(f"{'PROJECT ID':<20} | {'TOT':<4} | {'BAD':<4} | {'NORM':<4} | {'STATUS':<10} | {'PRIMARY DEFECTS'}")
        print("-" * 100)

        try:
            if not hasattr(self, 'project_stats') or not isinstance(self.project_stats, dict):
                return

            safe_list = []
            for k in list(self.project_stats.keys()):
                v = self.project_stats[k]
                if isinstance(v, dict):
                    record = {
                        'pid': str(k),
                        'total': int(v.get('total', 0)),
                        'low_conf': int(v.get('low_conf', 0)),
                        'norm_error': int(v.get('norm_error', 0)),
                        'missing_counts': v.get('missing_counts')
                    }
                    safe_list.append(record)

            safe_list.sort(key=lambda x: x['total'], reverse=True)

            for s in safe_list:
                tot = s['total']
                if tot == 0: continue

                defects = []
                missing = s['missing_counts']
                if missing and hasattr(missing, 'most_common'):
                    try:
                        for field, count in missing.most_common(3):
                            pct = int((count / tot) * 100) if tot > 0 else 0
                            defects.append(f"{field} ({pct}%)")
                    except Exception:
                        pass

                defect_str = ", ".join(defects) if defects else "None"
                low = s['low_conf']
                norm = s['norm_error']

                rate = (low / tot) * 100 if tot > 0 else 0
                status = "✅ OK"
                if norm > 0: status = "⚠️ NORM"
                if rate > 30: status = "🛑 RERUN"
                if norm > 5: status = "🛑 CRIT"

                safe_pid = s['pid'][:20]
                print(f"{safe_pid:<20} | {tot:<4} | {low:<4} | {norm:<4} | {status:<10} | {defect_str}")

        except Exception as e:
            print(f"    ⚠️ Report generation failed safely: {e}")

print("✅ deepcollector/kb/quality.py written (Description Bloat Fix).")

Writing deepcollector/kb/quality.py


####**Cell 20 Merge Reruns merger.py**

In [ ]:
%%writefile deepcollector/kb/merger.py
# =============================================================================
# V58.15: Project Merger (Short-Acronym Dedup Shield to Prevent False Merges)
# =============================================================================
import pandas as pd
import difflib
import re
import json
import time
import sys
from collections import defaultdict
from typing import Dict, Tuple, List, Any
from tabulate import tabulate
from pydantic import BaseModel, Field

try:
    from google.genai import types
except ImportError:
    types = None

class OracleResponseSchema(BaseModel):
    is_same: bool = Field(description="True if records describe the exact same entity.")
    rationale: str = Field(description="Brief reason for the decision.")

class SingletonVerificationSchema(BaseModel):
    in_project: bool = Field(description="True if dataset belongs to the Target Entity.")
    exists: bool = Field(description="True if dataset actually exists.")
    rationale: str = Field(description="Short explanation.")

class UniversalOracle:
    def __init__(self, models=None, verbosity=1, enable_search=False):
        self.models = models
        self.verbosity = verbosity
        self.enable_search = enable_search
        self.llm_cache = {}
        self.llm_limit = 1500
        self.model_cooldowns = {}
        self.model_stats = defaultdict(lambda: {"count": 0, "time": 0.0, "time_sq": 0.0})
        self.stats = {'llm_calls': 0, 'approved': 0, 'rejected': 0, 'errors': 0, 'hard_blocks': 0, 'rate_limits_hit': 0}
        self.MISSING = {"", "nan", "none", "[missing]", "[skipped]", "unknown", "n/a", "null"}
        self.STOP_WORDS = {"dataset", "data", "benchmark", "collection", "repository", "archive", "series", "time", "prediction", "forecasting", "competition"}
        self.GENERIC_URLS = ["search", "query", "google.com", "bing.com", "kaggle.com", "archive.ics.uci.edu", "zenodo.org", "github.com", "timeseriesclassification.com", "huggingface.co", "hf.co"]
        self.DISCRIMINATORS = {"france", "belgium", "germany", "spain", "italy", "uk", "usa", "china", "japan", "australia", "nord", "pjm", "california", "texas", "ny", "london", "ottawa", "halifax", "hourly", "daily", "weekly", "monthly", "yearly", "minute", "second", "test", "train", "val", "validation", "small", "large", "full", "subset", "np", "de", "fr", "be", "es", "it", "pems03", "pems04", "pems07", "pems08", "m1", "m3", "m4", "m5", "gef12", "gfc12", "gfc14", "gfc17"}

    def _call_llm_with_cascade(self, prompt: str, pool_preference: str = "FLASH", **kwargs) -> str:
        if not self.models or not getattr(self.models, 'CLIENT', None):
            sys.exit("LLM Client missing in Oracle.")

        all_models = (self.models.FLASH_POOL or []) + (self.models.PRO_POOL or []) if pool_preference == "FLASH" else (self.models.PRO_POOL or []) + (self.models.FLASH_POOL or [])
        base_sleep = 1.0 if pool_preference == "FLASH" else 3.0

        if not all_models: sys.exit("No models available.")

        for attempt in range(4):
            for model_name in all_models:
                if time.time() < self.model_cooldowns.get(model_name, 0): continue
                api_start = time.time()
                req_kwargs = {"model": model_name, "contents": prompt}
                req_kwargs.update(kwargs)

                if self.enable_search and types:
                    existing_config = req_kwargs.get("config")
                    if existing_config: existing_config.tools = [types.Tool(google_search=types.GoogleSearch())]
                    else: req_kwargs["config"] = types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])

                try:
                    time.sleep(base_sleep)
                    response = self.models.CLIENT.models.generate_content(**req_kwargs)
                    dur = time.time() - api_start
                    self.model_stats[model_name]["time"] += dur
                    self.model_stats[model_name]["time_sq"] += dur ** 2
                    self.model_stats[model_name]["count"] += 1
                    return response.text

                except Exception as e:
                    dur = time.time() - api_start
                    self.model_stats[model_name]["time"] += dur
                    self.model_stats[model_name]["time_sq"] += dur ** 2
                    self.model_stats[model_name]["count"] += 1

                    err_str = str(e).lower()
                    if "429" in err_str or "exhausted" in err_str or "quota" in err_str or "503" in err_str:
                        self.stats['rate_limits_hit'] += 1
                        if self.verbosity >= 2: print(f"        ⏭️ Quota/503 hit on {model_name}. Cascading...")
                        self.model_cooldowns[model_name] = time.time() + 120
                        continue
                    else:
                        self.stats['errors'] += 1
                        if self.verbosity >= 2: print(f"        ⚠️ API Error on {model_name}: {str(e)[:100]}")
                        continue

            max_cooldown = max(self.model_cooldowns.values()) if self.model_cooldowns else time.time()
            sleep_time = max(1.0, max_cooldown - time.time() + 1.0)
            if sleep_time > 120: sleep_time = 15
            if self.verbosity >= 1: print(f"        💤 Models exhausted. Sleeping {sleep_time:.1f}s for quota recovery...")
            time.sleep(sleep_time)

        if self.verbosity >= 1: print("\n🛑 CRITICAL ABORT: Failed across all available models after multiple attempts.")
        raise RuntimeError("LLM Cascade failed: All models exhausted or on cooldown.")

    def _are_names_distinct_variants(self, n1, n2):
        n1 = str(n1).lower().strip(); n2 = str(n2).lower().strip()
        if n1 == n2: return False

        # FIXED: Hard block for distinct short-acronyms (e.g. "etth1" vs "ettm1")
        if len(n1) <= 6 and len(n2) <= 6 and n1 != n2:
            self.stats['hard_blocks'] += 1
            return True

        m1 = re.search(r'(\d+)$', n1); m2 = re.search(r'(\d+)$', n2)
        if m1 and m2 and m1.group(1) != m2.group(1):
            if n1[:m1.start()].strip() == n2[:m2.start()].strip():
                self.stats['hard_blocks'] += 1
                return True
        tokens1 = set(re.findall(r'[a-z0-9]+', n1)); tokens2 = set(re.findall(r'[a-z0-9]+', n2))
        d1 = self.DISCRIMINATORS.intersection(tokens1)
        d2 = self.DISCRIMINATORS.intersection(tokens2)

        if (d1 or d2) and (d1 != d2):
            self.stats['hard_blocks'] += 1
            return True
        return False

    def _clean_name(self, name):
        try:
            if not name: return ""
            n_str = str(name).lower().strip()
            if n_str in self.MISSING: return ""
            n_str = re.sub(r'\(.*?\)', '', n_str)
            return " ".join([t for t in re.split(r'[\s_\-]+', n_str) if t not in self.STOP_WORDS and t.isalnum()])
        except Exception:
            return ""

    def _extract_digits(self, val):
        try:
            if not val: return -1
            v_str = str(val).lower().strip()
            if v_str in self.MISSING: return -1
            return int(re.findall(r'\d+', v_str.replace(',', ''))[0])
        except Exception:
            return -1

    def _clean_url(self, url):
        try:
            if not url: return ""
            u_str = str(url).strip().lower()
            if u_str in self.MISSING: return ""
            if '?' in u_str: u_str = u_str.split('?')[0]
            if hasattr(u_str, 'rstrip'): return u_str.rstrip('/')
            return str(u_str)
        except Exception:
            return str(url)

    def _is_generic_url(self, url):
        if not url or len(str(url)) < 15: return True
        for bad in self.GENERIC_URLS:
            if bad in str(url): return True
        return False

    def evaluate_pair(self, row_a: Dict, row_b: Dict) -> Tuple[bool, str]:
        name_a = str(row_a.get("Variant Name", "") or row_a.get("Canonical Name", "")).strip()
        name_b = str(row_b.get("Variant Name", "") or row_b.get("Canonical Name", "")).strip()

        if self._are_names_distinct_variants(name_a, name_b): return False, "HARD_DISCRIMINATOR_CONFLICT"

        clean_a = self._clean_name(name_a); clean_b = self._clean_name(name_b)
        t_a = self._extract_digits(row_a.get("Num Time Points", ""))
        t_b = self._extract_digits(row_b.get("Num Time Points", ""))
        v_a = self._extract_digits(row_a.get("Total Variables", ""))
        v_b = self._extract_digits(row_b.get("Total Variables", ""))
        url_a = self._clean_url(row_a.get("Primary URL", "")); url_b = self._clean_url(row_b.get("Primary URL", ""))
        data_a = self._clean_url(row_a.get("Link to Data (Actual Source)", "")); data_b = self._clean_url(row_b.get("Link to Data (Actual Source)", ""))
        type_a = str(row_a.get("Type", "")).strip(); type_b = str(row_b.get("Type", "")).strip()

        ratio = difflib.SequenceMatcher(None, clean_a, clean_b).ratio()
        is_substring = (clean_a in clean_b or clean_b in clean_a) and len(clean_a) > 4 and len(clean_b) > 4
        match_reason = None

        fuzzy_threshold = 0.90 if len(clean_a) <= 8 or len(clean_b) <= 8 else 0.82

        if (url_a and url_b and url_a != "[missing]" and url_a == url_b and not self._is_generic_url(url_a)) or \
           (data_a and data_b and data_a != "[missing]" and data_a == data_b and not self._is_generic_url(data_a)):
            match_reason = "EXACT_URL_MATCH_WEAK_NAME"
        elif t_a > 10 and t_b > 10 and v_a > 0 and v_b > 0 and t_a == t_b and v_a == v_b and ratio >= 0.40: match_reason = "FINGERPRINT_STRONG"
        elif clean_a and clean_b and clean_a == clean_b: match_reason = "EXACT_NAME"
        elif ratio >= 0.90: match_reason = "FUZZY_STRONG"
        elif ratio >= fuzzy_threshold or is_substring: match_reason = "FUZZY_WEAK"
        elif len(clean_a) > 3 and len(clean_b) > 3 and (clean_a.startswith(clean_b) or clean_b.startswith(clean_a)): match_reason = "PREFIX_MATCH_WEAK"

        if not match_reason: return False, ""

        if type_a and type_b and type_a.lower() != type_b.lower() and type_a.lower() not in self.MISSING and type_b.lower() not in self.MISSING:
            match_reason = f"{match_reason}_TYPE_MISMATCH"

        if "WEAK" in match_reason or "MISMATCH" in match_reason or "FINGERPRINT" in match_reason:
            if self.models and hasattr(self.models, 'CLIENT') and self.models.CLIENT:
                if self.stats['llm_calls'] < self.llm_limit:
                    is_llm_match = self._verify_with_llm(row_a, row_b, match_reason)
                    if is_llm_match: return True, f"{match_reason} (LLM Verified)"
                    else: return False, f"{match_reason} (LLM Rejected)"
                else: return False, f"{match_reason} (LLM Limit Reached)"
            else: return False, f"{match_reason} (LLM Disabled)"

        return True, match_reason

    def _verify_with_llm(self, row_a, row_b, match_type):
        name_a = row_a.get("Variant Name", "") or row_a.get("Canonical Name", "")
        name_b = row_b.get("Variant Name", "") or row_b.get("Canonical Name", "")
        type_a = str(row_a.get("Type", "Dataset")).strip()
        type_b = str(row_b.get("Type", "Dataset")).strip()
        desc_a = str(row_a.get("Description", ""))[:300]
        desc_b = str(row_b.get("Description", ""))[:300]
        t_a = row_a.get("Num Time Points", ""); v_a = row_a.get("Total Variables", "")
        t_b = row_b.get("Num Time Points", ""); v_b = row_b.get("Total Variables", "")
        url_a = row_a.get("Primary URL", ""); url_b = row_b.get("Primary URL", "")

        entity_a = (str(name_a), str(t_a), str(v_a), str(url_a))
        entity_b = (str(name_b), str(t_b), str(v_b), str(url_b))
        cache_key = tuple(sorted([entity_a, entity_b]))

        if cache_key in self.llm_cache: return self.llm_cache[cache_key]

        task_desc = f"Arbitrating (Rule: {match_type}): '{name_a[:35]}' vs '{name_b[:35]}'"
        prompt = (
            f"Act as a Data Archivist deduplicating a Time Series dataset catalog.\n"
            f"Determine if these two records describe the EXACT SAME underlying entity, even if names vary.\n"
            f"CRITICAL: Pay close attention to version numbers and geographic regions.\n"
            f"Type labels are sometimes applied inconsistently. Do NOT reject solely on Type mismatch.\n\n"
            f"RECORD 1:\nName: {name_a}\nType: {type_a}\nTime Points: {t_a}\nVariables: {v_a}\nURL: {url_a}\nDescription: {desc_a}\n\n"
            f"RECORD 2:\nName: {name_b}\nType: {type_b}\nTime Points: {t_b}\nVariables: {v_b}\nURL: {url_b}\nDescription: {desc_b}\n\n"
            f"Triggered by Rule: {match_type}"
        )

        try:
            self.stats['llm_calls'] += 1
            kwargs = {}
            if types: kwargs["config"] = types.GenerateContentConfig(response_mime_type="application/json", response_schema=OracleResponseSchema)
            response_text = self._call_llm_with_cascade(prompt, pool_preference="FLASH", **kwargs)

            try:
                result = json.loads(response_text)
                is_match = result.get('is_same', False)
            except json.JSONDecodeError:
                match = re.search(r"\{.*?\}", response_text.strip(), re.DOTALL)
                if match:
                    try:
                        result = json.loads(match.group())
                        is_match = result.get('is_same', False)
                    except json.JSONDecodeError:
                        is_match = False
                else: is_match = False

            self.llm_cache[cache_key] = is_match
            if is_match:
                self.stats['approved'] += 1
                if self.verbosity >= 1: print(f"      ✅ Confirmed: {task_desc}")
            else:
                self.stats['rejected'] += 1
                if self.verbosity >= 1: print(f"      ❌ Rejected:  {task_desc}")
            return is_match

        except Exception as e:
            self.stats['errors'] += 1
            if self.verbosity >= 1: print(f"      ⚠️ Error/Timeout. Defaulting to Rejected. ({str(e)[:40]}): {task_desc}")
            return False

class ProjectMerger:
    def __init__(self, kb_manager, verbosity=1, tools=None, models=None):
        self.kb_manager = kb_manager
        self.verbosity = verbosity
        self.tools = tools
        enable_search = getattr(self.kb_manager.config, 'ENABLE_ORACLE_SEARCH', False)
        self.oracle = UniversalOracle(models=models, verbosity=verbosity, enable_search=enable_search)
        self.MISSING = self.oracle.MISSING
        self.DATA_FIELDS = ["Domain", "Frequency", "Num Time Points", "Num Locations/Series", "Total Variables", "Variables per Location", "Primary Creator", "Primary URL", "Link to Data (Actual Source)", "Other URL", "Description", "Type"]

    def _build_project_grounding_context(self, project_context, urls):
        if not urls: return ""
        if self.verbosity >= 1: print(f"    🌐 [Merger] Fetching grounding text from {len(urls)} URL(s)...")
        combined_text = ""
        for i, url in enumerate(urls[:5]):
            try:
                fetched = self.tools.tool_load_url(url)
                if fetched and isinstance(fetched, list) and len(fetched) > 0 and "content" in fetched[0]:
                    combined_text += f"\n\n--- SOURCE {i+1}: {url} ---\n{fetched[0]['content']}"
            except Exception as e:
                if self.verbosity >= 2: print(f"      ⚠️ Failed to fetch {url}: {e}")
        if not combined_text.strip(): return ""
        final_text = re.sub(r'\s+', ' ', combined_text).strip()[:1000000]
        if self.verbosity >= 1: print(f"      ✅ Loaded {len(final_text)} characters of raw context for Oracle Verification.")
        return final_text

    def execute_merge(self, project_id: str, job_id: str, dry_run: bool = False, models_verifier=None, enable_singleton_verification: bool = None):
        if models_verifier: self.oracle.models = models_verifier
        if enable_singleton_verification is None: enable_singleton_verification = getattr(self.kb_manager.config, 'ENABLE_SINGLETON_VERIFICATION', True)

        is_global = (project_id == "GLOBAL")
        if is_global: enable_singleton_verification = False

        if enable_singleton_verification and (not self.oracle.models or not getattr(self.oracle.models, 'CLIENT', None)):
            sys.exit("LLM Client missing in MERGE mode.")

        project_context = getattr(self.kb_manager.config, 'PROJECT_CONTEXT', project_id.replace("PROJ_", ""))
        mode_str = "🚫 DRY RUN" if dry_run else "💾 LIVE"
        scope_str = "🌍 GLOBAL SCAN" if is_global else f"🎯 PROJECT: {project_id}"
        search_str = "🔍 Web Grounding ON" if self.oracle.enable_search else "📖 Closed Book"
        print(f"\n🤝 STARTING MERGE: {scope_str} [{mode_str}] | {search_str} (V58.15)")

        if not enable_singleton_verification:
            if is_global: print("    🌍 GLOBAL MODE: Bypassing Singleton Project-Relevance Checks (Deduplication Only).")
            else: print("    ⏩ Note: LLM Singleton Verification is bypassed.")

        project_source_text = ""
        if not is_global and enable_singleton_verification and self.tools and getattr(self.kb_manager.config, 'INITIAL_URLS', []):
            project_source_text = self._build_project_grounding_context(project_context, getattr(self.kb_manager.config, 'INITIAL_URLS', []))

        from deepcollector.kb.manager import SheetLock
        lock = SheetLock(self.kb_manager, job_id, self.verbosity)
        if not lock.acquire(timeout_seconds=14400):
            print("    ❌ Failed to acquire lock for merge operation.")
            return

        try:
            if not self.kb_manager.read_and_validate_kb():
                print("    ❌ Failed to load Knowledge Base.")
                return

            df_ds = self.kb_manager.get_kb_data("Datasets")
            df_links = self.kb_manager.get_kb_data("Project_Dataset_Link")

            if df_ds.empty: print("    ℹ️ No data."); return

            if is_global:
                project_datasets = df_ds.to_dict('records')
                linked_ids = set()
            else:
                proj_links = df_links[df_links["ProjectID"] == project_id]
                linked_job_map = dict(zip(proj_links["DatasetID"], proj_links["Linked_By_Job"]))
                linked_ids = proj_links["DatasetID"].tolist()
                project_datasets = df_ds[df_ds["DatasetID"].isin(linked_ids)].to_dict('records')
                for row in project_datasets: row["_Active_Linked_Job"] = linked_job_map.get(row["DatasetID"], "UnknownJob")

            if not project_datasets: print(f"    ℹ️ No datasets found."); return

            total_rows = len(project_datasets)
            print(f"    📥 Loaded {total_rows} rows. Grouping & Verifying with Oracle...")

            groups = {}
            project_datasets.sort(key=lambda x: float(x.get("Overall Confidence", 0) or 0), reverse=True)

            for i, row in enumerate(project_datasets):
                if i > 0 and i % 20 == 0: print(f"      ⏳ Grouping progress: {i}/{total_rows} datasets evaluated...")
                key, match_type = self._find_group_key(row, groups)
                if key: groups[key].append(row)
                else: groups[row.get("DatasetID")] = [row]

            golden_records = []
            all_dropped_ids = []
            report_rows = []
            unmerged_singletons = []

            for grp_id, rows in groups.items():
                if len(rows) == 1:
                    unmerged_singletons.append(rows[0])
                    golden_records.append(rows[0])
                    continue

                golden = rows[0].copy()
                kept_id = golden["DatasetID"]

                dropped_ids = []
                dropped_names = []
                for r in rows[1:]:
                    dropped_ids.append(r["DatasetID"])
                    dropped_names.append(r.get("Variant Name", "") or r.get("Canonical Name", "Unknown"))

                changes = []
                for challenger in rows[1:]: self._merge_into_golden(golden, challenger, changes)

                projs = {str(r.get("Project_Created")) for r in rows if r.get("Project_Created")}
                projs.update({str(r.get("Project_Updated")) for r in rows if r.get("Project_Updated")})
                projs = {p for p in projs if p and p.lower() != "nan"}

                report_rows.append({
                    "Golden ID": kept_id, "Golden Name": golden.get("Variant Name", "")[:30],
                    "Absorbed Names": ", ".join(dropped_names)[:60], "Merged IDs": ", ".join(dropped_ids),
                    "Projects": ", ".join(sorted(projs)), "Changes": len(changes)
                })
                all_dropped_ids.extend(dropped_ids)

                if not dry_run:
                    golden["Job_Updated"] = job_id
                    golden["Date_Updated"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
                    if not is_global: golden["Project_Updated"] = project_id

                if "_Active_Linked_Job" in golden: del golden["_Active_Linked_Job"]
                golden_records.append(golden)

            singleton_actions = {}
            if unmerged_singletons:
                singleton_actions = self._evaluate_and_print_singletons(unmerged_singletons, project_id, project_context, is_global, run_llm=enable_singleton_verification, project_source_text=project_source_text)

            final_golden_records = []
            dropped_singletons = []
            unlink_singletons = []
            lost_singletons = []

            for r in golden_records:
                ds_id = r["DatasetID"]
                action = singleton_actions.get(ds_id, "KEEP")
                total_links_count = len(df_links[df_links["DatasetID"] == ds_id]["ProjectID"].unique())

                if action == "DROP":
                    if total_links_count > 1 and not is_global:
                        unlink_singletons.append(ds_id); final_golden_records.append(r)
                    else: dropped_singletons.append(ds_id)
                elif action == "LOST":
                    if total_links_count > 1 and not is_global:
                        unlink_singletons.append(ds_id); final_golden_records.append(r)
                    else: lost_singletons.append(ds_id); final_golden_records.append(r)
                else: final_golden_records.append(r)

            if dropped_singletons: print(f"    🗑️ Hallucinations Dropped (Will be removed from DB completely): {len(dropped_singletons)}")
            if unlink_singletons: print(f"    ✂️ Unlinked from {project_id} (Does not belong here, but kept in DB): {len(unlink_singletons)}")
            if lost_singletons: print(f"    🗂️ Reassigned completely to PROJ_LOST: {len(lost_singletons)}")

            print("\n📊 MERGE REPORT")
            print(f"    Input: {len(project_datasets)} -> Output: {len(final_golden_records)}")
            print(f"    Duplicates Absorbed: {len(all_dropped_ids)}")

            if report_rows:
                pd.DataFrame(report_rows).to_csv("Merge_Log.csv", index=False)
                print("    📝 Full Merge Log saved to: Merge_Log.csv")
                print(f"\n    🟢 SUCCESSFUL MERGES ({len(report_rows)} groups consolidated):")
                print(tabulate(report_rows, headers="keys", tablefmt="simple"))
            else: print("    ℹ️ No duplicates found.")

            if dry_run:
                print("\n🚫 DRY RUN: Changes NOT saved.")
                return

            print("\n💾 SAVING CHANGES...")

            if lost_singletons:
                df_proj = self.kb_manager.get_kb_data("Projects")
                if "PROJ_LOST" not in df_proj["ProjectID"].values:
                    df_proj = pd.concat([df_proj, pd.DataFrame([{"ProjectID": "PROJ_LOST", "Name": "Lost / Incidental Datasets", "Type": "System"}])], ignore_index=True)
                    self.kb_manager.write_dataframe_to_tab("Projects", df_proj)

            if is_global: df_final = pd.DataFrame(final_golden_records)
            else:
                df_ds_clean = df_ds[~df_ds["DatasetID"].isin(linked_ids)]
                df_final = pd.concat([df_ds_clean, pd.DataFrame(final_golden_records)], ignore_index=True)

            self.kb_manager.write_dataframe_to_tab("Datasets", df_final)

            if all_dropped_ids or dropped_singletons or lost_singletons or unlink_singletons:
                if all_dropped_ids:
                    id_map = {}
                    for grp_id, rows in groups.items():
                        if len(rows) > 1:
                            for r in rows[1:]: id_map[r["DatasetID"]] = rows[0]["DatasetID"]
                    mask = df_links["DatasetID"].isin(id_map.keys())
                    if mask.any():
                        df_links.loc[mask, "DatasetID"] = df_links.loc[mask, "DatasetID"].map(id_map).fillna(df_links.loc[mask, "DatasetID"])
                        curr_c = df_links.loc[mask, "Data Preparation Comments"].fillna("").astype(str)
                        df_links.loc[mask, "Data Preparation Comments"] = curr_c.apply(lambda x: f"{x} [OracleMerge]" if "[OracleMerge]" not in x else x)

                if dropped_singletons:
                    df_links = df_links[~df_links["DatasetID"].isin(dropped_singletons)].copy()
                if unlink_singletons and not is_global:
                    df_links = df_links[~((df_links["DatasetID"].isin(unlink_singletons)) & (df_links["ProjectID"] == project_id))].copy()
                if lost_singletons:
                    mask_lost = df_links["DatasetID"].isin(lost_singletons)
                    if not is_global: mask_lost = mask_lost & (df_links["ProjectID"] == project_id)
                    df_links.loc[mask_lost, "ProjectID"] = "PROJ_LOST"
                    curr_c = df_links.loc[mask_lost, "Data Preparation Comments"].fillna("").astype(str)
                    df_links.loc[mask_lost, "Data Preparation Comments"] = curr_c.apply(lambda x: f"{x} [Reassigned to PROJ_LOST]" if "[Reassigned to PROJ_LOST]" not in x else x)

                df_links = df_links.drop_duplicates(subset=["ProjectID", "DatasetID"])
                self.kb_manager.write_dataframe_to_tab("Project_Dataset_Link", df_links)
                print(f"    🔗 Project Links Updated.")

            print(f"✅ Merge Complete.")

        except Exception as e:
            print(f"    ❌ [Merge Error] Critical Failure: {e}")

        finally:
            lock.release()

    def _verify_singleton_context_with_llm(self, row, project_id, project_context, project_source_text=""):
        if not self.oracle.models or not hasattr(self.oracle.models, 'CLIENT'): return "N/A", "N/A", "LLM models missing"
        name = row.get("Variant Name", "") or row.get("Canonical Name", "Unknown")
        type_val = str(row.get("Type", "Dataset")).strip()
        desc = str(row.get("Description", ""))[:500]
        url = str(row.get("Primary URL", ""))
        proj_name = project_id.replace("PROJ_", "")
        source_context_block = f"--- BEGIN SOURCE ARTICLES / REPOSITORIES ---\n{project_source_text}\n--- END SOURCE ARTICLES ---\n\n" if project_source_text else ""
        prompt = (
            f"Act as a Data Archivist. We are auditing a catalog of time-series datasets and collections.\n\n"
            f"TARGET PARENT ENTITY:\nName: {proj_name}\nContext/Description: {project_context}\n\n{source_context_block}"
            f"CANDIDATE DATASET TO EVALUATE:\nName: {name}\nType: {type_val}\nURL: {url}\nDescription: {desc}\n\n"
            f"TASK:\n1. Real-World Existence: Does this candidate dataset actually exist in the real world?\n"
            f"2. Contextual Relevance: Does this dataset belong to the Target Parent Entity? \n"
            f"   - VERY IMPORTANT: Datasets frequently belong to multiple collections, repositories, or listicles.\n"
            f"   - If 'SOURCE ARTICLES / REPOSITORIES' text is provided above, you MUST scan it. If the candidate dataset is explicitly mentioned, utilized, or listed anywhere in that text, it unequivocally belongs to this Project. Do NOT reject it just because it is hosted elsewhere.\n"
            f"   - Be highly lenient with naming variations. If the Target features 'Ozone' and candidate is 'Ozone Level Detection Data Set (UCI)', that is a MATCH.\n"
        )
        try:
            self.oracle.stats['llm_calls'] += 1
            kwargs = {}
            if types: kwargs["config"] = types.GenerateContentConfig(response_mime_type="application/json", response_schema=SingletonVerificationSchema)
            response_text = self.oracle._call_llm_with_cascade(prompt, pool_preference="PRO" if project_source_text else "FLASH", **kwargs)
            try:
                res = json.loads(response_text)
                return "Yes" if res.get("in_project") else "No", "Yes" if res.get("exists") else "No", res.get("rationale", "")[:80]
            except json.JSONDecodeError:
                match = re.search(r"\{.*?\}", response_text.strip(), re.DOTALL)
                if match:
                    try:
                        res = json.loads(match.group())
                        return "Yes" if res.get("in_project") else "No", "Yes" if res.get("exists") else "No", res.get("rationale", "")[:80]
                    except json.JSONDecodeError:
                        pass
        except Exception as e:
            self.oracle.stats['errors'] += 1
            return "Error", "Error", str(e)[:40]
        return "Unknown", "Unknown", "Failed to parse JSON"

    def _evaluate_and_print_singletons(self, singletons, project_id, project_context, is_global, run_llm=False, project_source_text=""):
        print(f"\n    🟡 UNMERGED DATASETS ({len(singletons)} Singletons)")
        if is_global: run_llm = False; print("    ⏩ GLOBAL Mode: Skipping LLM context verification for singletons (Already verified at project level).")
        elif run_llm and self.oracle.models and hasattr(self.oracle.models, 'CLIENT'):
            search_badge = " (Search Grounding ON)" if self.oracle.enable_search else ""
            if project_source_text: search_badge += f" (+ {len(project_source_text)} chars of Source Text)"
            print(f"    🕵️ Running Context-Aware LLM Verification against '{project_id}'{search_badge}...")
        else: run_llm = False

        s_data = []
        for i, r in enumerate(singletons):
            did = r.get("DatasetID", "UnknownID")
            name = r.get("Variant Name", "") or r.get("Canonical Name", "Unknown")
            clean_name = self.oracle._clean_name(name)
            try: conf_val = float(r.get("Overall Confidence", 0))
            except: conf_val = 0.0

            job = str(r["_Active_Linked_Job"]) if not is_global and "_Active_Linked_Job" in r else str(r.get("Job_Created", ""))
            if not job or job.lower() in self.MISSING or job == "nan": job = "UnknownJob"
            in_proj, exists, rationale = "Skipped", "Skipped", "LLM Check Bypassed"

            if conf_val >= 0.98:
                rt_action, in_proj, exists, rationale = "✅ KEEP (Harvester/Registry Source - 100% Valid)", "Yes", "Yes", "Deterministic Source - Auto Verified"
                if self.verbosity >= 1: print(f"      🤖 [{i+1}/{len(singletons)}] '{name[:35]}': {rt_action}")
            elif run_llm:
                in_proj, exists, rationale = self._verify_singleton_context_with_llm(r, project_id, project_context, project_source_text=project_source_text)
                if exists == "No": rt_action = "🗑️ DROP (Hallucinated)"
                elif exists == "Yes" and in_proj == "No": rt_action = "✂️ UNLINK / LOST (Out of Scope)"
                else: rt_action = "✅ KEEP (Valid)"
                if self.verbosity >= 1: print(f"      🤖 [{i+1}/{len(singletons)}] '{name[:35]}': {rt_action} -> Exists:{exists}, InProj:{in_proj}")
            else: rt_action = "✅ KEEP (Valid)"
            s_data.append({"id": did, "name": name, "clean_name": clean_name, "job": job, "row": r, "in_proj": in_proj, "exists": exists, "rationale": rationale})

        report_data, actions, display_name = [], {}, project_id.replace("PROJ_", "") if project_id else "Target Project"
        for current in s_data:
            if current["exists"] == "No": actions[current["id"]] = "DROP"
            elif current["exists"] == "Yes" and current["in_proj"] == "No": actions[current["id"]] = "LOST"
            else: actions[current["id"]] = "KEEP"
            best_match_name, best_match_id, best_score = "None", "", 0.0
            for candidate in s_data:
                if current["job"] != candidate["job"]:
                    score = difflib.SequenceMatcher(None, current["clean_name"], candidate["clean_name"]).ratio()
                    if score > best_score: best_score, best_match_name, best_match_id = score, candidate["name"], candidate["id"]
            match_str = f"{best_match_id} ({best_match_name[:20]}) [Sim: {best_score:.2f}]" if best_score > 0 else "No Cross-Job Match"
            report_data.append({"Dataset ID": current["id"], "Name": current["name"][:30], "Job": current["job"], f"In {display_name}?": current["in_proj"], "Real Dataset?": current["exists"], "LLM Rationale": current["rationale"], "Closest Cross-Job Match": match_str})

        if not is_global:
            for job in sorted(list(set(d["Job"] for d in report_data))):
                if self.verbosity >= 1:
                    print(f"\n      🔹 LINKED TO {display_name} BY RUN: {job}")
                    print("      " + tabulate([{k: v for k, v in d.items() if k != "Job"} for d in report_data if d["Job"] == job], headers="keys", tablefmt="simple").replace("\n", "\n      "))
        return actions

    def _find_group_key(self, row, groups):
        for key, members in groups.items():
            is_dup, reason = self.oracle.evaluate_pair(row, members[0])
            if is_dup: return key, reason
        return None, None

    def _merge_into_golden(self, golden, challenger, changes):
        for field in self.DATA_FIELDS:
            val_g, val_c = str(golden.get(field, "")).strip(), str(challenger.get(field, "")).strip()
            conf_f = f"{field} (C)"
            try: c_g, c_c = float(golden.get(conf_f, 0)), float(challenger.get(conf_f, 0))
            except: c_g, c_c = 0, 0

            if c_g == 1.0: continue

            if "URL" in field or "Link" in field:
                if val_c and val_c not in self.MISSING:
                    if val_g and val_g not in self.MISSING:
                        clean_c = re.search(r'=HYPERLINK\("([^"]+)"', val_c, re.IGNORECASE)
                        base_c = clean_c.group(1) if clean_c else val_c
                        if base_c not in val_g:
                            safe_append = base_c.replace('"', '""')
                            golden[field], golden[conf_f] = f"{val_g} & \" | Alt: {safe_append}\"", max(c_g, c_c)
                            changes.append(f"Appended {field}")
                    elif not val_g or val_g in self.MISSING:
                        golden[field], golden[conf_f] = val_c, c_c
                        changes.append(f"Filled {field}")
            elif "Description" in field or "Rationale" in field:
                if val_c and val_c not in self.MISSING:
                    if not val_g or val_g in self.MISSING:
                        golden[field], golden[conf_f] = val_c, c_c
                        changes.append(f"Filled {field}")
                    elif len(val_c) > len(val_g):
                        golden[field], golden[conf_f] = val_c, max(c_g, c_c)
                        changes.append(f"Replaced {field} with longer text")
            else:
                if c_c == 1.0 or ((val_g in self.MISSING) and (val_c not in self.MISSING)) or ((val_c not in self.MISSING) and c_c > c_g):
                    golden[field], golden[conf_f] = val_c, 1.0 if c_c == 1.0 else c_c
                    changes.append(f"Merge {field}")

print("✅ deepcollector/kb/merger.py written (Short-Acronym Dedup Shield).")

Writing deepcollector/kb/merger.py


# -----

### Execution Harness

These final cells configure and run the agent using the newly created modules.

#### **Execution Cell 21: Configuration Definition**

🧪 How to Test the New Cellular RAG Enhancements
In the new setup for Cell 21, you will find an --- ADVANCED RAG TOGGLES --- section. Here is what each parameter controls:

**ENABLE_PREFLIGHT_CRAWLER = True**

What it does: During the Bootstrapping phase, the agent uses an LLM to scan the initial project URL specifically for outbound links to GitHub, HuggingFace, Zenodo, and PDFs. It automatically fetches and indexes these before starting Discovery.

Best for: Projects where the actual dataset list is buried in a supplementary GitHub repo or technical appendix rather than the main project page.

**ENABLE_DISCREPANCY_ARBITRATION = True**

What it does: Injects a strict arbitration prompt during the extraction phase. If the RAG engine retrieves conflicting numbers (e.g., the abstract says 55 datasets, but the repo lists 67), it forces the LLM to prioritize the repository/technical tables over the abstract text and explain its choice.

Best for: High-cardinality collections where paper text and code repos disagree.

**ENABLE_ENTITY_TAXONOMY = True**

What it does: Forces the Discovery extraction to use a strict enum. It will aggressively filter out tools, software, synthetic generators (like KernelSynth), and evaluation scripts, ensuring only actual datasets and collections make it into the catalog.

Best for: Repositories that mix software libraries, synthetic augmentation tools, and real datasets.

**ENABLE_MULTI_QUERY_RAG = True**

What it does: Instead of asking the LlamaIndex retriever one massive question per cell, it automatically generates specific sub-queries (e.g., "How many features does [Dataset] have?" or "What is the time interval for [Dataset]?") and retrieves chunks for all of them, combining the context before passing it to the extraction LLM.

Best for: Datasets where metadata is scattered across multiple different pages or distant paragraphs.

**ENABLE_VARIANT_MAPPING = True**

What it does: Tells the Discovery agent to explicitly look for and split out Parent-Child hierarchies (e.g., splitting WeatherBench into WeatherBench-Hourly and WeatherBench-Daily as separate rows).

Best for: Benchmarks that use multiple temporal resolutions of the same underlying data.

In [ ]:
# =============================================================================
# Cell 21: Unified Executor (With Console Log Export & Config Driven IDs)
# =============================================================================
from google.colab import output
output.no_vertical_scroll()

import os
import sys
import pandas as pd
import gspread
import importlib
import time
import warnings
import uuid
import copy
import re
from datetime import datetime
from tabulate import tabulate
from collections import Counter

# -----------------------------------------------------------------------------
# 🛠️ CONSOLE LOGGER (Captures terminal output to a file)
# -----------------------------------------------------------------------------
class TeeLogger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log_file = open(filename, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)
        self.flush()

    def flush(self):
        self.terminal.flush()
        self.log_file.flush()

    def close(self):
        self.log_file.close()

# Start capturing console output immediately
run_timestamp = time.strftime('%Y%m%d_%H%M')
PROJECT_NAMES = ["M6", "Tempo"]
proj_str = "_".join(PROJECT_NAMES)[:30]
log_filename = f"{proj_str}_{run_timestamp}_ConsoleLog.txt"

# Only redirect if it hasn't been redirected yet to prevent deep nesting
if not hasattr(sys.stdout, 'log_file'):
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = TeeLogger(log_filename)
    sys.stderr = sys.stdout

try:
    warnings.filterwarnings("ignore", category=FutureWarning, module="google.generativeai")
    warnings.filterwarnings("ignore", category=UserWarning, module="google.generativeai")

    print("🔄 Reloading Modules...")
    modules_to_reload = [
        'deepcollector.config.schema', 'deepcollector.config.settings', 'deepcollector.utils.profiler',
        'deepcollector.utils.initialization', 'deepcollector.tools.research', 'deepcollector.kb.manager',
        'deepcollector.kb.maintenance', 'deepcollector.kb.quality', 'deepcollector.kb.merger',
        'deepcollector.core.state', 'deepcollector.core.rag_engine', 'deepcollector.harvesting.base_harvester',
        'deepcollector.harvesting.uci_harvester', 'deepcollector.utils.project_loader', 'deepcollector.core.agent',
        'deepcollector.tools.deep_research_runner', 'deepcollector.utils.analytics'
    ]

    for module_name in modules_to_reload:
        if module_name in sys.modules: importlib.reload(sys.modules[module_name])
        else:
            try: __import__(module_name)
            except ImportError: pass

    from deepcollector.core.agent import CatalogAgent
    from deepcollector.utils.initialization import initialize_apis, configure_llama_index
    from deepcollector.config.settings import AppConfig
    from deepcollector.kb.manager import KnowledgeBaseManager
    from deepcollector.kb.quality import QualityAuditor
    from deepcollector.utils.project_loader import ProjectLoader, ExternalKnowledge
    from deepcollector.utils.profiler import profiler

    # -----------------------------------------------------------------------------
    # CONFIGURATION / USER INPUTS
    # -----------------------------------------------------------------------------
    MODE = "AGENT"

    ENABLE_DEEP_RESEARCH = True
    DRY_RUN = False

    # --- ADVANCED RAG TOGGLES ---
    ENABLE_PREFLIGHT_CRAWLER = True
    ENABLE_DISCREPANCY_ARBITRATION = True
    ENABLE_ENTITY_TAXONOMY = True
    ENABLE_MULTI_QUERY_RAG = True
    ENABLE_VARIANT_MAPPING = True

    # --- DEEP RESEARCH SETTINGS ---
    DEEP_RESEARCH_TIMEOUT_MINUTES = 25
    DEEP_RESEARCH_MAX_RETRIES = 4
    DEEP_RESEARCH_POLLING_INTERVAL_SECONDS = 15
    FORCE_ASYNC_POLLING = False
    ABORT_ON_DEEP_RESEARCH_FAILURE = False

    # --- STANDARD CONFIGS ---
    LLM_ARBITRATION_LIMIT = 1500
    WIPE_CURRENT_PROJECT_ONLY = False
    ENABLE_SINGLETON_VERIFICATION = True
    ENABLE_ORACLE_SEARCH = True

    # --- GLOBAL IDs AND LOCATIONS ---
    GOOGLE_SHEET_KB_ID = "1-PuWrHO30E4WPM-rOed03n42gfo5AlEtscKqqtjznA0"
    GOOGLE_SHEET_HINTS_ID = "1mpv0V5dEQOAv1R1H0ggim2c9TrWoq2wxfVKSUHNDVHo"
    GOOGLE_SHEET_PROJECT_LIST_ID = "1gJ6oHZj0NzCHNOeFNyJTBTtlmS0b7gBSHF3iOqJrFwE"

    # NEW: Separation of Drive Exports (Passed to config)
    GOOGLE_DRIVE_SHEET_FOLDER_ID = "15nvgAVcvoigcQUyla9T9A66KDdj7ZPoK"
    GOOGLE_DRIVE_LOG_FOLDER_ID = "1mw42keL9BNmaNgrW_ssDFGxe8_lcXBQ2"

    KNOWLEDGE_MASTER_DOC_URL = "https://docs.google.com/document/d/16oN5NyOC2lFBQvLgept4y9yiTUbgKqiz-mLRXnokefw/export?format=html"

    VERBOSITY = 1

    # -----------------------------------------------------------------------------
    # SETUP & VALIDATION
    # -----------------------------------------------------------------------------
    print(f"🚀 Initializing (Mode: {MODE})...")

    if 'gc' not in globals() or gc is None:
        sys.exit("🛑 CRITICAL ERROR: Google Sheets client 'gc' not found. Please run Cell 1 to authenticate first.")

    SECRETS = {}
    try:
        from google.colab import userdata
        SECRETS["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception: pass

    base_config = AppConfig(
        CURRENT_PROJECT_ID="TEMP",
        CURRENT_PROJECT_NAME="TEMP",
        GOOGLE_SHEET_KB_INPUT=GOOGLE_SHEET_KB_ID,
        GOOGLE_SHEET_HINTS_INPUT=GOOGLE_SHEET_HINTS_ID,
        GOOGLE_SHEET_PROJECT_LIST_INPUT=GOOGLE_SHEET_PROJECT_LIST_ID,
        GOOGLE_DRIVE_SHEET_FOLDER_ID=GOOGLE_DRIVE_SHEET_FOLDER_ID,
        GOOGLE_DRIVE_LOG_FOLDER_ID=GOOGLE_DRIVE_LOG_FOLDER_ID,
        KNOWLEDGE_MASTER_DOC_URL=KNOWLEDGE_MASTER_DOC_URL,
        VERBOSITY_LEVEL=VERBOSITY,
        SECRETS=SECRETS,
        WIPE_CURRENT_PROJECT_ONLY=WIPE_CURRENT_PROJECT_ONLY,
        LLM_ARBITRATION_LIMIT=LLM_ARBITRATION_LIMIT,
        ENABLE_DEEP_RESEARCH=ENABLE_DEEP_RESEARCH,
        DEEP_RESEARCH_TIMEOUT_MINUTES=DEEP_RESEARCH_TIMEOUT_MINUTES,
        DEEP_RESEARCH_MAX_RETRIES=DEEP_RESEARCH_MAX_RETRIES,
        ABORT_ON_DEEP_RESEARCH_FAILURE=ABORT_ON_DEEP_RESEARCH_FAILURE,
        FORCE_ASYNC_POLLING=FORCE_ASYNC_POLLING,
        DEEP_RESEARCH_POLLING_INTERVAL_SECONDS=DEEP_RESEARCH_POLLING_INTERVAL_SECONDS,
        ENABLE_PREFLIGHT_CRAWLER=ENABLE_PREFLIGHT_CRAWLER,
        ENABLE_ARBITRATION_PROMPT=ENABLE_DISCREPANCY_ARBITRATION,
        ENABLE_STRICT_TAXONOMY=ENABLE_ENTITY_TAXONOMY,
        ENABLE_MULTI_QUERY_RAG=ENABLE_MULTI_QUERY_RAG,
        ENABLE_VARIANT_MAPPING=ENABLE_VARIANT_MAPPING
    )

    base_config.ENABLE_SINGLETON_VERIFICATION = ENABLE_SINGLETON_VERIFICATION
    base_config.ENABLE_ORACLE_SEARCH = ENABLE_ORACLE_SEARCH

    if MODE in ["MERGE", "REVIEW", "PLAN"]: base_config.LLAMA_INDEX_AVAILABLE = False
    else: base_config.LLAMA_INDEX_AVAILABLE = True

    profiler.reset()

    # -----------------------------------------------------------------------------
    # PLANNER LOGIC (Runs once if MODE == 'PLAN')
    # -----------------------------------------------------------------------------
    def run_planner(kb_mgr):
        print("\n" + "="*80)
        print("📋 DEEPCOLLECTOR JOB PLANNER & DIAGNOSTICS")
        print("="*80)

        df_jobs = kb_mgr.get_kb_data("Jobs")
        df_ds = kb_mgr.get_kb_data("Datasets")
        df_links = kb_mgr.get_kb_data("Project_Dataset_Link")

        if not df_jobs.empty and 'Start_Time' in df_jobs.columns:
            df_jobs['Start_Time_DT'] = pd.to_datetime(df_jobs['Start_Time'], errors='coerce')
        else:
            df_jobs['Start_Time_DT'] = pd.NaT

        completed_jobs = df_jobs[df_jobs['Status'].str.contains('COMPLETED', na=False)] if not df_jobs.empty else pd.DataFrame()

        merge_targets = []
        if not completed_jobs.empty:
            for pid in completed_jobs['ProjectID'].dropna().unique():
                if pid in ["GLOBAL", "PROJ_UNKNOWN", "PROJ_LOST", "PROJ_GLOBAL_MAINTENANCE"]: continue
                proj_jobs = completed_jobs[completed_jobs['ProjectID'] == pid]
                agent_jobs = proj_jobs[proj_jobs['Mode'].isin(['AGENT', 'HARVEST'])]
                merge_jobs = proj_jobs[proj_jobs['Mode'] == 'MERGE']
                agent_count = len(agent_jobs)
                if agent_count > 1:
                    last_agent_time = agent_jobs['Start_Time_DT'].max()
                    last_merge_time = merge_jobs['Start_Time_DT'].max() if not merge_jobs.empty else pd.Timestamp.min
                    if pd.isna(last_merge_time) or last_agent_time > last_merge_time:
                        merge_targets.append((pid, agent_count))

        if merge_targets:
            print("\n🛠️ STEP 1: PROJECT-LEVEL MERGES")
            merge_table = [{"Set ProjectName": str(pid).replace('PROJ_', ''), "Set MODE": "MERGE", "Reason": f"{count} overlapping runs"} for pid, count in merge_targets]
            print(tabulate(merge_table, headers="keys", tablefmt="simple"))
        else:
            print("\n✅ STEP 1: No project-level MERGEs required.")

        repair_targets = []
        if not df_ds.empty and not df_links.empty:
            ds_to_proj = dict(zip(df_links['DatasetID'], df_links['ProjectID']))
            df_ds['Assigned_Project'] = df_ds['DatasetID'].map(ds_to_proj)
            missing_markers = {"", "[missing]", "nan", "none"}
            proj_repair_needs = Counter()
            proj_totals = Counter()

            fields_to_check = ["Primary URL", "Link to Data (Actual Source)", "Other URL", "Num Time Points", "Total Variables"]

            for _, row in df_ds.iterrows():
                pid = row.get('Assigned_Project', 'Unknown')
                if pd.isna(pid) or pid in ["GLOBAL", "PROJ_UNKNOWN", "PROJ_LOST", "PROJ_GLOBAL_MAINTENANCE", "Unknown"]: continue
                proj_totals[pid] += 1
                needs_repair = False
                for field in fields_to_check:
                    val = str(row.get(field, "")).strip().lower()
                    if val in missing_markers:
                        needs_repair = True; break
                    conf_col = f"{field} (C)"
                    try:
                        if float(row.get(conf_col, 0.0)) < 0.80: needs_repair = True; break
                    except: pass
                if needs_repair: proj_repair_needs[pid] += 1

            for pid, need_count in proj_repair_needs.items():
                total = proj_totals[pid]
                pct = (need_count / total) * 100
                if pct > 30 and total > 2:
                    proj_jobs = completed_jobs[completed_jobs['ProjectID'] == pid] if not completed_jobs.empty else pd.DataFrame()
                    repair_jobs = proj_jobs[proj_jobs['Mode'] == 'REPAIR']
                    other_jobs = proj_jobs[proj_jobs['Mode'].isin(['AGENT', 'HARVEST', 'MERGE'])]
                    last_repair_time = repair_jobs['Start_Time_DT'].max() if not repair_jobs.empty else pd.Timestamp.min
                    last_other_time = other_jobs['Start_Time_DT'].max() if not other_jobs.empty else pd.Timestamp.min

                    if pd.isna(last_repair_time) or last_repair_time < last_other_time:
                        repair_targets.append((pid, need_count, total, pct))

        if repair_targets:
            print("\n🛠️ STEP 2: PROJECT-LEVEL REPAIRS")
            repair_targets.sort(key=lambda x: x[3], reverse=True)
            repair_table = [{"Set ProjectName": str(pid).replace('PROJ_', ''), "Set MODE": "REPAIR", "Reason": f"{need}/{tot} datasets ({pct:.1f}%) need fixes"} for pid, need, tot, pct in repair_targets]
            print(tabulate(repair_table, headers="keys", tablefmt="simple"))
        else:
            print("\n✅ STEP 2: No project-level REPAIRs required.")

        print("\n🌍 STEP 3: GLOBAL ACTIONS")
        global_table = [
            {"Order": "1st", "Set ProjectName": "GLOBAL", "Set MODE": "MERGE", "Description": "Cross-project deduplication."},
            {"Order": "2nd", "Set ProjectName": "GLOBAL", "Set MODE": "REVIEW", "Description": "Quality audit & move flaky datasets to Quarantine."}
        ]
        print(tabulate(global_table, headers="keys", tablefmt="simple"))

        try:
            ek = ExternalKnowledge(kb_mgr.config, kb_mgr.gc, verbosity=0)
            ek.load()
            if ek.projects:
                completed_pids = set(completed_jobs['ProjectID'].dropna().unique()) if not completed_jobs.empty else set()
                missing_projects = []

                for p in ek.projects:
                    pid_raw = ""
                    for k, v in p.items():
                        if any(term in str(k).lower() for term in ["canonical", "projectid", "id"]):
                            val = str(v).strip()
                            m = re.search(r'=HYPERLINK\([^,]+,\s*"([^"]+)"\)', val, re.IGNORECASE)
                            if m: val = m.group(1)
                            pid_raw = val.upper()
                            break

                    if not pid_raw and p:
                        val = str(list(p.values())[0]).strip()
                        m = re.search(r'=HYPERLINK\([^,]+,\s*"([^"]+)"\)', val, re.IGNORECASE)
                        if m: val = m.group(1)
                        pid_raw = val.upper()

                    pid = f"PROJ_{pid_raw}" if pid_raw and not pid_raw.startswith("PROJ_") else pid_raw

                    method_str = ""
                    for k, v in p.items():
                        if "method" in str(k).lower():
                            method_str = str(v)
                            break
                    try:
                        clean_val = re.sub(r'[^0-9-]', '', method_str)
                        method_val = int(clean_val) if clean_val else 1
                    except ValueError:
                        method_val = 1

                    if method_val == -1: continue

                    if pid not in completed_pids and pid != "PROJ_" and pid_raw != "LOST" and pid != "PROJ_LOST":
                        missing_projects.append(pid.replace('PROJ_', ''))

                if missing_projects:
                    print("\n🆕 STEP 4: UNRUN PROJECTS (Found in Master Sheet but not in KB)")
                    print(f"    You have {len(missing_projects)} active projects left to run.")
                    print("    Next 5 suggestions: " + ", ".join(missing_projects[:5]))
                else:
                    print("\n✅ STEP 4: All active projects from the Master Sheet have been run!")

        except Exception as e:
            print(f"    (Could not check Master Sheet for unrun projects: {e})")

        print("="*80 + "\n")

    # -----------------------------------------------------------------------------
    # EXECUTION LOGIC (Batch Processing Loop)
    # -----------------------------------------------------------------------------
    if MODE == "PLAN":
        kb_manager = KnowledgeBaseManager(base_config)
        kb_manager.initialize_connection(gc)
        if kb_manager.read_and_validate_kb(): run_planner(kb_manager)
        else: print("    ❌ Failed to load Knowledge Base.")

    elif MODE == "REVIEW":
        keys, models = initialize_apis(base_config)
        kb_manager = KnowledgeBaseManager(base_config)
        kb_manager.initialize_connection(gc)
        if kb_manager.read_and_validate_kb():
            job_id = f"JOB_{uuid.uuid4().hex[:6].upper()}"
            auditor = QualityAuditor(kb_manager, models=models, llm_limit=base_config.LLM_ARBITRATION_LIMIT)
            auditor.run_audit_and_clean(run_quarantine=True, dry_run=DRY_RUN, job_id=job_id, project_id="PROJ_GLOBAL_MAINTENANCE", job_comment="Global Review")

    elif MODE in ["HARVEST", "AGENT", "REPAIR", "MERGE"]:
        keys, models = initialize_apis(base_config)
        if MODE != "MERGE": llama_success = configure_llama_index(base_config, models, keys)

        total_projects = len(PROJECT_NAMES)
        for i, p_name in enumerate(PROJECT_NAMES):
            print(f"\n{'='*70}")
            print(f"🚀 BATCH PROGRESS: [{i+1}/{total_projects}] Processing Project: {p_name}")
            print(f"{'='*70}")

            current_job_id = f"JOB_{uuid.uuid4().hex[:6].upper()}"
            current_project_id = "PROJ_GLOBAL_MAINTENANCE" if p_name == "GLOBAL" else f"PROJ_{p_name.upper()}"
            job_comment = f"{p_name} Batch {MODE} execution."

            import copy
            current_config = copy.deepcopy(base_config)
            current_config.CURRENT_PROJECT_ID = current_project_id
            current_config.CURRENT_PROJECT_NAME = p_name

            run_mode = MODE

            if p_name != "GLOBAL":
                print(f"🔄 Loading Project Context for {p_name}...")
                loader = ProjectLoader(current_config, gc, verbosity=VERBOSITY)

                if loader.resolve_configuration():
                    print(f"    ✅ Context Loaded: '{current_config.PROJECT_CONTEXT[:80]}...'")
                    if current_config.PROJECT_METHOD == 2 and run_mode == "AGENT":
                        print(f"    ℹ️  Project defines Method 2 in Master Sheet. Switching to HARVEST mode.")
                        run_mode = "HARVEST"
                else:
                    print(f"    ⚠️ Skipping '{p_name}': Invalid Project Configuration or missing from Master Sheet.")
                    continue

            try:
                agent = CatalogAgent(current_config, gc, keys=keys, models=models)
                agent.job_id = current_job_id
                if p_name == "GLOBAL": agent.config.CURRENT_PROJECT_ID = "GLOBAL"
                agent.execute_workflow(mode=run_mode, job_comment=job_comment, merge_dry_run=DRY_RUN)
                print(f"\n🎉 Successfully finished [{i+1}/{total_projects}]: {p_name}")
            except Exception as e:
                print(f"\n❌ Error processing {p_name}: {e}")
                print("⚠️ Catching error and safely continuing to next project in queue...")

            if i < total_projects - 1:
                print("⏳ Pausing for 5 seconds before next project...")
                time.sleep(5)

        print(f"\n{'='*70}\n🏁 BATCH PROCESSING COMPLETE. Processed {total_projects} items.\n{'='*70}")

    if MODE != "PLAN":
        print("\n" + "="*40)
        print("⏱️ OVERALL PROFILING & HEALTH REPORT")
        print("="*40)
        try:
            df_profile = profiler.get_report()
            if not df_profile.empty: print(tabulate(df_profile, headers='keys', tablefmt='pipe', showindex=False))
        except: pass

finally:
    # -------------------------------------------------------------------------
    # UPLOAD LOG FILE TO GDRIVE FOLDER BEFORE CLOSING
    # -------------------------------------------------------------------------
    log_folder_id = getattr(base_config, 'GOOGLE_DRIVE_LOG_FOLDER_ID', None)
    if log_folder_id:
        print(f"\n☁️ Uploading Console Log ({log_filename}) to Google Drive...")

        # Restore normal printing so the log file can close properly
        if hasattr(sys.stdout, 'log_file'):
            sys.stdout.close()
            sys.stdout = original_stdout
            sys.stderr = original_stderr

        try:
            from googleapiclient.discovery import build
            from google.auth import default
            from googleapiclient.http import MediaFileUpload

            creds, _ = default()
            drive_service = build('drive', 'v3', credentials=creds)

            file_metadata = {
                'name': log_filename,
                'parents': [log_folder_id]
            }
            media = MediaFileUpload(log_filename, mimetype='text/plain')
            file = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
            print(f"✅ Success! Log File uploaded to Drive with ID: {file.get('id')}")
        except ImportError:
            print("⚠️ googleapiclient not installed. Skipping Drive upload.")
        except Exception as e:
            print(f"❌ Failed to upload log file to Google Drive: {e}")

<IPython.core.display.Javascript object>

🔄 Reloading Modules...
✅ deepcollector/config/schema.py written (V83: URL Separation).
✅ deepcollector/config/schema.py written (V83: URL Separation).
✅ deepcollector/config/settings.py written (100% Fully Expanded & Uncompressed).
✅ deepcollector/utils/profiler.py written.
✅ deepcollector/utils/initialization.py written (Warnings Suppressed).
✅ deepcollector/tools/research.py written (Native PyTorch 4-Bit Waterfall Restored).
✅ deepcollector/kb/manager.py written (100% Full Un-Truncated + GSpread V5/V6 Safe Fetch).
✅ deepcollector/kb/maintenance.py written (SyntaxWarning Fix).
✅ deepcollector/kb/merger.py written (Short-Acronym Dedup Shield).
✅ deepcollector/kb/quality.py written (Description Bloat Fix).
✅ deepcollector/kb/merger.py written (Short-Acronym Dedup Shield).
✅ deepcollector/core/state.py written (Added Search Memory Set).
✅ deepcollector/core/rag_engine.py written (100% Fully Expanded + Native JSON Prompt Fix).
✅ deepcollector/harvesting/base_harvester.py written (HDK Comp

    ✅ [BM25 Index] Rebuilt with 1 nodes.
    🕷️ [Pre-Flight Crawler] Scanning context for outbound repositories...
      🔗 Found 5 secondary URLs.
        🕸️ Fetching & Indexing: https://github.com/Mcompetitions/M6-methods/tree/main
    ✅ [Vector Index] Inserted 1 docs (24 nodes).
    ✅ [BM25 Index] Rebuilt with 25 nodes.
        🕸️ Fetching & Indexing: https://github.com/microprediction/m6/blob/main/docs/M6-forecasting-competition-Guidelines-20210908.pdf
    ✅ [Vector Index] Inserted 1 docs (9 nodes).
    ✅ [BM25 Index] Rebuilt with 34 nodes.
        🕸️ Fetching & Indexing: https://github.com/microprediction/m6/blob/main/docs/clarifications.md
    ✅ [Vector Index] Inserted 1 docs (22 nodes).
    ✅ [BM25 Index] Rebuilt with 56 nodes.
        🕸️ Fetching & Indexing: https://github.com/microprediction/m6/tree/main/data
    ✅ [Vector Index] Inserted 1 docs (9 nodes).
    ✅ [BM25 Index] Rebuilt with 65 nodes.
        🕸️ Fetching & Indexing: https://github.com/microprediction/m6/blob/main/P

    ✅ Success! CSV cleanly uploaded to Drive with ID: 1D5FP4Z6ne_y2kXHc-EFVuVWQfxPMuRCu

💰 VALUE ANALYSIS REPORT (Tri-State Updates)
Phase                             Time (s)  Time %      Net Added    Deletions    Filled    Refined    Confirmed    Total Cell Ops    Velocity
------------------------------  ----------  --------  -----------  -----------  --------  ---------  -----------  ----------------  ----------
Bootstrapping                         57    11.0%               0            0         0          0            0                 0        0
Phase 1a: Deep Research              220.5  42.5%               2            0        16          0            0                16        0.54
Phase 1b: Standard Discovery          38    7.3%                0            0         0          0            0                 0        0
Phase 1.5: Golden KB Fast-Path         0.4  0.1%                0            0         0          0            0                 0        0
Phase 2: Grounding

    ✅ [BM25 Index] Rebuilt with 3 nodes.
    🕷️ [Pre-Flight Crawler] Scanning context for outbound repositories...
      🔗 Found 2 secondary URLs.
        🕸️ Fetching & Indexing: https://github.com/DC-research/TEMPO
        🕸️ Fetching & Indexing: https://huggingface.co/Melady/TEMPO
    ✅ [Vector Index] Inserted 1 docs (15 nodes).
    ✅ [BM25 Index] Rebuilt with 18 nodes.

🚀 STANDARD RAG WORKFLOW (Method 1)

--- Phase 1a: Deep Research (Agentic) ---
    ℹ️ Deep Research Context: TEMPO: PROJ_Tempo (Source URL: ['http://arxiv.org/abs/2310.04948', 'https://github.com/DC-research/T...

🧠 [Deep Research] Submitting job to 'deep-research-pro-preview-12-2025' (Attempt 1/4): 'Act as a Data Archivist. You are tasked with mapping the SPE...'
    🚀 Task ID: v1_ChdqcklUYXJXWEZOLUFrZFVQX3F5Z3VRRRIXanJJVGFyV1hGTi1Ba2RVUF9xeWd1UUU | Status: STREAMING
    🧠 Thinking: **Identifying core research**  I have successfully linked the project identifier to the specific technical publication titled "TEMPO: Pr

    ✅ Success! CSV cleanly uploaded to Drive with ID: 15axB-SCCJJu20TYdBRBTl94QM_iaDTo4

💰 VALUE ANALYSIS REPORT (Tri-State Updates)
Phase                             Time (s)  Time %      Net Added    Deletions    Filled    Refined    Confirmed    Total Cell Ops    Velocity
------------------------------  ----------  --------  -----------  -----------  --------  ---------  -----------  ----------------  ----------
Bootstrapping                         10.2  0.2%                0            0         0          0            0                 0        0
Phase 1a: Deep Research              402.5  8.2%                9            0        72          0            0                72        1.34
Phase 1b: Standard Discovery          42.7  0.9%                0            0         0          0            0                 0        0
Phase 1.5: Golden KB Fast-Path         0.4  0.0%                0            0        11          0            0                11        0
Phase 2: Grounding

✅ Success! Log File uploaded to Drive with ID: 1Zheiq-WiDpmGZmeM3cIAruxWt2saY4xI
